# Paragraph-Level Organizational Unlearning Evaluation — GPT Test First

This Colab notebook runs a **restart-safe, auditable, sequential experiment** using the workbook's existing sheets without rewriting them:

- `GPT Test` — the curated paragraph-level benchmark used for all prompt, model, threshold, and ensemble decisions.
- `Codebook (revised)` — read exactly as supplied; the notebook never edits this sheet or saves a modified copy into the source workbook.
- `GPT Test ALL` — run only in the final locked-deployment stage, after every earlier decision is complete.

## Registered sequence

1. **Examples-column A/B test on GPT Test only**
   - Codebook without its native `Examples` column.
   - Codebook with its native `Examples` column.
   - All enabled candidate models are held to the same prompt scaffold, output schema, passage text, and provider-specific inference settings.
   - Winner: highest arithmetic mean model-level precision at threshold 0.50; deterministic tie-breakers are declared in code.

2. **Checklist A/B test on GPT Test only**
   - Carry forward only the winning examples configuration.
   - Compare no checklist versus the exact checklist from `Unlearning_Final_FullPDF_AB_Evaluation_Pipeline.ipynb`.
   - Only the new checklist arm requires calls; the no-checklist arm is reused.

3. **Best model selection within each provider on GPT Test**
   - Select exactly one OpenAI, one Anthropic, and one Google model under the final winning prompt.
   - Export a manual-review workbook containing every GPT Test row, the selected three models' predictions, Anmol and Prerana label columns, disagreement/error counts, evidence, and rationales.

4. **Direct-only baseline on GPT Test**
   - Run only the three selected provider models.
   - Baseline system: simple arithmetic mean of their direct-only probabilities at threshold 0.50.

5. **Threshold and weighted-ensemble analysis on GPT Test**
   - Tune per-model thresholds, equal-weight threshold, and nonnegative three-model weights plus threshold.
   - Save the full-data locked deployment configuration.
   - Use out-of-fold tuning for the primary optimized-versus-baseline performance comparison.

6. **Final GPT Test ALL deployment — last stage only**
   - Hard-gated by completed prompt, model, and deployment checkpoints.
   - Run only the selected model from each provider with the locked final prompt.
   - Reuse exact GPT Test paragraphs from the response cache to avoid duplicate charges.
   - Apply locked weights/threshold without any reselection or retuning.

## Reproducibility and inference safeguards

- The source workbook and complete `Codebook (revised)` sheet are SHA-256 hashed before analysis and checked again before final export.
- Prompts are preserved verbatim and hashed; factor-parity assertions verify that only the intended A/B factor changes.
- Successful responses are appended immediately to model-aware JSONL caches.
- Hosted API outputs may drift on a completely fresh request; the notebook therefore treats preserved raw responses—not an assumption of bitwise API determinism—as the reproducible experimental record.
- Raw responses, parsed fields, evidence validation, provider/model identifiers, settings, tokens, costs, latency, retries, and errors are preserved.
- Confidence intervals and paired significance tests are produced for prompt comparisons, model comparisons, and the final system comparison.
- Winner-selection tests are labeled **exploratory**, because the same finite GPT Test benchmark is used to choose the winner. Out-of-fold ensemble evaluation reduces threshold/weight tuning optimism but does not create an independent external validation set.

> `GPT Test ALL` contains operational `No` labels for passages not explicitly labeled as unlearning by a human. Its results are reported descriptively and with a human-labeled-only sensitivity analysis; they do not change any locked decision.


## Recommended run order

Run setup and prompt inspection first with `ALLOW_PAID_API_CALLS = False`.

1. Review the model table and Stage 1 preflight.
2. Set `ALLOW_PAID_API_CALLS = True` and `RUN_STAGE1_GPT_TEST = True`; run the OpenAI, Anthropic, and Google Stage 1 cells separately.
3. Run Stage 1 analysis to lock the examples-column winner.
4. Set `RUN_STAGE2_GPT_TEST = True`; run the three Stage 2 provider cells separately.
5. Run Stage 2 analysis to lock the checklist winner.
6. Run best-model selection and create the selected-model GPT Test manual-review workbook.
7. Set `RUN_BASELINE_GPT_TEST = True`; run the three direct-baseline provider cells.
8. Run threshold/weight optimization, out-of-fold comparison, confidence intervals, and significance tests.
9. Inspect and accept the locked deployment checkpoint.
10. Only then set `RUN_FINAL_GPT_TEST_ALL = True` and run the three final GPT Test ALL provider cells.
11. Run final GPT Test ALL evaluation, reports, validation, and ZIP packaging.

Each successful prompt–text–model combination is cache-addressed independently of dataset name, so identical GPT Test rows are reused automatically in GPT Test ALL. A runtime interruption does not erase completed work.


In [ ]:
# Install/update dependencies once per Colab runtime.
%pip install -q -U \
    scipy pyarrow openpyxl xlsxwriter \
    tqdm tenacity pydantic tabulate matplotlib \
    openai anthropic google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/

In [ ]:
from __future__ import annotations

import copy
import csv
import hashlib
import importlib.metadata
import itertools
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import tempfile
import textwrap
import time
import traceback
import warnings
import zipfile

from collections import defaultdict
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Iterable, Literal, Optional, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pydantic import BaseModel, ConfigDict, Field
from scipy.stats import binomtest
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 240)

print("Python:", sys.version)
print("UTC setup time:", datetime.now(timezone.utc).isoformat())

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
UTC setup time: 2026-08-21T08:27:10.462726+00:00


In [ ]:
# Mount Google Drive.
try:
    from google.colab import drive, userdata

    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except Exception as exc:
    IN_COLAB = False
    print("Not running inside Google Colab or Drive could not be mounted:", exc)

# API keys: Colab Secrets first, then environment variables.
# Secret names expected: OPENAI_API_KEY, ANTHROPIC_API_KEY, GEMINI_API_KEY
def load_secret(name: str) -> Optional[str]:
    value = os.environ.get(name)
    if value:
        return value
    if IN_COLAB:
        try:
            value = userdata.get(name)
            if value:
                os.environ[name] = value
                return value
        except Exception:
            pass
    return None

for _secret_name in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY"]:
    _value = load_secret(_secret_name)
    print(f"{_secret_name}: {'available' if _value else 'NOT SET'}")

Mounted at /content/drive
OPENAI_API_KEY: available
ANTHROPIC_API_KEY: available
GEMINI_API_KEY: available


In [ ]:
# =========================
# USER CONFIGURATION
# =========================

EXPERIMENT_NAME = "paragraph_level_gpt_test_first_v2_20260821"
DRIVE_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Unlearning_Paragraph_Level_LLM_Experiment"
) / EXPERIMENT_NAME

WORKBOOK_PATH_OVERRIDE: Optional[str] = None
EXPECTED_WORKBOOK_FILENAME = "Unlearning_GPT_Test_Paragraph_Aligned_ALL.xlsx"

GPT_TEST_SHEET = "GPT Test"
GPT_TEST_ALL_SHEET = "GPT Test ALL"
CODEBOOK_SHEET = "Codebook (revised)"

# Paid-call gate plus stage-specific switches. Keep all False for setup/preflight.
ALLOW_PAID_API_CALLS = True
RUN_STAGE1_GPT_TEST = True
RUN_STAGE2_GPT_TEST = True
RUN_BASELINE_GPT_TEST = True
RUN_FINAL_GPT_TEST_ALL = True

RUN_OPENAI = True
RUN_ANTHROPIC = True
RUN_GEMINI = True

# Smoke tests only. Use a different EXPERIMENT_NAME whenever ROW_LIMIT is not None.
ROW_LIMIT: Optional[int] = None
REPLICATES_PER_PROMPT = 1
if REPLICATES_PER_PROMPT != 1:
    raise ValueError(
        "This registered pipeline requires REPLICATES_PER_PROMPT=1. Use a separate "
        "replication study rather than mixing repeated calls into model selection."
    )

# Safety limits are applied to each provider job. Set after reviewing preflight.
MAX_NEW_CALLS_PER_JOB: Optional[int] = None
MAX_ESTIMATED_COST_USD_PER_JOB: Optional[float] = None

MAX_OUTPUT_TOKENS = 8000
SNAPSHOT_EVERY_N_NEW_CALLS = 40
MAX_ATTEMPTS_PER_CALL = 6
MAX_SCHEMA_RETRIES = 2
BASE_RETRY_SECONDS = 2.0
MAX_RETRY_SECONDS = 90.0
REQUEST_SLEEP_SECONDS = 0.10
STOP_MODEL_ON_PERMANENT_ERROR = True

# Reproducible analysis settings.
ANALYSIS_RANDOM_SEED = 20260821
N_BOOTSTRAP = 5000
N_PERMUTATIONS = 5000
CI_LEVEL = 0.95
CV_FOLDS = 5
CV_REPEATS_FOR_STABILITY = 5

# Threshold and ensemble search.
THRESHOLD_GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)
WEIGHT_GRID_STEP = 0.05
MIN_RECALL_FOR_PRIMARY_SEARCH = 0.50
MIN_PREDICTED_POSITIVES_FOR_PRIMARY_SEARCH = 5

# Prompt/model selection is now performed on GPT Test only.
PROMPT_SELECTION_DATASET = "GPT Test"
PROMPT_SELECTION_THRESHOLD = 0.50
PROMPT_SELECTION_PRIMARY_METRIC = "precision"

PROMPT_VERSION = "latest_fullpdf_prompt_exact_v1"
OUTPUT_SCHEMA_VERSION = "unlearning_probability_target_evidence_v1"

# Candidate models. The A/B tests run every enabled candidate on GPT Test.
# Keep the same number of candidates per provider if you want a literal unweighted
# mean over all models to also give equal provider weight.
MODEL_CONFIGS = [
    # OpenAI: same low reasoning effort across the GPT-5.6 family.
    {
        "run_name": "openai_gpt_5_6_terra_low",
        "provider": "openai",
        "model": "gpt-5.6-terra",
        "display_name": "GPT-5.6 Terra (low)",
        "reasoning_effort": "low",
        "input_usd_per_1m": 2.00,
        "cached_input_usd_per_1m": 0.20,
        "output_usd_per_1m": 12.00,
        "enabled": RUN_OPENAI,
    },

    # Anthropic: Haiku has thinking off by default; Sonnet/Opus explicitly disable
    # adaptive thinking and use low effort to keep the classification setting controlled.
    {
        "run_name": "anthropic_claude_haiku_4_5_no_thinking",
        "provider": "anthropic",
        "model": "claude-haiku-4-5-20251001",
        "display_name": "Claude Haiku 4.5 (no thinking)",
        "adaptive_thinking_model": False,
        "thinking": "disabled",
        "effort": None,
        "input_usd_per_1m": 1.00,
        "cached_input_usd_per_1m": 0.10,
        "output_usd_per_1m": 5.00,
        "enabled": RUN_ANTHROPIC,
    },
    # Google: same low thinking level; temperature omitted for every candidate.
    {
        "run_name": "google_gemini_3_1_flash_lite_low",
        "provider": "google",
        "model": "gemini-3.1-flash-lite",
        "display_name": "Gemini 3.1 Flash-Lite (low)",
        "thinking_level": "low",
        "input_usd_per_1m": 0.25,
        "cached_input_usd_per_1m": 0.025,
        "output_usd_per_1m": 1.50,
        "enabled": RUN_GEMINI,
    },
    # Optional higher/specialized models can be added here, but changing the candidate
    # set after calls begin requires a new EXPERIMENT_NAME.
]

MODEL_CONFIG_DF = pd.DataFrame(MODEL_CONFIGS)
MODEL_CONFIG_DF = MODEL_CONFIG_DF.loc[MODEL_CONFIG_DF["enabled"]].reset_index(drop=True)
if MODEL_CONFIG_DF.empty:
    raise ValueError("No model configurations are enabled.")
if MODEL_CONFIG_DF["run_name"].duplicated().any():
    raise ValueError("MODEL_CONFIGS contains duplicate run_name values.")

ENABLED_RUN_NAMES = MODEL_CONFIG_DF["run_name"].tolist()
ENABLED_PROVIDERS = MODEL_CONFIG_DF["provider"].drop_duplicates().tolist()
EXPECTED_PROVIDERS = ["openai", "anthropic", "google"]
if sorted(ENABLED_PROVIDERS) != sorted(EXPECTED_PROVIDERS):
    raise ValueError("Primary experiment requires enabled OpenAI, Anthropic, and Google candidates.")

MODEL_BY_RUN_NAME = {
    cfg["run_name"]: cfg for cfg in MODEL_CONFIG_DF.to_dict(orient="records")
}
PROVIDER_MODEL_CONFIGS = {
    provider: [MODEL_BY_RUN_NAME[name] for name in ENABLED_RUN_NAMES if MODEL_BY_RUN_NAME[name]["provider"] == provider]
    for provider in ENABLED_PROVIDERS
}

candidate_counts = {p: len(v) for p, v in PROVIDER_MODEL_CONFIGS.items()}
if len(set(candidate_counts.values())) != 1:
    warnings.warn(
        "Providers have unequal candidate counts. The registered prompt-selection score "
        "still averages models equally, so providers with more candidates receive more weight.",
        stacklevel=2,
    )

random.seed(ANALYSIS_RANDOM_SEED)
np.random.seed(ANALYSIS_RANDOM_SEED)

if ROW_LIMIT is not None:
    warnings.warn(
        "ROW_LIMIT is active. Use a separate EXPERIMENT_NAME for every smoke test.",
        stacklevel=2,
    )

print("Project root:", DRIVE_PROJECT_ROOT)
print("Candidate counts:", candidate_counts)
print("Paid calls enabled:", ALLOW_PAID_API_CALLS)
display(MODEL_CONFIG_DF[[
    "run_name", "provider", "model", "display_name", "enabled"
]])


Project root: /content/drive/MyDrive/Unlearning_Paragraph_Level_LLM_Experiment/paragraph_level_gpt_test_first_v2_20260821
Candidate counts: {'openai': 1, 'anthropic': 1, 'google': 1}
Paid calls enabled: True


,run_name,provider,model,display_name,enabled
0,openai_gpt_5_6_terra_low,openai,gpt-5.6-terra,GPT-5.6 Terra (low),True
1,anthropic_claude_haiku_4_5_no_thinking,anthropic,claude-haiku-4-5-20251001,Claude Haiku 4.5 (no thinking),True
2,google_gemini_3_1_flash_lite_low,google,gemini-3.1-flash-lite,Gemini 3.1 Flash-Lite (low),True


In [ ]:
# Create the experiment directory tree.

SUBDIRS = {
    "inputs": DRIVE_PROJECT_ROOT / "inputs",
    "manifests": DRIVE_PROJECT_ROOT / "manifests",
    "prompts": DRIVE_PROJECT_ROOT / "prompts",
    "raw_jsonl": DRIVE_PROJECT_ROOT / "predictions" / "raw_jsonl",
    "snapshots": DRIVE_PROJECT_ROOT / "predictions" / "snapshots",
    "analysis": DRIVE_PROJECT_ROOT / "analysis",
    "optimization": DRIVE_PROJECT_ROOT / "optimization",
    "inference": DRIVE_PROJECT_ROOT / "inference",
    "audits": DRIVE_PROJECT_ROOT / "audits",
    "reports": DRIVE_PROJECT_ROOT / "reports",
    "figures": DRIVE_PROJECT_ROOT / "figures",
    "logs": DRIVE_PROJECT_ROOT / "logs",
    "checkpoints": DRIVE_PROJECT_ROOT / "checkpoints",
    "exports": DRIVE_PROJECT_ROOT / "exports",
}

for path in [DRIVE_PROJECT_ROOT, *SUBDIRS.values()]:
    path.mkdir(parents=True, exist_ok=True)

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)
    return h.hexdigest()

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if np.isnan(value) else float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (datetime,)):
        return value.isoformat()
    if isinstance(value, BaseModel):
        return value.model_dump()
    if not isinstance(value, (list, dict, tuple, set)):
        try:
            if pd.isna(value):
                return None
        except (TypeError, ValueError):
            pass
    return str(value)

def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(text, encoding="utf-8")
    os.replace(temp, path)

def atomic_write_json(path: Path, obj: Any) -> None:
    atomic_write_text(
        path,
        json.dumps(obj, indent=2, ensure_ascii=False, default=json_default),
    )

def atomic_to_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(temp, index=False)
    os.replace(temp, path)

def atomic_to_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    df.to_parquet(temp, index=False)
    os.replace(temp, path)

def append_jsonl(path: Path, record: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    line = json.dumps(record, ensure_ascii=False, default=json_default)
    with path.open("a", encoding="utf-8") as f:
        f.write(line + "\n")
        f.flush()
        os.fsync(f.fileno())

print("Drive directory tree created.")
for name, path in SUBDIRS.items():
    print(f"  {name:12s} {path}")

Drive directory tree created.
  inputs       /content/drive/MyDrive/Unlearning_Paragraph_Level_LLM_Experiment/paragraph_level_gpt_test_first_v2_20260821/inputs
  manifests    /content/drive/MyDrive/Unlearning_Paragraph_Level_LLM_Experiment/paragraph_level_gpt_test_first_v2_20260821/manifests
  prompts      /content/drive/MyDrive/Unlearning_Paragraph_Level_LLM_Experiment/paragraph_level_gpt_test_first_v2_20260821/prompts
  raw_jsonl    /content/drive/MyDrive/Unlearning_Paragraph_Level_LLM_Experiment/paragraph_level_gpt_test_first_v2_20260821/predictions/raw_jsonl
  snapshots    /content/drive/MyDrive/Unlearning_Paragraph_Level_LLM_Experiment/paragraph_level_gpt_test_first_v2_20260821/predictions/snapshots
  analysis     /content/drive/MyDrive/Unlearning_Paragraph_Level_LLM_Experiment/paragraph_level_gpt_test_first_v2_20260821/analysis
  optimization /content/drive/MyDrive/Unlearning_Paragraph_Level_LLM_Experiment/paragraph_level_gpt_test_first_v2_20260821/optimization
  inference    /co

In [ ]:
# Locate, hash, and archive the source workbook.

def resolve_workbook_path() -> Path:
    candidates: list[Path] = []

    if WORKBOOK_PATH_OVERRIDE:
        candidates.append(Path(WORKBOOK_PATH_OVERRIDE))

    candidates.extend(
        [
            Path("/content") / EXPECTED_WORKBOOK_FILENAME,
            SUBDIRS["inputs"] / EXPECTED_WORKBOOK_FILENAME,
            Path.cwd() / EXPECTED_WORKBOOK_FILENAME,
        ]
    )

    if Path("/content/drive/MyDrive").exists():
        candidates.extend(
            Path("/content/drive/MyDrive").rglob(EXPECTED_WORKBOOK_FILENAME)
        )
    candidates.extend(Path("/content").glob("*.xlsx"))

    seen: set[str] = set()
    clean_candidates: list[Path] = []
    for candidate in candidates:
        key = str(candidate.resolve()) if candidate.exists() else str(candidate)
        if key not in seen:
            seen.add(key)
            clean_candidates.append(candidate)

    exact_matches = [
        p for p in clean_candidates
        if p.exists() and p.name == EXPECTED_WORKBOOK_FILENAME
    ]
    if exact_matches:
        exact_matches.sort(
            key=lambda p: (
                0 if str(p).startswith("/content/") and "/drive/" not in str(p)
                else 1 if str(p).startswith(str(SUBDIRS["inputs"]))
                else 2,
                len(str(p)),
            )
        )
        return exact_matches[0]

    if IN_COLAB:
        from google.colab import files
        print(
            f"Could not find {EXPECTED_WORKBOOK_FILENAME}. "
            "Please upload it in the file picker."
        )
        uploaded = files.upload()
        if EXPECTED_WORKBOOK_FILENAME in uploaded:
            return Path("/content") / EXPECTED_WORKBOOK_FILENAME
        xlsx_names = [name for name in uploaded if name.lower().endswith(".xlsx")]
        if len(xlsx_names) == 1:
            return Path("/content") / xlsx_names[0]

    raise FileNotFoundError(
        f"Could not locate {EXPECTED_WORKBOOK_FILENAME}. "
        "Upload it to /content, place it in the project inputs folder, "
        "or set WORKBOOK_PATH_OVERRIDE."
    )

SOURCE_WORKBOOK_PATH = resolve_workbook_path()
SOURCE_WORKBOOK_SHA256 = sha256_file(SOURCE_WORKBOOK_PATH)

ARCHIVED_WORKBOOK_PATH = SUBDIRS["inputs"] / EXPECTED_WORKBOOK_FILENAME
if (
    not ARCHIVED_WORKBOOK_PATH.exists()
    or sha256_file(ARCHIVED_WORKBOOK_PATH) != SOURCE_WORKBOOK_SHA256
):
    shutil.copy2(SOURCE_WORKBOOK_PATH, ARCHIVED_WORKBOOK_PATH)

SOURCE_MANIFEST = {
    "experiment_name": EXPERIMENT_NAME,
    "resolved_source_path": str(SOURCE_WORKBOOK_PATH),
    "archived_source_path": str(ARCHIVED_WORKBOOK_PATH),
    "source_filename": SOURCE_WORKBOOK_PATH.name,
    "source_sha256": SOURCE_WORKBOOK_SHA256,
    "source_size_bytes": SOURCE_WORKBOOK_PATH.stat().st_size,
    "archived_at_utc": utc_now(),
}
atomic_write_json(SUBDIRS["manifests"] / "source_workbook_manifest.json", SOURCE_MANIFEST)

print(json.dumps(SOURCE_MANIFEST, indent=2))

{
  "experiment_name": "paragraph_level_gpt_test_first_v2_20260821",
  "resolved_source_path": "/content/drive/MyDrive/Unlearning_Paragraph_Level_LLM_Experiment/paragraph_level_gpt_test_first_v2_20260821/inputs/Unlearning_GPT_Test_Paragraph_Aligned_ALL.xlsx",
  "archived_source_path": "/content/drive/MyDrive/Unlearning_Paragraph_Level_LLM_Experiment/paragraph_level_gpt_test_first_v2_20260821/inputs/Unlearning_GPT_Test_Paragraph_Aligned_ALL.xlsx",
  "source_filename": "Unlearning_GPT_Test_Paragraph_Aligned_ALL.xlsx",
  "source_sha256": "a148051e24ccab080b35a2404cebddeff24cd7c0aec9b5af1f45c42a39c85d31",
  "source_size_bytes": 284948,
  "archived_at_utc": "2026-08-21T08:48:45.510374+00:00"
}


## 1. Immutable input loading and paragraph-level dataset checks

The source workbook is copied into the experiment's Drive input folder and then treated as read-only. The entire `Codebook (revised)` sheet—including the objective/preamble above its table header—is hashed before any prompt is rendered. The notebook reads the codebook into memory but never writes a workbook containing a modified version of that sheet.

In [ ]:
# Load the workbook, detect real header rows, and register immutable source hashes.

excel_file = pd.ExcelFile(ARCHIVED_WORKBOOK_PATH)
print("Workbook sheets:", excel_file.sheet_names)

required_sheets = {GPT_TEST_SHEET, GPT_TEST_ALL_SHEET, CODEBOOK_SHEET}
missing_sheets = required_sheets - set(excel_file.sheet_names)
if missing_sheets:
    raise ValueError(f"Missing required workbook sheets: {sorted(missing_sheets)}")

required_gpt_test = {
    "Passage_ID", "Text_Content", "Document",
    "new_gold_unlearning", "new_gold_binary",
}
required_all = {
    "Passage_ID", "Text_Content", "Document",
    "Gold_Unlearning", "Gold_Binary", "Human_Label_Status",
}
required_codebook = {
    "Code", "Definition", "Detection Logic", "Examples",
    "Positive Clarification", "Negative Clarification",
}

def _header_cell_text(value: Any) -> str:
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value).strip()

def read_sheet_with_detected_header(
    workbook_path: Path,
    sheet_name: str,
    required_columns: set[str],
    max_header_rows: int = 25,
) -> tuple[pd.DataFrame, int]:
    preview = pd.read_excel(
        workbook_path,
        sheet_name=sheet_name,
        header=None,
        nrows=max_header_rows,
    )
    for zero_based_row, row in preview.iterrows():
        values = {_header_cell_text(value) for value in row.tolist()}
        if required_columns.issubset(values):
            frame = pd.read_excel(
                workbook_path,
                sheet_name=sheet_name,
                header=int(zero_based_row),
            )
            frame.columns = [_header_cell_text(column) for column in frame.columns]
            return frame, int(zero_based_row)
    raise ValueError(
        f"Could not detect a header row in {sheet_name!r} containing "
        f"{sorted(required_columns)} within the first {max_header_rows} rows."
    )

# Preserve the whole codebook sheet, including any objective/preamble above the header,
# in the immutable hash. The notebook never writes to ARCHIVED_WORKBOOK_PATH.
CODEBOOK_WHOLE_SHEET_RAW = pd.read_excel(
    ARCHIVED_WORKBOOK_PATH,
    sheet_name=CODEBOOK_SHEET,
    header=None,
    dtype=object,
)

def dataframe_content_hash(frame: pd.DataFrame) -> str:
    serializable = frame.copy().fillna("").astype(str)
    payload = serializable.to_csv(index=False, header=False, lineterminator="\n")
    return sha256_text(payload)

CODEBOOK_WHOLE_SHEET_SHA256 = dataframe_content_hash(CODEBOOK_WHOLE_SHEET_RAW)
SOURCE_WORKBOOK_SHA256_AT_LOAD = sha256_file(ARCHIVED_WORKBOOK_PATH)

GPT_TEST_RAW, GPT_TEST_HEADER_ROW = read_sheet_with_detected_header(
    ARCHIVED_WORKBOOK_PATH, GPT_TEST_SHEET, required_gpt_test
)
GPT_TEST_ALL_RAW, GPT_TEST_ALL_HEADER_ROW = read_sheet_with_detected_header(
    ARCHIVED_WORKBOOK_PATH, GPT_TEST_ALL_SHEET, required_all
)
CODEBOOK_DF, CODEBOOK_HEADER_ROW = read_sheet_with_detected_header(
    ARCHIVED_WORKBOOK_PATH, CODEBOOK_SHEET, required_codebook
)

for label, frame, required in [
    (GPT_TEST_SHEET, GPT_TEST_RAW, required_gpt_test),
    (GPT_TEST_ALL_SHEET, GPT_TEST_ALL_RAW, required_all),
    (CODEBOOK_SHEET, CODEBOOK_DF, required_codebook),
]:
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{label} is missing required columns: {sorted(missing)}")

IMMUTABLE_SOURCE_MANIFEST = {
    "source_workbook_path": str(ARCHIVED_WORKBOOK_PATH),
    "source_workbook_sha256": SOURCE_WORKBOOK_SHA256_AT_LOAD,
    "codebook_whole_sheet_sha256": CODEBOOK_WHOLE_SHEET_SHA256,
    "codebook_sheet_name": CODEBOOK_SHEET,
    "codebook_zero_based_header_row": CODEBOOK_HEADER_ROW,
    "gpt_test_zero_based_header_row": GPT_TEST_HEADER_ROW,
    "gpt_test_all_zero_based_header_row": GPT_TEST_ALL_HEADER_ROW,
    "loaded_at_utc": utc_now(),
    "read_only_policy": (
        "The source workbook and Codebook (revised) sheet are input-only. "
        "All experiment outputs are written to separate Drive files."
    ),
}
atomic_write_json(
    SUBDIRS["manifests"] / "immutable_source_manifest.json",
    IMMUTABLE_SOURCE_MANIFEST,
)

print("GPT Test shape:", GPT_TEST_RAW.shape)
print("GPT Test ALL shape:", GPT_TEST_ALL_RAW.shape)
print("Codebook shape:", CODEBOOK_DF.shape)
print("Codebook whole-sheet SHA-256:", CODEBOOK_WHOLE_SHEET_SHA256)
print("Source workbook SHA-256:", SOURCE_WORKBOOK_SHA256_AT_LOAD)


Workbook sheets: ['GPT Test', 'Codebook (revised)', 'GPT Test ALL', 'Extraction Audit']
GPT Test shape: (84, 36)
GPT Test ALL shape: (677, 23)
Codebook shape: (14, 6)
Codebook whole-sheet SHA-256: 892a055512cdecccc191a768792ad1182933fb77ca3df31b2a2e4e470549d1e6
Source workbook SHA-256: a148051e24ccab080b35a2404cebddeff24cd7c0aec9b5af1f45c42a39c85d31


In [ ]:
# Normalize passage records conservatively and preserve human-label metadata.

import unicodedata

def clean_cell(value: Any) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value).strip()

def normalized_text_for_key(text: str) -> str:
    # Used only for identity/cache reuse. The original Text_Content is sent to models.
    text = unicodedata.normalize("NFKC", clean_cell(text))
    replacements = {
        "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
        "\u2013": "-", "\u2014": "-", "\u2212": "-",
        "\u00a0": " ", "\u200b": "",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return re.sub(r"\s+", " ", text).strip()

def repair_unicode(text: Any) -> str:
    # Matches the latest FullPDF pipeline's conservative prompt cleaning behavior.
    return unicodedata.normalize("NFKC", clean_cell(text)).replace("\u00ad", "")

def normalize_yes_no(value: Any) -> Optional[int]:
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(value, (bool, np.bool_)):
        return int(value)
    if isinstance(value, (int, float, np.integer, np.floating)):
        if float(value) in (0.0, 1.0):
            return int(float(value))
    text = str(value).strip().lower()
    if text in {"yes", "y", "true", "1", "unlearning"}:
        return 1
    if text in {"no", "n", "false", "0", "not unlearning"}:
        return 0
    return None

def normalize_label_display(value: Any) -> str:
    parsed = normalize_yes_no(value)
    return "Yes" if parsed == 1 else "No" if parsed == 0 else "Not labeled"

def normalize_bool_flag(value: Any) -> bool:
    parsed = normalize_yes_no(value)
    return bool(parsed) if parsed is not None else False

def column_or_blank(frame: pd.DataFrame, name: str) -> pd.Series:
    return (
        frame[name].map(clean_cell)
        if name in frame.columns
        else pd.Series("", index=frame.index, dtype="object")
    )

def prepare_dataset(
    raw: pd.DataFrame,
    dataset_name: str,
    gold_binary_col: str,
    gold_text_col: str,
) -> pd.DataFrame:
    frame = raw.copy()
    frame.columns = [clean_cell(c) for c in frame.columns]

    out = pd.DataFrame({
        "dataset": dataset_name,
        "passage_id": frame["Passage_ID"].map(clean_cell),
        "text": frame["Text_Content"].map(clean_cell),
        "document": frame["Document"].map(clean_cell),
        "source_file": column_or_blank(frame, "Source_File"),
        "reference": column_or_blank(frame, "Reference"),
        "original_number": column_or_blank(frame, "Original_Number"),
    })

    numeric_gold = frame[gold_binary_col].map(normalize_yes_no)
    text_gold = frame[gold_text_col].map(normalize_yes_no)
    out["gold"] = numeric_gold.where(numeric_gold.notna(), text_gold)
    out["gold_label"] = out["gold"].map(
        lambda x: "Yes" if x == 1 else "No" if x == 0 else ""
    )

    optional_map = {
        "human_label_status": "Human_Label_Status",
        "human_labelers": "Human_Labelers",
        "matched_gpt_test_ids": "Matched_GPT_Test_IDs",
        "label_basis": "Label_Basis",
        "adjudication_rationale": "Adjudication_Rationale",
        "paragraph_type": "Paragraph_Type",
        "original_labeler": "Original_Labeler",
        "label_history": "Label_History",
        "selected_codes": "Selected_Codes",
        "latest_label_source": "Latest_Label_Source",
        "rationale_source": "Rationale_Source",
        "source_records": "Source_Records",
        "old_gold_unlearning": "old_gold_unlearning",
        "new_gold_unlearning": "new_gold_unlearning",
        "anmol_unlearning_raw": "anmol_unlearning",
        "prerana_unlearning_raw": "prerana_unlearning",
        "kyle_unlearning_raw": "kyle_unlearning",
    }
    for target, source in optional_map.items():
        out[target] = column_or_blank(frame, source)

    out["anmol_unlearning"] = out["anmol_unlearning_raw"].map(normalize_label_display)
    out["prerana_unlearning"] = out["prerana_unlearning_raw"].map(normalize_label_display)
    out["kyle_unlearning"] = out["kyle_unlearning_raw"].map(normalize_label_display)

    review_source = (
        "Extraction_Review_Flag"
        if "Extraction_Review_Flag" in frame.columns
        else "Review_Flag"
    )
    out["review_flag"] = (
        frame[review_source].map(normalize_bool_flag)
        if review_source in frame.columns
        else pd.Series(False, index=frame.index)
    )
    review_issue_source = (
        "Extraction_Review_Issue"
        if "Extraction_Review_Issue" in frame.columns
        else "Review_Issue"
    )
    out["review_issue"] = (
        frame[review_issue_source].map(clean_cell)
        if review_issue_source in frame.columns
        else pd.Series("", index=frame.index)
    )

    out["normalized_text"] = out["text"].map(normalized_text_for_key)
    out["text_sha256"] = out["normalized_text"].map(sha256_text)
    out["text_char_count"] = out["text"].str.len()
    out["text_word_count"] = out["text"].str.split().str.len()

    if ROW_LIMIT is not None:
        out = out.head(int(ROW_LIMIT)).copy()
    return out.reset_index(drop=True)

GPT_TEST_DF = prepare_dataset(
    GPT_TEST_RAW,
    dataset_name="GPT Test",
    gold_binary_col="new_gold_binary",
    gold_text_col="new_gold_unlearning",
)
GPT_TEST_ALL_DF = prepare_dataset(
    GPT_TEST_ALL_RAW,
    dataset_name="GPT Test ALL",
    gold_binary_col="Gold_Binary",
    gold_text_col="Gold_Unlearning",
)

def dataset_qc(frame: pd.DataFrame) -> list[str]:
    errors: list[str] = []
    name = frame["dataset"].iloc[0] if len(frame) else "<empty>"
    if frame.empty:
        errors.append(f"{name}: dataset is empty")
    if (frame["passage_id"] == "").any():
        errors.append(f"{name}: blank Passage_ID values")
    if (frame["text"] == "").any():
        errors.append(f"{name}: blank Text_Content values")
    if frame["passage_id"].duplicated().any():
        ids = frame.loc[frame["passage_id"].duplicated(False), "passage_id"].tolist()
        errors.append(f"{name}: duplicate Passage_ID values: {ids[:20]}")
    if frame["gold"].isna().any():
        ids = frame.loc[frame["gold"].isna(), "passage_id"].tolist()
        errors.append(f"{name}: unparseable gold labels: {ids[:20]}")
    if not set(frame["gold"].dropna().astype(int).unique()).issubset({0, 1}):
        errors.append(f"{name}: gold labels outside {{0,1}}")
    return errors

QC_ERRORS = dataset_qc(GPT_TEST_DF) + dataset_qc(GPT_TEST_ALL_DF)

# Exact normalized passages must not carry conflicting gold labels across sheets.
combined_gold = pd.concat(
    [
        GPT_TEST_DF[["dataset", "passage_id", "text_sha256", "gold"]],
        GPT_TEST_ALL_DF[["dataset", "passage_id", "text_sha256", "gold"]],
    ],
    ignore_index=True,
)
conflicting_hashes = combined_gold.groupby("text_sha256")["gold"].nunique()
conflicting_hashes = conflicting_hashes[conflicting_hashes > 1]
if not conflicting_hashes.empty:
    conflicts = combined_gold[
        combined_gold["text_sha256"].isin(conflicting_hashes.index)
    ].sort_values(["text_sha256", "dataset", "passage_id"])
    atomic_to_csv(conflicts, SUBDIRS["manifests"] / "conflicting_gold_labels.csv")
    QC_ERRORS.append(
        f"Conflicting gold labels found for {len(conflicting_hashes)} exact texts."
    )

if QC_ERRORS:
    raise ValueError("Hard data-quality checks failed:\n- " + "\n- ".join(QC_ERRORS))

GPT_TEST_DF["gold"] = GPT_TEST_DF["gold"].astype(int)
GPT_TEST_ALL_DF["gold"] = GPT_TEST_ALL_DF["gold"].astype(int)
PASSAGE_TABLES = {"GPT Test": GPT_TEST_DF, "GPT Test ALL": GPT_TEST_ALL_DF}

print("Hard quality checks passed.")
for name, frame in PASSAGE_TABLES.items():
    print(name, {
        "rows": len(frame),
        "yes": int(frame["gold"].sum()),
        "no": int((frame["gold"] == 0).sum()),
        "review_flagged": int(frame["review_flag"].sum()),
        "unique_texts": int(frame["text_sha256"].nunique()),
    })


Hard quality checks passed.
GPT Test {'rows': 84, 'yes': 42, 'no': 42, 'review_flagged': 34, 'unique_texts': 84}
GPT Test ALL {'rows': 677, 'yes': 38, 'no': 639, 'review_flagged': 35, 'unique_texts': 677}


In [ ]:
# Build the exact-text crosswalk and archive normalized input manifests.

crosswalk = GPT_TEST_DF[
    ["passage_id", "text_sha256", "gold", "document"]
].merge(
    GPT_TEST_ALL_DF[
        ["passage_id", "text_sha256", "gold", "document"]
    ],
    on="text_sha256",
    how="left",
    suffixes=("_gpt_test", "_all"),
    indicator=True,
)

crosswalk["exact_text_reusable"] = crosswalk["_merge"].eq("both")
crosswalk = crosswalk.drop(columns=["_merge"])

for name, frame in PASSAGE_TABLES.items():
    safe_name = name.lower().replace(" ", "_")
    atomic_to_csv(frame, SUBDIRS["manifests"] / f"{safe_name}_normalized.csv")
    atomic_to_parquet(frame, SUBDIRS["manifests"] / f"{safe_name}_normalized.parquet")

atomic_to_csv(crosswalk, SUBDIRS["manifests"] / "gpt_test_to_all_exact_text_crosswalk.csv")

DATASET_SUMMARY_DF = pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": len(frame),
            "positive_rows": int(frame["gold"].sum()),
            "negative_rows": int((frame["gold"] == 0).sum()),
            "positive_prevalence": float(frame["gold"].mean()),
            "unique_normalized_texts": int(frame["text_sha256"].nunique()),
            "review_flagged_rows": int(frame["review_flag"].sum()),
            "human_labeled_rows": int(
                (~frame["human_label_status"].str.lower().eq("unlabeled")).sum()
            ),
        }
        for name, frame in PASSAGE_TABLES.items()
    ]
)
atomic_to_csv(DATASET_SUMMARY_DF, SUBDIRS["manifests"] / "dataset_summary.csv")

print(DATASET_SUMMARY_DF.to_string(index=False))
print(
    "\nExact GPT Test rows reusable in GPT Test ALL:",
    int(crosswalk["exact_text_reusable"].sum()),
    "of",
    len(crosswalk),
)

     dataset  rows  positive_rows  negative_rows  positive_prevalence  unique_normalized_texts  review_flagged_rows  human_labeled_rows
    GPT Test    84             42             42              0.50000                       84                   34                  84
GPT Test ALL   677             38            639              0.05613                      677                   35                  79

Exact GPT Test rows reusable in GPT Test ALL: 79 of 84


In [ ]:
# Render the supplied codebook exactly as the latest FullPDF prompt pipeline does.
# This operates on an in-memory copy only; no workbook sheet is edited or rewritten.

CODEBOOK_WORKING = CODEBOOK_DF.copy()
CODEBOOK_WORKING.columns = [clean_cell(c) for c in CODEBOOK_WORKING.columns]
CODEBOOK_WORKING = CODEBOOK_WORKING[
    CODEBOOK_WORKING["Code"].map(clean_cell).ne("")
].reset_index(drop=True)

CODEBOOK_DATA_SHA256 = dataframe_content_hash(CODEBOOK_WORKING)

def render_codebook(codebook: pd.DataFrame, include_examples: bool) -> str:
    columns = [
        "Code",
        "Definition",
        "Detection Logic",
        "Positive Clarification",
        "Negative Clarification",
    ]
    if include_examples:
        columns.insert(3, "Examples")
    sections: list[str] = []
    for _, row in codebook.iterrows():
        code_name = repair_unicode(row.get("Code", ""))
        if not code_name:
            continue
        parts = [f"CODE: {code_name}"]
        for column in columns[1:]:
            value = repair_unicode(row.get(column, ""))
            if value:
                parts.append(f"{column.upper()}:\n{value}")
        sections.append("\n".join(parts))
    return "\n\n---\n\n".join(sections)

CODEBOOK_NO_EXAMPLES_TEXT = render_codebook(CODEBOOK_WORKING, False)
CODEBOOK_WITH_EXAMPLES_TEXT = render_codebook(CODEBOOK_WORKING, True)

atomic_write_text(
    SUBDIRS["prompts"] / "codebook_without_examples.txt",
    CODEBOOK_NO_EXAMPLES_TEXT,
)
atomic_write_text(
    SUBDIRS["prompts"] / "codebook_with_examples.txt",
    CODEBOOK_WITH_EXAMPLES_TEXT,
)

# Audit possible evaluation overlap with native codebook examples. This is a
# sensitivity flag only; labels and passages are never changed.
def content_tokens(text: str) -> set[str]:
    return set(re.findall(r"[a-z0-9]+", normalized_text_for_key(text).lower()))

example_cells = [
    clean_cell(value)
    for value in CODEBOOK_WORKING["Examples"].tolist()
    if clean_cell(value)
]
example_norms = [normalized_text_for_key(value).lower() for value in example_cells]
example_token_sets = [content_tokens(value) for value in example_cells]

def example_overlap_for_text(text: str) -> dict[str, Any]:
    norm = normalized_text_for_key(text).lower()
    tokens = content_tokens(text)
    exact_or_contained = False
    matched_cell_index: Optional[int] = None
    max_jaccard = 0.0
    max_jaccard_cell: Optional[int] = None

    for idx, (example_norm, example_tokens) in enumerate(
        zip(example_norms, example_token_sets)
    ):
        if len(norm) >= 40 and (norm in example_norm or example_norm in norm):
            exact_or_contained = True
            matched_cell_index = idx
        union = tokens | example_tokens
        jaccard = len(tokens & example_tokens) / len(union) if union else 0.0
        if jaccard > max_jaccard:
            max_jaccard = jaccard
            max_jaccard_cell = idx

    high_similarity = len(tokens) >= 8 and max_jaccard >= 0.80
    return {
        "example_exact_or_contained": exact_or_contained,
        "example_max_token_jaccard": max_jaccard,
        "example_overlap_risk": bool(exact_or_contained or high_similarity),
        "matched_example_cell_number": (
            matched_cell_index + 1
            if matched_cell_index is not None
            else max_jaccard_cell + 1
            if max_jaccard_cell is not None
            else None
        ),
    }

EXAMPLE_LEAKAGE_AUDITS: dict[str, pd.DataFrame] = {}
for dataset_name, frame in list(PASSAGE_TABLES.items()):
    audit = pd.concat(
        [
            frame[["dataset", "passage_id", "text_sha256", "gold", "document"]]
            .reset_index(drop=True),
            pd.DataFrame([example_overlap_for_text(text) for text in frame["text"]]),
        ],
        axis=1,
    )
    EXAMPLE_LEAKAGE_AUDITS[dataset_name] = audit
    safe_name = dataset_name.lower().replace(" ", "_")
    atomic_to_csv(
        audit,
        SUBDIRS["manifests"] / f"{safe_name}_codebook_example_overlap_audit.csv",
    )
    enriched = frame.merge(
        audit[["passage_id", "example_overlap_risk", "example_max_token_jaccard"]],
        on="passage_id",
        how="left",
        validate="one_to_one",
    )
    PASSAGE_TABLES[dataset_name] = enriched
    print(
        f"{dataset_name}: {int(audit['example_overlap_risk'].sum())} "
        "rows flagged for possible native-example overlap."
    )

GPT_TEST_DF = PASSAGE_TABLES["GPT Test"]
GPT_TEST_ALL_DF = PASSAGE_TABLES["GPT Test ALL"]

CODEBOOK_RENDER_MANIFEST = {
    "whole_sheet_sha256": CODEBOOK_WHOLE_SHEET_SHA256,
    "parsed_codebook_sha256": CODEBOOK_DATA_SHA256,
    "without_examples_sha256": sha256_text(CODEBOOK_NO_EXAMPLES_TEXT),
    "with_examples_sha256": sha256_text(CODEBOOK_WITH_EXAMPLES_TEXT),
    "renderer_source": "Exact renderer copied from Unlearning_Final_FullPDF_AB_Evaluation_Pipeline.ipynb",
}
atomic_write_json(
    SUBDIRS["manifests"] / "codebook_render_manifest.json",
    CODEBOOK_RENDER_MANIFEST,
)


GPT Test: 1 rows flagged for possible native-example overlap.
GPT Test ALL: 1 rows flagged for possible native-example overlap.


## 2. Exact latest prompt scaffold and controlled factors

The system prompt, direct-only baseline instruction, checklist, output instruction, codebook renderer, and target-passage boundaries below are copied verbatim from `Unlearning_Final_FullPDF_AB_Evaluation_Pipeline.ipynb`. Prompt hashes and parity assertions make unintended prompt drift visible before API execution.

In [ ]:
# Exact prompt construction from Unlearning_Final_FullPDF_AB_Evaluation_Pipeline.ipynb.

TARGET_VALUES = [
    "leadership",
    "laws_plans_policies",
    "capabilities",
    "funds_resources",
    "misc_organizational",
    "none",
]

SYSTEM_PROMPT = """
You are a careful research annotator studying organizational unlearning in U.S. government
disaster policy. Classify only the supplied target passage. Do not use outside knowledge,
surrounding paragraphs, document title, gold labels, annotator rationales, or earlier model
outputs. Return a calibrated probability that the passage itself meets the active instruction.
Use the requested JSON schema exactly.
""".strip()

DIRECT_BASELINE_INSTRUCTION = """
Decide whether this passage demonstrates organizational unlearning. Use the ordinary research
meaning of unlearning: an institution recognizes that an established assumption, policy,
practice, routine, or technical system is inadequate and deliberately moves away from it,
rather than merely learning something new or making an additive improvement.
""".strip()

CHECKLIST_TEXT = """
Apply this checklist in order:
1. PRIOR ITEM: Does the passage identify an existing assumption, policy, plan, practice,
   routine, governance arrangement, or technical system?
2. INADEQUACY: Does it present that prior item as obsolete, harmful, failed, or misaligned?
3. SUBTRACTION / DISCONTINUITY: Does it call for abandoning, replacing, dismantling, or
   fundamentally rethinking the prior item?
4. EXCLUSIONS: Do not count problem diagnosis alone, ordinary implementation, routine
   updating, additional resources/capacity, a new tool that leaves the underlying logic
   unchanged, or generic "lessons learned" language.
5. TARGET: If and only if Unlearning=Yes, assign the closest target category. Otherwise use
   target_category="none".
A Yes classification normally requires steps 1-3 and must not be explained only by an item
in step 4.
""".strip()

OUTPUT_INSTRUCTION = """
Return:
- unlearning_probability: a number from 0 through 1 representing P(Unlearning=Yes);
- unlearning_label: "Yes" or "No";
- target_category: one of leadership, laws_plans_policies, capabilities,
  funds_resources, misc_organizational, none;
- evidence_quote: the shortest exact quote from the target passage that supports the label,
  or an empty string when no exact evidentiary phrase exists;
- rationale: a concise explanation grounded only in the target passage.

The binary label should ordinarily be Yes when probability is at least 0.50 and No otherwise.
Do not wrap JSON in Markdown.
""".strip()

PROMPT_VARIANTS = pd.DataFrame([
    {
        "prompt_name": "direct_target_only",
        "uses_codebook": False,
        "includes_examples": False,
        "includes_checklist": False,
        "stage": "baseline",
        "description": "Direct target-passage baseline with no codebook or checklist.",
    },
    {
        "prompt_name": "codebook_no_examples_no_checklist",
        "uses_codebook": True,
        "includes_examples": False,
        "includes_checklist": False,
        "stage": "stage1_examples_ab",
        "description": "Supplied revised codebook excluding Examples; no checklist.",
    },
    {
        "prompt_name": "codebook_with_examples_no_checklist",
        "uses_codebook": True,
        "includes_examples": True,
        "includes_checklist": False,
        "stage": "stage1_examples_ab",
        "description": "Supplied revised codebook including Examples; no checklist.",
    },
    {
        "prompt_name": "codebook_no_examples_with_checklist",
        "uses_codebook": True,
        "includes_examples": False,
        "includes_checklist": True,
        "stage": "stage2_checklist_ab",
        "description": "Supplied revised codebook excluding Examples; checklist included.",
    },
    {
        "prompt_name": "codebook_with_examples_with_checklist",
        "uses_codebook": True,
        "includes_examples": True,
        "includes_checklist": True,
        "stage": "stage2_checklist_ab",
        "description": "Supplied revised codebook including Examples; checklist included.",
    },
])

def prompt_variant_record(prompt_name: str) -> dict[str, Any]:
    match = PROMPT_VARIANTS.loc[PROMPT_VARIANTS["prompt_name"].eq(prompt_name)]
    if len(match) != 1:
        raise KeyError(f"Unknown prompt variant: {prompt_name}")
    return match.iloc[0].to_dict()

def build_prompt(
    passage_text: str,
    prompt_name: str,
    codebook: pd.DataFrame = CODEBOOK_WORKING,
) -> tuple[str, str]:
    variant = prompt_variant_record(prompt_name)
    sections: list[str] = []
    if variant["uses_codebook"]:
        sections.append(
            "Use the following revised codebook as the governing definition:\n\n"
            + render_codebook(codebook, bool(variant["includes_examples"]))
        )
    else:
        sections.append(DIRECT_BASELINE_INSTRUCTION)
    if variant["includes_checklist"]:
        sections.append(CHECKLIST_TEXT)
    sections.append(OUTPUT_INSTRUCTION)
    sections.append(
        "TARGET PASSAGE START\n"
        + repair_unicode(passage_text)
        + "\nTARGET PASSAGE END"
    )
    return SYSTEM_PROMPT, "\n\n".join(sections)

@dataclass(frozen=True)
class PromptSpec:
    prompt_name: str
    uses_codebook: bool
    include_examples: bool
    include_checklist: bool
    stage: str
    description: str
    system_prompt: str
    user_template: str
    prompt_hash: str


def make_prompt_spec(prompt_name: str) -> PromptSpec:
    row = prompt_variant_record(prompt_name)
    system_prompt, user_prompt = build_prompt("{{TARGET_PASSAGE}}", prompt_name)
    prompt_hash = sha256_text(
        json.dumps(
            {
                "prompt_version": PROMPT_VERSION,
                "system_prompt": system_prompt,
                "user_template": user_prompt,
            },
            sort_keys=True,
            ensure_ascii=False,
        )
    )
    return PromptSpec(
        prompt_name=prompt_name,
        uses_codebook=bool(row["uses_codebook"]),
        include_examples=bool(row["includes_examples"]),
        include_checklist=bool(row["includes_checklist"]),
        stage=str(row["stage"]),
        description=str(row["description"]),
        system_prompt=system_prompt,
        user_template=user_prompt,
        prompt_hash=prompt_hash,
    )

PROMPT_REGISTRY = {
    name: make_prompt_spec(name)
    for name in PROMPT_VARIANTS["prompt_name"]
}

def render_user_prompt(spec: PromptSpec, passage_text: str) -> str:
    return spec.user_template.replace("{{TARGET_PASSAGE}}", repair_unicode(passage_text))

STAGE1_PROMPTS = [
    PROMPT_REGISTRY["codebook_no_examples_no_checklist"],
    PROMPT_REGISTRY["codebook_with_examples_no_checklist"],
]
DIRECT_BASELINE_SPEC = PROMPT_REGISTRY["direct_target_only"]

# A/B parity assertions: only the intended codebook Examples factor changes in Stage 1.
sample_text = GPT_TEST_DF.iloc[0]["text"]
for spec in PROMPT_REGISTRY.values():
    user = render_user_prompt(spec, sample_text)
    if user.count("TARGET PASSAGE START") != 1 or user.count("TARGET PASSAGE END") != 1:
        raise AssertionError(f"Target boundary failure in {spec.prompt_name}")
    if user.count(repair_unicode(sample_text)) != 1:
        raise AssertionError(f"Target text is missing or duplicated in {spec.prompt_name}")
    if spec.include_checklist != (CHECKLIST_TEXT in user):
        raise AssertionError(f"Checklist factor mismatch in {spec.prompt_name}")
    if spec.uses_codebook != ("Use the following revised codebook" in user):
        raise AssertionError(f"Codebook factor mismatch in {spec.prompt_name}")

stage1_a = STAGE1_PROMPTS[0]
stage1_b = STAGE1_PROMPTS[1]
a_user = render_user_prompt(stage1_a, sample_text)
b_user = render_user_prompt(stage1_b, sample_text)
if a_user.replace(CODEBOOK_NO_EXAMPLES_TEXT, "<CODEBOOK>") != b_user.replace(
    CODEBOOK_WITH_EXAMPLES_TEXT, "<CODEBOOK>"
):
    raise AssertionError("Stage 1 prompt variants differ outside the rendered codebook.")

PROMPT_MANIFEST_DF = pd.DataFrame([
    {
        "prompt_name": spec.prompt_name,
        "stage": spec.stage,
        "uses_codebook": spec.uses_codebook,
        "includes_examples": spec.include_examples,
        "includes_checklist": spec.include_checklist,
        "description": spec.description,
        "system_prompt_sha256": sha256_text(spec.system_prompt),
        "user_template_sha256": sha256_text(spec.user_template),
        "prompt_hash": spec.prompt_hash,
        "system_prompt": spec.system_prompt,
        "user_template": spec.user_template,
    }
    for spec in PROMPT_REGISTRY.values()
])
atomic_to_csv(PROMPT_MANIFEST_DF, SUBDIRS["prompts"] / "prompt_manifest.csv")
for spec in PROMPT_REGISTRY.values():
    atomic_write_text(
        SUBDIRS["prompts"] / f"{spec.prompt_name}__system.txt",
        spec.system_prompt,
    )
    atomic_write_text(
        SUBDIRS["prompts"] / f"{spec.prompt_name}__user_template.txt",
        spec.user_template,
    )

display(PROMPT_MANIFEST_DF[[
    "prompt_name", "uses_codebook", "includes_examples",
    "includes_checklist", "prompt_hash"
]])


,prompt_name,uses_codebook,includes_examples,includes_checklist,prompt_hash
0,direct_target_only,False,False,False,fc9bde8ab2ec14ac09d20a61fab6c41a050cba9dae14a01a90c196bf084ee997
1,codebook_no_examples_no_checklist,True,False,False,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe
2,codebook_with_examples_no_checklist,True,True,False,c81e46faac5d0f8d24ff99cf635f491a89e69c02d901ccedbc210e3f59965053
3,codebook_no_examples_with_checklist,True,False,True,5ef0ed097ed63718a949e101ae68ad12b372d8fb0b0f5d1f6902d6007c2fa4c9
4,codebook_with_examples_with_checklist,True,True,True,faf42ab8c21b2d0a6a189db0d06be8ebec7afdafd1d13fae352e0795ae378e0f


### Prompt leakage interpretation

The notebook does **not** silently remove passages that overlap the codebook's native `Examples` column, because the requested primary comparison is on the full `GPT Test ALL` set. Instead it:

- saves exact/containment and token-overlap audits,
- reports the requested full-set result,
- repeats the analysis after excluding rows flagged for possible example leakage.

A large difference between the full and leakage-safe results should be treated as evidence that the examples condition may be benefiting from benchmark overlap rather than generalizable guidance.

In [ ]:
# Shared provider-neutral structured-output contract.
# Numeric range constraints are checked semantically rather than embedded in the
# schema, avoiding provider-specific rejection of minimum/maximum keywords.

class UnlearningOutput(BaseModel):
    model_config = ConfigDict(extra="forbid")

    unlearning_probability: float
    unlearning_label: Literal["Yes", "No"]
    target_category: Literal[
        "leadership",
        "laws_plans_policies",
        "capabilities",
        "funds_resources",
        "misc_organizational",
        "none",
    ]
    evidence_quote: str
    rationale: str

UNSUPPORTED_SCHEMA_KEYS = {
    "minimum", "maximum", "exclusiveMinimum", "exclusiveMaximum",
    "multipleOf", "default", "examples", "$comment",
}

def sanitize_json_schema(value: Any) -> Any:
    if isinstance(value, dict):
        return {
            key: sanitize_json_schema(child)
            for key, child in value.items()
            if key not in UNSUPPORTED_SCHEMA_KEYS
        }
    if isinstance(value, list):
        return [sanitize_json_schema(child) for child in value]
    return value

RAW_OUTPUT_JSON_SCHEMA = UnlearningOutput.model_json_schema()
OUTPUT_JSON_SCHEMA = sanitize_json_schema(RAW_OUTPUT_JSON_SCHEMA)
OUTPUT_SCHEMA_SHA256 = sha256_text(
    json.dumps(OUTPUT_JSON_SCHEMA, sort_keys=True, ensure_ascii=False)
)
atomic_write_json(
    SUBDIRS["manifests"] / "structured_output_schema.json",
    OUTPUT_JSON_SCHEMA,
)

class SemanticValidationError(RuntimeError):
    pass

class ProviderRefusalError(RuntimeError):
    pass

def normalize_for_evidence_match(value: Any) -> str:
    return normalized_text_for_key(clean_cell(value)).lower()

def validate_semantic_output(
    parsed: UnlearningOutput,
    passage_text: str,
) -> dict[str, Any]:
    probability = float(parsed.unlearning_probability)
    if not np.isfinite(probability) or not 0.0 <= probability <= 1.0:
        raise SemanticValidationError(
            f"unlearning_probability must be finite and in [0,1]; got {probability!r}"
        )

    label = clean_cell(parsed.unlearning_label)
    derived_label = "Yes" if probability >= 0.50 else "No"
    target = clean_cell(parsed.target_category)
    evidence = clean_cell(parsed.evidence_quote)
    rationale = clean_cell(parsed.rationale)

    target_label_inconsistent = (
        (label == "No" and target != "none")
        or (label == "Yes" and target == "none")
    )
    evidence_valid = (
        evidence == ""
        or normalize_for_evidence_match(evidence)
        in normalize_for_evidence_match(passage_text)
    )

    return {
        "unlearning_probability": probability,
        "unlearning_label": label,
        "predicted_label_at_0_5": int(probability >= 0.50),
        "label_probability_inconsistent": label != derived_label,
        "target_category": target,
        "target_label_inconsistent": bool(target_label_inconsistent),
        "evidence_quote": evidence,
        "evidence_quote_valid": bool(evidence_valid),
        "rationale": rationale,
    }

def safe_model_dump(obj: Any) -> Any:
    """
    Serialize SDK response objects for audit logging.

    OpenAI responses.parse() returns ParsedResponse objects containing the
    user-supplied Pydantic model in nested `parsed` fields. Current versions
    of the OpenAI SDK can emit PydanticSerializationUnexpectedValue warnings
    when model_dump() serializes those fields.

    warnings=False suppresses that SDK/Pydantic serialization warning without
    changing the parsed prediction used by the experiment.
    """
    if obj is None:
        return None

    model_dump = getattr(obj, "model_dump", None)
    if callable(model_dump):
        try:
            return model_dump(
                mode="json",
                warnings=False,
            )
        except TypeError:
            # Compatibility with Pydantic/SDK objects that do not expose
            # the warnings argument.
            try:
                return model_dump(mode="json")
            except Exception:
                pass
        except Exception:
            pass

    for method_name in ["to_dict", "to_json_dict"]:
        method = getattr(obj, method_name, None)
        if callable(method):
            try:
                return method()
            except Exception:
                pass

    try:
        return json.loads(json.dumps(obj, default=json_default))
    except Exception:
        return repr(obj)

print(json.dumps(OUTPUT_JSON_SCHEMA, indent=2)[:2500])
print("Schema SHA-256:", OUTPUT_SCHEMA_SHA256)


{
  "additionalProperties": false,
  "properties": {
    "unlearning_probability": {
      "title": "Unlearning Probability",
      "type": "number"
    },
    "unlearning_label": {
      "enum": [
        "Yes",
        "No"
      ],
      "title": "Unlearning Label",
      "type": "string"
    },
    "target_category": {
      "enum": [
        "leadership",
        "laws_plans_policies",
        "capabilities",
        "funds_resources",
        "misc_organizational",
        "none"
      ],
      "title": "Target Category",
      "type": "string"
    },
    "evidence_quote": {
      "title": "Evidence Quote",
      "type": "string"
    },
    "rationale": {
      "title": "Rationale",
      "type": "string"
    }
  },
  "required": [
    "unlearning_probability",
    "unlearning_label",
    "target_category",
    "evidence_quote",
    "rationale"
  ],
  "title": "UnlearningOutput",
  "type": "object"
}
Schema SHA-256: 8fc8adb38acc4454f07cd31d52dcb896c52b90b91de6f4ca081b869a34cf8984

## 3. Provider-isolated structured prediction and restart-safe caching

Each provider has a separate wrapper and separate execution cell. Successful responses are appended immediately to JSONL with `fsync`; malformed output is retried and logged rather than converted to a negative prediction. Run keys include the model configuration, prompt hash, schema hash, normalized paragraph hash, and replicate number.

In [ ]:
# Provider-isolated clients and structured prediction wrappers.

from openai import OpenAI
from anthropic import Anthropic
from google import genai
from google.genai import types as gemini_types

_OPENAI_CLIENT: Optional[OpenAI] = None
_ANTHROPIC_CLIENT: Optional[Anthropic] = None
_GOOGLE_CLIENT: Any = None

def get_openai_client() -> OpenAI:
    global _OPENAI_CLIENT
    if _OPENAI_CLIENT is None:
        key = load_secret("OPENAI_API_KEY")
        if not key:
            raise RuntimeError("OPENAI_API_KEY is not set.")
        _OPENAI_CLIENT = OpenAI(api_key=key)
    return _OPENAI_CLIENT

def get_anthropic_client() -> Anthropic:
    global _ANTHROPIC_CLIENT
    if _ANTHROPIC_CLIENT is None:
        key = load_secret("ANTHROPIC_API_KEY")
        if not key:
            raise RuntimeError("ANTHROPIC_API_KEY is not set.")
        _ANTHROPIC_CLIENT = Anthropic(api_key=key)
    return _ANTHROPIC_CLIENT

def get_google_client():
    global _GOOGLE_CLIENT
    if _GOOGLE_CLIENT is None:
        key = load_secret("GEMINI_API_KEY")
        if not key:
            raise RuntimeError("GEMINI_API_KEY is not set.")
        _GOOGLE_CLIENT = genai.Client(api_key=key)
    return _GOOGLE_CLIENT

def get_nested_attr(obj: Any, path: str, default: Any = None) -> Any:
    current = obj
    for part in path.split("."):
        if current is None:
            return default
        if isinstance(current, dict):
            current = current.get(part, default)
        elif isinstance(current, (list, tuple)) and part.isdigit():
            index = int(part)
            current = current[index] if 0 <= index < len(current) else default
        else:
            current = getattr(current, part, default)
    return current

def calculate_estimated_cost(
    config: dict[str, Any],
    input_tokens: int,
    output_tokens: int,
    cached_input_tokens: int = 0,
) -> float:
    cached_input_tokens = max(0, min(int(cached_input_tokens), int(input_tokens)))
    uncached_input_tokens = max(0, int(input_tokens) - cached_input_tokens)
    return (
        uncached_input_tokens * float(config.get("input_usd_per_1m", 0.0))
        + cached_input_tokens * float(config.get("cached_input_usd_per_1m", 0.0))
        + int(output_tokens) * float(config.get("output_usd_per_1m", 0.0))
    ) / 1_000_000.0

def call_openai(
    config: dict[str, Any],
    system_prompt: str,
    user_prompt: str,
    passage_text: str,
) -> dict[str, Any]:
    client = get_openai_client()
    started = time.perf_counter()
    response = client.responses.parse(
        model=config["model"],
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        text_format=UnlearningOutput,
        reasoning={"effort": config.get("reasoning_effort", "low")},
        max_output_tokens=MAX_OUTPUT_TOKENS,
        store=False,
    )
    latency = time.perf_counter() - started

    parsed = getattr(response, "output_parsed", None)
    if parsed is None:
        refusal = get_nested_attr(response, "output.0.content.0.refusal", "")
        raise ProviderRefusalError(
            f"OpenAI returned no parsed output. refusal={refusal!r}"
        )
    semantic = validate_semantic_output(parsed, passage_text)

    usage = getattr(response, "usage", None)
    input_tokens = int(getattr(usage, "input_tokens", 0) or 0)
    output_tokens = int(getattr(usage, "output_tokens", 0) or 0)
    cached_tokens = int(
        get_nested_attr(usage, "input_tokens_details.cached_tokens", 0) or 0
    )

    return {
        **semantic,
        "provider": "openai",
        "run_name": config["run_name"],
        "model_requested": config["model"],
        "model_returned": clean_cell(getattr(response, "model", config["model"])),
        "request_id": clean_cell(getattr(response, "id", "")),
        "stop_reason": clean_cell(get_nested_attr(response, "output.0.status", "")),
        "input_tokens": input_tokens,
        "cached_input_tokens": cached_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens,
        "estimated_cost_usd": calculate_estimated_cost(
            config, input_tokens, output_tokens, cached_tokens
        ),
        "latency_seconds": latency,
        "raw_output_text": clean_cell(getattr(response, "output_text", "")),
        "raw_response": safe_model_dump(response),
        "wrapper": "openai.responses.parse",
    }

def call_anthropic(
    config: dict[str, Any],
    system_prompt: str,
    user_prompt: str,
    passage_text: str,
) -> dict[str, Any]:
    client = get_anthropic_client()
    output_config: dict[str, Any] = {
    "format": {"type": "json_schema", "schema": OUTPUT_JSON_SCHEMA}
}

# Pandas may convert an intentionally missing config value (None)
# into float NaN when MODEL_CONFIGS is round-tripped through a DataFrame.
# Never transmit None/NaN as an Anthropic API parameter.
    effort = config.get("effort")
    if effort is not None and not pd.isna(effort):
        output_config["effort"] = str(effort)

    request_kwargs: dict[str, Any] = {
        "model": config["model"],
        "max_tokens": MAX_OUTPUT_TOKENS,
        "system": system_prompt,
        "messages": [{"role": "user", "content": user_prompt}],
        "output_config": output_config,
    }
    # Keep the candidate-model comparison controlled. Claude Sonnet 5 and Opus 5
    # use adaptive thinking by default when the field is omitted, so configurations
    # registered as no-thinking must send the officially supported disabled value.
    # Haiku 4.5 does not support adaptive thinking and receives no thinking field.
    if config.get("adaptive_thinking_model"):
        thinking_mode = config.get("thinking")
        if thinking_mode == "disabled":
            request_kwargs["thinking"] = {"type": "disabled"}
        elif thinking_mode == "adaptive":
            request_kwargs["thinking"] = {"type": "adaptive"}
        else:
            raise ValueError(
                f"Adaptive-thinking model {config['model']!r} requires an explicit "
                "thinking mode of 'disabled' or 'adaptive'."
            )

    started = time.perf_counter()
    response = client.messages.create(**request_kwargs)
    latency = time.perf_counter() - started

    if clean_cell(getattr(response, "stop_reason", "")) == "refusal":
        raise ProviderRefusalError("Anthropic returned stop_reason='refusal'.")

    raw_text = "".join(
        clean_cell(getattr(block, "text", ""))
        for block in (getattr(response, "content", None) or [])
        if getattr(block, "type", None) == "text"
    )
    if not raw_text:
        raise ProviderRefusalError(
            f"Anthropic returned no text. stop_reason={getattr(response, 'stop_reason', None)!r}"
        )
    parsed = UnlearningOutput.model_validate_json(raw_text)
    semantic = validate_semantic_output(parsed, passage_text)

    usage = getattr(response, "usage", None)
    input_tokens = int(getattr(usage, "input_tokens", 0) or 0)
    output_tokens = int(getattr(usage, "output_tokens", 0) or 0)
    cached_tokens = int(getattr(usage, "cache_read_input_tokens", 0) or 0)

    return {
        **semantic,
        "provider": "anthropic",
        "run_name": config["run_name"],
        "model_requested": config["model"],
        "model_returned": clean_cell(getattr(response, "model", config["model"])),
        "request_id": clean_cell(getattr(response, "id", "")),
        "stop_reason": clean_cell(getattr(response, "stop_reason", "")),
        "input_tokens": input_tokens,
        "cached_input_tokens": cached_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens,
        "estimated_cost_usd": calculate_estimated_cost(
            config, input_tokens, output_tokens, cached_tokens
        ),
        "latency_seconds": latency,
        "raw_output_text": raw_text,
        "raw_response": safe_model_dump(response),
        "wrapper": "anthropic.messages.create.output_config.format",
    }

def call_google(
    config: dict[str, Any],
    system_prompt: str,
    user_prompt: str,
    passage_text: str,
) -> dict[str, Any]:
    client = get_google_client()
    generate_config = gemini_types.GenerateContentConfig(
        system_instruction=system_prompt,
        response_mime_type="application/json",
        response_json_schema=OUTPUT_JSON_SCHEMA,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        thinking_config=gemini_types.ThinkingConfig(
            thinking_level=config.get("thinking_level", "low")
        ),
    )
    # Temperature is intentionally omitted for all Google candidates.
    started = time.perf_counter()
    response = client.models.generate_content(
        model=config["model"],
        contents=user_prompt,
        config=generate_config,
    )
    latency = time.perf_counter() - started

    raw_text = clean_cell(getattr(response, "text", ""))
    if not raw_text:
        block_reason = get_nested_attr(response, "prompt_feedback.block_reason", "")
        raise ProviderRefusalError(
            f"Google returned empty text. block_reason={block_reason!r}"
        )
    parsed = UnlearningOutput.model_validate_json(raw_text)
    semantic = validate_semantic_output(parsed, passage_text)

    usage = getattr(response, "usage_metadata", None)
    input_tokens = int(getattr(usage, "prompt_token_count", 0) or 0)
    output_tokens = int(getattr(usage, "candidates_token_count", 0) or 0)
    cached_tokens = int(getattr(usage, "cached_content_token_count", 0) or 0)
    total_tokens = int(
        getattr(usage, "total_token_count", input_tokens + output_tokens)
        or input_tokens + output_tokens
    )

    return {
        **semantic,
        "provider": "google",
        "run_name": config["run_name"],
        "model_requested": config["model"],
        "model_returned": clean_cell(
            getattr(response, "model_version", "") or config["model"]
        ),
        "request_id": clean_cell(
            getattr(response, "response_id", "") or getattr(response, "id", "")
        ),
        "stop_reason": clean_cell(get_nested_attr(response, "candidates.0.finish_reason", "")),
        "input_tokens": input_tokens,
        "cached_input_tokens": cached_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "estimated_cost_usd": calculate_estimated_cost(
            config, input_tokens, output_tokens, cached_tokens
        ),
        "latency_seconds": latency,
        "raw_output_text": raw_text,
        "raw_response": safe_model_dump(response),
        "wrapper": "google.genai.models.generate_content.response_json_schema",
    }

PROVIDER_CALLERS: dict[str, Callable[..., dict[str, Any]]] = {
    "openai": call_openai,
    "anthropic": call_anthropic,
    "google": call_google,
}
print("Provider wrappers loaded:", list(PROVIDER_CALLERS))


Provider wrappers loaded: ['openai', 'anthropic', 'google']


In [ ]:
# Model-aware restart-safe cache, run keys, preflight, and materialization.

CACHE_FILES = {
    provider: SUBDIRS["raw_jsonl"] / f"{provider}_responses.jsonl"
    for provider in ENABLED_PROVIDERS
}

SNAPSHOT_COLUMNS = [
    "dataset", "passage_id", "text_sha256", "gold", "document", "source_file",
    "reference", "prompt_name", "prompt_hash", "stage", "provider", "run_name",
    "model_requested", "model_returned", "replicate", "run_key", "status",
    "unlearning_probability", "unlearning_label", "predicted_label_at_0_5",
    "label_probability_inconsistent", "target_category", "target_label_inconsistent",
    "evidence_quote", "evidence_quote_valid", "rationale",
    "input_tokens", "cached_input_tokens", "output_tokens", "total_tokens",
    "estimated_cost_usd", "latency_seconds", "request_id", "stop_reason",
    "wrapper", "created_at_utc", "cache_source_dataset", "cache_source_passage_id",
    "model_config_sha256", "source_workbook_sha256", "codebook_whole_sheet_sha256",
]

def model_config_signature(config: dict[str, Any]) -> dict[str, Any]:
    keys = [
        "run_name", "provider", "model", "reasoning_effort", "thinking",
        "effort", "thinking_level", "adaptive_thinking_model",
    ]
    return {
        **{key: config.get(key) for key in keys if key in config},
        "max_output_tokens": MAX_OUTPUT_TOKENS,
        "output_schema_version": OUTPUT_SCHEMA_VERSION,
        "output_schema_sha256": OUTPUT_SCHEMA_SHA256,
    }

def model_config_hash(config: dict[str, Any]) -> str:
    return sha256_text(
        json.dumps(model_config_signature(config), sort_keys=True, ensure_ascii=False)
    )

def make_run_key(
    config: dict[str, Any],
    prompt_spec: PromptSpec,
    text_sha256: str,
    replicate: int,
) -> str:
    payload = {
        "run_name": config["run_name"],
        "model": config["model"],
        "prompt_hash": prompt_spec.prompt_hash,
        "text_sha256": text_sha256,
        "replicate": int(replicate),
        "model_config_signature": model_config_signature(config),
    }
    return sha256_text(json.dumps(payload, sort_keys=True, ensure_ascii=False))

def read_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    records: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Malformed JSONL in {path} at line {line_number}: {exc}"
                ) from exc
            if isinstance(record, dict):
                records.append(record)
    return records

CACHE_SUCCESS: dict[str, dict[str, dict[str, Any]]] = {}
CACHE_ALL_RECORDS: dict[str, list[dict[str, Any]]] = {}

def reload_caches() -> None:
    CACHE_SUCCESS.clear()
    CACHE_ALL_RECORDS.clear()
    for provider, path in CACHE_FILES.items():
        records = read_jsonl(path)
        CACHE_ALL_RECORDS[provider] = records
        success_map: dict[str, dict[str, Any]] = {}
        for record in records:
            if record.get("status") == "success" and record.get("run_key"):
                success_map[str(record["run_key"])] = record
        CACHE_SUCCESS[provider] = success_map
        print(
            f"{provider}: {len(success_map)} successful cached run keys; "
            f"{len(records)} total records."
        )

reload_caches()

def classify_exception(exc: Exception) -> tuple[str, Optional[int]]:
    status_code = getattr(exc, "status_code", None)
    if status_code is None:
        status_code = getattr(getattr(exc, "response", None), "status_code", None)
    try:
        status_code = int(status_code) if status_code is not None else None
    except Exception:
        status_code = None

    if isinstance(exc, ProviderRefusalError):
        return "refusal", status_code
    if isinstance(exc, SemanticValidationError):
        return "schema_or_semantic", status_code
    if type(exc).__name__ in {"ValidationError", "JSONDecodeError"}:
        return "schema_or_semantic", status_code
    if status_code in {400, 401, 403, 404, 422}:
        return "permanent", status_code
    if status_code in {408, 409, 425, 429}:
        return "transient", status_code
    if status_code is not None and status_code >= 500:
        return "transient", status_code

    message = str(exc).lower()
    if any(marker in message for marker in [
        "rate limit", "rate_limit", "temporarily unavailable", "timeout",
        "timed out", "connection reset", "connection error", "overloaded",
        "service unavailable", "internal server error", "resource exhausted",
        "deadline exceeded",
    ]):
        return "transient", status_code
    if any(marker in message for marker in [
        "invalid api key", "authentication", "permission denied",
        "invalid_request_error", "unsupported parameter", "model not found",
        "does not exist", "billing",
    ]):
        return "permanent", status_code
    if any(marker in message for marker in [
        "validation error", "invalid json", "json decode", "missing required",
    ]):
        return "schema_or_semantic", status_code
    return "unknown", status_code

def retry_delay_seconds(exc: Exception, attempt: int) -> float:
    for source in [getattr(exc, "response", None), exc]:
        headers = getattr(source, "headers", None)
        if headers:
            for key in ["retry-after", "Retry-After"]:
                try:
                    value = headers.get(key)
                    if value is not None:
                        return min(MAX_RETRY_SECONDS, max(0.0, float(value)))
                except Exception:
                    pass
    exponential = min(MAX_RETRY_SECONDS, BASE_RETRY_SECONDS * (2 ** (attempt - 1)))
    deterministic_jitter = random.Random(
        ANALYSIS_RANDOM_SEED + attempt
    ).uniform(0.0, 1.0)
    return exponential + deterministic_jitter

def estimate_tokens_from_text(text: str) -> int:
    return max(1, math.ceil(len(text) / 3.6))

def build_job_rows(
    dataset_name: str,
    prompt_specs: Sequence[PromptSpec],
    model_configs: Sequence[dict[str, Any]],
) -> pd.DataFrame:
    frame = PASSAGE_TABLES[dataset_name]
    rows: list[dict[str, Any]] = []
    for passage in frame.itertuples(index=False):
        for spec in prompt_specs:
            user_prompt = render_user_prompt(spec, passage.text)
            for config in model_configs:
                for replicate in range(REPLICATES_PER_PROMPT):
                    run_key = make_run_key(config, spec, passage.text_sha256, replicate)
                    estimated_input = estimate_tokens_from_text(
                        spec.system_prompt + "\n\n" + user_prompt
                    )
                    rows.append({
                        "dataset": dataset_name,
                        "passage_id": passage.passage_id,
                        "text_sha256": passage.text_sha256,
                        "gold": int(passage.gold),
                        "prompt_name": spec.prompt_name,
                        "prompt_hash": spec.prompt_hash,
                        "provider": config["provider"],
                        "run_name": config["run_name"],
                        "model": config["model"],
                        "replicate": replicate,
                        "run_key": run_key,
                        "cached_success": run_key in CACHE_SUCCESS.get(config["provider"], {}),
                        "estimated_input_tokens": estimated_input,
                        "estimated_output_tokens": 180,
                        "estimated_cost_usd": calculate_estimated_cost(
                            config, estimated_input, 180, 0
                        ),
                    })
    return pd.DataFrame(rows)

def materialize_cached_record(
    dataset_name: str,
    passage: Any,
    spec: PromptSpec,
    config: dict[str, Any],
    replicate: int,
) -> Optional[dict[str, Any]]:
    run_key = make_run_key(config, spec, passage.text_sha256, replicate)
    cached = CACHE_SUCCESS.get(config["provider"], {}).get(run_key)
    if cached is None:
        return None
    probability = float(cached["unlearning_probability"])
    return {
        "dataset": dataset_name,
        "passage_id": passage.passage_id,
        "text_sha256": passage.text_sha256,
        "gold": int(passage.gold),
        "document": passage.document,
        "source_file": passage.source_file,
        "reference": getattr(passage, "reference", ""),
        "prompt_name": spec.prompt_name,
        "prompt_hash": spec.prompt_hash,
        "stage": spec.stage,
        "provider": config["provider"],
        "run_name": config["run_name"],
        "model_requested": cached.get("model_requested", config["model"]),
        "model_returned": cached.get("model_returned", ""),
        "replicate": int(replicate),
        "run_key": run_key,
        "status": "success",
        "unlearning_probability": probability,
        "unlearning_label": cached.get("unlearning_label", ""),
        "predicted_label_at_0_5": int(probability >= 0.50),
        "label_probability_inconsistent": bool(
            cached.get("label_probability_inconsistent", False)
        ),
        "target_category": cached.get("target_category", ""),
        "target_label_inconsistent": bool(
            cached.get("target_label_inconsistent", False)
        ),
        "evidence_quote": cached.get("evidence_quote", ""),
        "evidence_quote_valid": bool(cached.get("evidence_quote_valid", False)),
        "rationale": cached.get("rationale", ""),
        "input_tokens": int(cached.get("input_tokens", 0) or 0),
        "cached_input_tokens": int(cached.get("cached_input_tokens", 0) or 0),
        "output_tokens": int(cached.get("output_tokens", 0) or 0),
        "total_tokens": int(cached.get("total_tokens", 0) or 0),
        "estimated_cost_usd": float(cached.get("estimated_cost_usd", 0.0) or 0.0),
        "latency_seconds": float(cached.get("latency_seconds", 0.0) or 0.0),
        "request_id": cached.get("request_id", ""),
        "stop_reason": cached.get("stop_reason", ""),
        "wrapper": cached.get("wrapper", ""),
        "created_at_utc": cached.get("created_at_utc", ""),
        "cache_source_dataset": cached.get("source_dataset", ""),
        "cache_source_passage_id": cached.get("source_passage_id", ""),
        "model_config_sha256": cached.get(
            "model_config_sha256", model_config_hash(config)
        ),
        "source_workbook_sha256": cached.get(
            "source_workbook_sha256", SOURCE_WORKBOOK_SHA256_AT_LOAD
        ),
        "codebook_whole_sheet_sha256": cached.get(
            "codebook_whole_sheet_sha256", CODEBOOK_WHOLE_SHEET_SHA256
        ),
    }

def configs_from_run_names(run_names: Sequence[str]) -> list[dict[str, Any]]:
    missing = [name for name in run_names if name not in MODEL_BY_RUN_NAME]
    if missing:
        raise KeyError(f"Unknown run_name values: {missing}")
    return [MODEL_BY_RUN_NAME[name] for name in run_names]

def materialize_prediction_table(
    dataset_name: str,
    prompt_specs: Sequence[PromptSpec],
    run_names: Optional[Sequence[str]] = None,
    require_complete: bool = True,
) -> pd.DataFrame:
    configs = configs_from_run_names(run_names or ENABLED_RUN_NAMES)
    frame = PASSAGE_TABLES[dataset_name]
    rows: list[dict[str, Any]] = []
    missing: list[dict[str, Any]] = []

    for passage in frame.itertuples(index=False):
        for spec in prompt_specs:
            for config in configs:
                for replicate in range(REPLICATES_PER_PROMPT):
                    record = materialize_cached_record(
                        dataset_name, passage, spec, config, replicate
                    )
                    if record is None:
                        missing.append({
                            "dataset": dataset_name,
                            "passage_id": passage.passage_id,
                            "text_sha256": passage.text_sha256,
                            "prompt_name": spec.prompt_name,
                            "provider": config["provider"],
                            "run_name": config["run_name"],
                            "model": config["model"],
                            "replicate": replicate,
                            "run_key": make_run_key(
                                config, spec, passage.text_sha256, replicate
                            ),
                        })
                    else:
                        rows.append(record)

    result = pd.DataFrame(rows, columns=SNAPSHOT_COLUMNS)
    if missing:
        missing_df = pd.DataFrame(missing)
        safe_dataset = dataset_name.lower().replace(" ", "_")
        stage = prompt_specs[0].stage if prompt_specs else "unknown"
        missing_path = SUBDIRS["logs"] / f"missing__{safe_dataset}__{stage}.csv"
        atomic_to_csv(missing_df, missing_path)
        if require_complete:
            raise RuntimeError(
                f"{len(missing)} expected predictions are missing for {dataset_name}. "
                f"See {missing_path}. Run the relevant provider/model cells first."
            )
    return result

def save_prediction_snapshot(
    dataset_name: str,
    prompt_specs: Sequence[PromptSpec],
    model_configs: Sequence[dict[str, Any]],
) -> pd.DataFrame:
    run_names = [config["run_name"] for config in model_configs]
    snapshot = materialize_prediction_table(
        dataset_name,
        prompt_specs,
        run_names=run_names,
        require_complete=False,
    )
    safe_dataset = dataset_name.lower().replace(" ", "_")
    provider = model_configs[0]["provider"] if model_configs else "none"
    prompt_group_hash = sha256_text(
        "|".join(sorted(spec.prompt_hash for spec in prompt_specs))
    )[:12]
    stem = f"{safe_dataset}__{prompt_specs[0].stage}__{provider}__{prompt_group_hash}"
    atomic_to_csv(snapshot, SUBDIRS["snapshots"] / f"{stem}.csv")
    if not snapshot.empty:
        atomic_to_parquet(snapshot, SUBDIRS["snapshots"] / f"{stem}.parquet")
    return snapshot

print("Cache/materialization engine loaded.")


openai: 7 successful cached run keys; 7 total records.
anthropic: 0 successful cached run keys; 0 total records.
google: 0 successful cached run keys; 0 total records.
Cache/materialization engine loaded.


In [ ]:
# Restart-safe API execution engine. A successful response is fsync'd to JSONL
# immediately. Permanent errors stop only the affected model configuration.

def run_prediction_job(
    dataset_name: str,
    prompt_specs: Sequence[PromptSpec],
    model_configs: Sequence[dict[str, Any]],
    *,
    execute: bool,
) -> pd.DataFrame:
    if not prompt_specs:
        raise ValueError("prompt_specs is empty")
    if not model_configs:
        raise ValueError("model_configs is empty")
    providers = {config["provider"] for config in model_configs}
    if len(providers) != 1:
        raise ValueError("Each job cell must contain exactly one provider.")
    provider = next(iter(providers))

    reload_caches()
    plan = build_job_rows(dataset_name, prompt_specs, model_configs)
    new_plan = plan.loc[~plan["cached_success"]].copy()
    new_calls = len(new_plan)
    estimated_new_cost = float(new_plan["estimated_cost_usd"].sum())

    summary = (
        plan.groupby(["provider", "run_name", "model", "prompt_name"], as_index=False)
        .agg(
            logical_rows=("run_key", "size"),
            cached_rows=("cached_success", "sum"),
            estimated_total_cost_usd=("estimated_cost_usd", "sum"),
        )
    )
    summary["new_calls"] = summary["logical_rows"] - summary["cached_rows"]
    display(summary)
    print({
        "dataset": dataset_name,
        "provider": provider,
        "stage": prompt_specs[0].stage,
        "new_calls": new_calls,
        "rough_new_cost_usd": round(estimated_new_cost, 4),
        "execute": execute,
    })

    if not execute:
        return plan
    if not ALLOW_PAID_API_CALLS:
        raise RuntimeError(
            "Paid calls are blocked. Set ALLOW_PAID_API_CALLS=True only after "
            "reviewing this exact preflight."
        )
    if MAX_NEW_CALLS_PER_JOB is not None and new_calls > MAX_NEW_CALLS_PER_JOB:
        raise RuntimeError(
            f"Job requires {new_calls} new calls, exceeding "
            f"MAX_NEW_CALLS_PER_JOB={MAX_NEW_CALLS_PER_JOB}."
        )
    if (
        MAX_ESTIMATED_COST_USD_PER_JOB is not None
        and estimated_new_cost > MAX_ESTIMATED_COST_USD_PER_JOB
    ):
        raise RuntimeError(
            f"Estimated job cost ${estimated_new_cost:.4f} exceeds "
            f"MAX_ESTIMATED_COST_USD_PER_JOB=${MAX_ESTIMATED_COST_USD_PER_JOB:.4f}."
        )
    if new_calls == 0:
        print("All expected calls are already cached.")
        return save_prediction_snapshot(dataset_name, prompt_specs, model_configs)

    frame = PASSAGE_TABLES[dataset_name]
    caller = PROVIDER_CALLERS[provider]
    job_failures: list[dict[str, Any]] = []

    for config in model_configs:
        model_new_calls = int(
            new_plan["run_name"].eq(config["run_name"]).sum()
        )
        if model_new_calls == 0:
            print(f"{config['run_name']}: fully cached; skipping.")
            continue

        progress = tqdm(total=model_new_calls, desc=f"{provider} | {config['run_name']}")
        completed_since_snapshot = 0
        abort_model = False
        try:
            for passage in frame.itertuples(index=False):
                if abort_model:
                    break
                for spec in prompt_specs:
                    if abort_model:
                        break
                    user_prompt = render_user_prompt(spec, passage.text)

                    # Prompt leakage checks: no IDs, gold labels, or rationale are inserted.
                    if passage.passage_id and passage.passage_id in user_prompt:
                        raise AssertionError("Passage_ID leaked into the user prompt.")
                    if user_prompt.count(repair_unicode(passage.text)) != 1:
                        raise AssertionError(
                            "Target paragraph is missing or duplicated in the user prompt."
                        )

                    for replicate in range(REPLICATES_PER_PROMPT):
                        run_key = make_run_key(
                            config, spec, passage.text_sha256, replicate
                        )
                        if run_key in CACHE_SUCCESS.get(provider, {}):
                            continue

                        last_exception: Optional[Exception] = None
                        schema_failures = 0
                        for attempt in range(1, MAX_ATTEMPTS_PER_CALL + 1):
                            try:
                                output = caller(
                                    config,
                                    spec.system_prompt,
                                    user_prompt,
                                    passage.text,
                                )
                                success_record = {
                                    "status": "success",
                                    "created_at_utc": utc_now(),
                                    "experiment_name": EXPERIMENT_NAME,
                                    "prompt_version": PROMPT_VERSION,
                                    "output_schema_version": OUTPUT_SCHEMA_VERSION,
                                    "output_schema_sha256": OUTPUT_SCHEMA_SHA256,
                                    "source_workbook_sha256": SOURCE_WORKBOOK_SHA256_AT_LOAD,
                                    "codebook_whole_sheet_sha256": CODEBOOK_WHOLE_SHEET_SHA256,
                                    "source_dataset": dataset_name,
                                    "source_passage_id": passage.passage_id,
                                    "text_sha256": passage.text_sha256,
                                    "prompt_name": spec.prompt_name,
                                    "prompt_hash": spec.prompt_hash,
                                    "system_prompt_sha256": sha256_text(spec.system_prompt),
                                    "user_prompt_sha256": sha256_text(user_prompt),
                                    "stage": spec.stage,
                                    "provider": provider,
                                    "run_name": config["run_name"],
                                    "model_requested": config["model"],
                                    "model_config_sha256": model_config_hash(config),
                                    "replicate": replicate,
                                    "run_key": run_key,
                                    "attempt": attempt,
                                    **output,
                                }
                                append_jsonl(CACHE_FILES[provider], success_record)
                                CACHE_ALL_RECORDS.setdefault(provider, []).append(success_record)
                                CACHE_SUCCESS.setdefault(provider, {})[run_key] = success_record
                                progress.update(1)
                                completed_since_snapshot += 1
                                if completed_since_snapshot >= SNAPSHOT_EVERY_N_NEW_CALLS:
                                    save_prediction_snapshot(
                                        dataset_name, prompt_specs, model_configs
                                    )
                                    completed_since_snapshot = 0
                                time.sleep(REQUEST_SLEEP_SECONDS)
                                break

                            except Exception as exc:
                                last_exception = exc
                                error_kind, status_code = classify_exception(exc)
                                if error_kind == "schema_or_semantic":
                                    schema_failures += 1
                                failure_record = {
                                    "status": "attempt_failure",
                                    "created_at_utc": utc_now(),
                                    "experiment_name": EXPERIMENT_NAME,
                                    "source_dataset": dataset_name,
                                    "source_passage_id": passage.passage_id,
                                    "text_sha256": passage.text_sha256,
                                    "prompt_name": spec.prompt_name,
                                    "prompt_hash": spec.prompt_hash,
                                    "stage": spec.stage,
                                    "provider": provider,
                                    "run_name": config["run_name"],
                                    "model_requested": config["model"],
                                    "model_config_sha256": model_config_hash(config),
                                    "replicate": replicate,
                                    "run_key": run_key,
                                    "attempt": attempt,
                                    "error_kind": error_kind,
                                    "http_status": status_code,
                                    "exception_type": type(exc).__name__,
                                    "error_message": str(exc),
                                    "traceback_tail": traceback.format_exc()[-4000:],
                                }
                                append_jsonl(CACHE_FILES[provider], failure_record)
                                CACHE_ALL_RECORDS.setdefault(provider, []).append(failure_record)

                                permanent = error_kind in {"permanent", "refusal"}
                                retryable_schema = (
                                    error_kind == "schema_or_semantic"
                                    and schema_failures <= MAX_SCHEMA_RETRIES
                                )
                                retryable = error_kind in {"transient", "unknown"} or retryable_schema

                                if permanent and STOP_MODEL_ON_PERMANENT_ERROR:
                                    abort_model = True
                                    job_failures.append(failure_record)
                                    print(
                                        f"\nStopping only {config['run_name']} after permanent "
                                        f"error; other completed model calls remain cached:\n{exc}"
                                    )
                                    break
                                if not retryable or attempt >= MAX_ATTEMPTS_PER_CALL:
                                    break
                                delay = retry_delay_seconds(exc, attempt)
                                print(
                                    f"\n{config['run_name']} attempt {attempt} failed "
                                    f"({error_kind}, status={status_code}); retrying in "
                                    f"{delay:.1f}s: {exc}"
                                )
                                time.sleep(delay)

                        if run_key not in CACHE_SUCCESS.get(provider, {}) and not abort_model:
                            final_record = {
                                "status": "final_failure",
                                "created_at_utc": utc_now(),
                                "experiment_name": EXPERIMENT_NAME,
                                "source_dataset": dataset_name,
                                "source_passage_id": passage.passage_id,
                                "text_sha256": passage.text_sha256,
                                "prompt_name": spec.prompt_name,
                                "prompt_hash": spec.prompt_hash,
                                "stage": spec.stage,
                                "provider": provider,
                                "run_name": config["run_name"],
                                "model_requested": config["model"],
                                "model_config_sha256": model_config_hash(config),
                                "replicate": replicate,
                                "run_key": run_key,
                                "exception_type": (
                                    type(last_exception).__name__
                                    if last_exception is not None else "UnknownError"
                                ),
                                "error_message": (
                                    str(last_exception)
                                    if last_exception is not None
                                    else "Call failed without an exception object."
                                ),
                            }
                            append_jsonl(CACHE_FILES[provider], final_record)
                            CACHE_ALL_RECORDS.setdefault(provider, []).append(final_record)
                            job_failures.append(final_record)
                            print(
                                f"\n{config['run_name']} exhausted retries for "
                                f"passage={passage.passage_id}. Continuing other model configs."
                            )
                            abort_model = True
                            break
        finally:
            progress.close()
            save_prediction_snapshot(dataset_name, prompt_specs, model_configs)

    if job_failures:
        failure_path = (
            SUBDIRS["logs"]
            / f"job_failures__{dataset_name.lower().replace(' ', '_')}__{provider}__{prompt_specs[0].stage}.csv"
        )
        atomic_to_csv(pd.DataFrame(job_failures), failure_path)
        print(f"Job completed with {len(job_failures)} model-level failures: {failure_path}")
    else:
        print("Job completed without final/permanent model failures.")

    return save_prediction_snapshot(dataset_name, prompt_specs, model_configs)

print("Prediction runner loaded.")


Prediction runner loaded.


## Stage 1 — Codebook without versus with the native `Examples` column

This stage runs on **GPT Test only**. All enabled candidate models receive the same target paragraph, output contract, maximum output tokens, and latest prompt scaffold. The only manipulated factor is whether the codebook renderer includes its native `Examples` column. The winner is the prompt with the highest arithmetic mean of model-level precision at threshold 0.50.

In [ ]:
# Stage 1 preflight — no API calls.
STAGE1_PREFLIGHT = pd.concat([
    build_job_rows("GPT Test", STAGE1_PROMPTS, PROVIDER_MODEL_CONFIGS[provider])
    for provider in ENABLED_PROVIDERS
], ignore_index=True)
STAGE1_PREFLIGHT_SUMMARY = (
    STAGE1_PREFLIGHT.groupby(
        ["provider", "run_name", "model", "prompt_name"], as_index=False
    )
    .agg(
        logical_rows=("run_key", "size"),
        cached_rows=("cached_success", "sum"),
        estimated_input_tokens=("estimated_input_tokens", "sum"),
        estimated_output_tokens=("estimated_output_tokens", "sum"),
        rough_estimated_cost_usd=("estimated_cost_usd", "sum"),
    )
)
STAGE1_PREFLIGHT_SUMMARY["new_calls"] = (
    STAGE1_PREFLIGHT_SUMMARY["logical_rows"]
    - STAGE1_PREFLIGHT_SUMMARY["cached_rows"]
)
atomic_to_csv(
    STAGE1_PREFLIGHT_SUMMARY,
    SUBDIRS["audits"] / "stage1_gpt_test_preflight.csv",
)
display(STAGE1_PREFLIGHT_SUMMARY)
print("Total new Stage 1 calls:", int(STAGE1_PREFLIGHT_SUMMARY["new_calls"].sum()))
print("Rough estimated Stage 1 cost: $", round(STAGE1_PREFLIGHT_SUMMARY["rough_estimated_cost_usd"].sum(), 4))


,provider,run_name,model,prompt_name,logical_rows,cached_rows,estimated_input_tokens,estimated_output_tokens,rough_estimated_cost_usd,new_calls
0,anthropic,anthropic_claude_haiku_4_5_no_thinking,claude-haiku-4-5-20251001,codebook_no_examples_no_checklist,84,0,239377,15120,0.314977,84
1,anthropic,anthropic_claude_haiku_4_5_no_thinking,claude-haiku-4-5-20251001,codebook_with_examples_no_checklist,84,0,288049,15120,0.363649,84
2,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,codebook_no_examples_no_checklist,84,0,239377,15120,0.082524,84
3,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,codebook_with_examples_no_checklist,84,0,288049,15120,0.094692,84
4,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,codebook_no_examples_no_checklist,84,4,239377,15120,0.660194,80
5,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,codebook_with_examples_no_checklist,84,3,288049,15120,0.757538,81


Total new Stage 1 calls: 497
Rough estimated Stage 1 cost: $ 2.2736


In [ ]:
# Stage 1 — GPT Test — OpenAI candidate models.
STAGE1_OPENAI_JOB = run_prediction_job(
    "GPT Test",
    STAGE1_PROMPTS,
    PROVIDER_MODEL_CONFIGS["openai"],
    execute=RUN_STAGE1_GPT_TEST,
)


openai: 7 successful cached run keys; 7 total records.
anthropic: 0 successful cached run keys; 0 total records.
google: 0 successful cached run keys; 0 total records.


,provider,run_name,model,prompt_name,logical_rows,cached_rows,estimated_total_cost_usd,new_calls
0,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,codebook_no_examples_no_checklist,84,4,0.660194,80
1,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,codebook_with_examples_no_checklist,84,3,0.757538,81


{'dataset': 'GPT Test', 'provider': 'openai', 'stage': 'stage1_examples_ab', 'new_calls': 161, 'rough_new_cost_usd': 1.3602, 'execute': True}


openai | openai_gpt_5_6_terra_low:   0%|          | 0/161 [00:00<?, ?it/s]

Job completed without final/permanent model failures.


In [ ]:
# Stage 1 — GPT Test — Anthropic candidate models.
STAGE1_ANTHROPIC_JOB = run_prediction_job(
    "GPT Test",
    STAGE1_PROMPTS,
    PROVIDER_MODEL_CONFIGS["anthropic"],
    execute=RUN_STAGE1_GPT_TEST,
)


openai: 168 successful cached run keys; 168 total records.
anthropic: 0 successful cached run keys; 2 total records.
google: 0 successful cached run keys; 0 total records.


,provider,run_name,model,prompt_name,logical_rows,cached_rows,estimated_total_cost_usd,new_calls
0,anthropic,anthropic_claude_haiku_4_5_no_thinking,claude-haiku-4-5-20251001,codebook_no_examples_no_checklist,84,0,0.314977,84
1,anthropic,anthropic_claude_haiku_4_5_no_thinking,claude-haiku-4-5-20251001,codebook_with_examples_no_checklist,84,0,0.363649,84


{'dataset': 'GPT Test', 'provider': 'anthropic', 'stage': 'stage1_examples_ab', 'new_calls': 168, 'rough_new_cost_usd': 0.6786, 'execute': True}


anthropic | anthropic_claude_haiku_4_5_no_thinking:   0%|          | 0/168 [00:00<?, ?it/s]

Job completed without final/permanent model failures.


In [ ]:
# Stage 1 — GPT Test — Google candidate models.
STAGE1_GOOGLE_JOB = run_prediction_job(
    "GPT Test",
    STAGE1_PROMPTS,
    PROVIDER_MODEL_CONFIGS["google"],
    execute=RUN_STAGE1_GPT_TEST,
)


openai: 168 successful cached run keys; 168 total records.
anthropic: 168 successful cached run keys; 170 total records.
google: 0 successful cached run keys; 0 total records.


,provider,run_name,model,prompt_name,logical_rows,cached_rows,estimated_total_cost_usd,new_calls
0,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,codebook_no_examples_no_checklist,84,0,0.082524,84
1,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,codebook_with_examples_no_checklist,84,0,0.094692,84


{'dataset': 'GPT Test', 'provider': 'google', 'stage': 'stage1_examples_ab', 'new_calls': 168, 'rough_new_cost_usd': 0.1772, 'execute': True}


google | google_gemini_3_1_flash_lite_low:   0%|          | 0/168 [00:00<?, ?it/s]


google_gemini_3_1_flash_lite_low attempt 1 failed (transient, status=429); retrying in 2.5s: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.1-flash-lite\nPlease retry in 31.178815214s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPe

In [ ]:
# Core evaluation utilities.

METRIC_NAMES = [
    "accuracy", "precision", "recall", "f1", "auroc", "auprc", "brier"
]

def compute_binary_metrics(
    y_true: Sequence[int],
    probabilities: Sequence[float],
    threshold: float = 0.50,
) -> dict[str, Any]:
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(probabilities, dtype=float)
    if len(y) != len(p) or len(y) == 0:
        raise ValueError("Metric inputs must have the same nonzero length.")
    if not np.all(np.isfinite(p)):
        raise ValueError("Probabilities contain non-finite values.")

    pred = (p >= float(threshold)).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "n": int(len(y)),
        "positive_prevalence": float(y.mean()),
        "threshold": float(threshold),
        "predicted_positive_count": int(pred.sum()),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "accuracy": float(accuracy_score(y, pred)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "auroc": float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else np.nan,
        "auprc": float(average_precision_score(y, p)) if int(y.sum()) > 0 else np.nan,
        "brier": float(brier_score_loss(y, p)),
    }

def metric_from_predictions(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    metric: str,
    threshold: float = 0.50,
) -> float:
    return float(compute_binary_metrics(y_true, probabilities, threshold)[metric])

def add_analysis_metadata(predictions: pd.DataFrame) -> pd.DataFrame:
    pieces: list[pd.DataFrame] = []
    for dataset_name, group in predictions.groupby("dataset", sort=False):
        metadata_columns = [
            "passage_id", "human_label_status", "review_flag", "review_issue",
            "example_overlap_risk", "example_max_token_jaccard",
            "adjudication_rationale", "original_labeler", "anmol_unlearning",
            "prerana_unlearning", "kyle_unlearning",
        ]
        metadata = PASSAGE_TABLES[dataset_name][metadata_columns].copy()
        pieces.append(
            group.merge(metadata, on="passage_id", how="left", validate="many_to_one")
        )
    return pd.concat(pieces, ignore_index=True) if pieces else predictions.copy()

def scope_mask(frame: pd.DataFrame, scope: str) -> pd.Series:
    if scope == "all_rows":
        return pd.Series(True, index=frame.index)
    if scope == "review_unflagged":
        return ~frame["review_flag"].fillna(False).astype(bool)
    if scope == "example_leakage_safe":
        return ~frame["example_overlap_risk"].fillna(False).astype(bool)
    if scope == "human_labeled_only":
        if frame["dataset"].nunique() == 1 and frame["dataset"].iloc[0] == "GPT Test":
            return pd.Series(True, index=frame.index)
        status = frame["human_label_status"].fillna("").str.strip().str.lower()
        return status.ne("unlabeled") & status.ne("")
    raise ValueError(f"Unknown analysis scope: {scope}")

ANALYSIS_SCOPES = [
    "all_rows", "human_labeled_only", "review_unflagged", "example_leakage_safe"
]

def model_level_metrics(
    predictions: pd.DataFrame,
    threshold: float = 0.50,
) -> pd.DataFrame:
    enriched = add_analysis_metadata(predictions)
    grouping = [
        "dataset", "prompt_name", "prompt_hash", "provider", "run_name",
        "model_requested", "model_returned",
    ]
    rows: list[dict[str, Any]] = []
    for keys, group in enriched.groupby(grouping, sort=False, dropna=False):
        base = dict(zip(grouping, keys))
        for scope in ANALYSIS_SCOPES:
            scoped = group.loc[scope_mask(group, scope)]
            if scoped.empty:
                continue
            rows.append({
                **base,
                "scope": scope,
                **compute_binary_metrics(
                    scoped["gold"], scoped["unlearning_probability"], threshold
                ),
            })
    return pd.DataFrame(rows)

def document_level_metrics(
    predictions: pd.DataFrame,
    threshold: float = 0.50,
) -> pd.DataFrame:
    grouping = [
        "dataset", "prompt_name", "provider", "run_name", "model_requested", "document"
    ]
    rows: list[dict[str, Any]] = []
    for keys, group in predictions.groupby(grouping, sort=False, dropna=False):
        rows.append({
            **dict(zip(grouping, keys)),
            **compute_binary_metrics(
                group["gold"], group["unlearning_probability"], threshold
            ),
        })
    return pd.DataFrame(rows)

def labeler_level_metrics(
    predictions: pd.DataFrame,
    threshold: float = 0.50,
) -> pd.DataFrame:
    if predictions["dataset"].nunique() != 1 or predictions["dataset"].iloc[0] != "GPT Test":
        return pd.DataFrame()
    rows: list[dict[str, Any]] = []
    metadata = GPT_TEST_DF[[
        "passage_id", "anmol_unlearning_raw", "prerana_unlearning_raw"
    ]].copy()
    for labeler, source_column in [
        ("anmol", "anmol_unlearning_raw"),
        ("prerana", "prerana_unlearning_raw"),
    ]:
        label_map = metadata[["passage_id", source_column]].copy()
        label_map["labeler_gold"] = label_map[source_column].map(normalize_yes_no)
        label_map = label_map.dropna(subset=["labeler_gold"])
        merged = predictions.merge(
            label_map[["passage_id", "labeler_gold"]],
            on="passage_id", how="inner", validate="many_to_one"
        )
        grouping = ["prompt_name", "provider", "run_name", "model_requested"]
        for keys, group in merged.groupby(grouping, sort=False):
            rows.append({
                "labeler": labeler,
                **dict(zip(grouping, keys)),
                **compute_binary_metrics(
                    group["labeler_gold"].astype(int),
                    group["unlearning_probability"],
                    threshold,
                ),
            })
    return pd.DataFrame(rows)

def choose_prompt_configuration(
    model_metrics: pd.DataFrame,
    dataset: str = PROMPT_SELECTION_DATASET,
    scope: str = "all_rows",
) -> tuple[str, pd.DataFrame]:
    candidate = model_metrics[
        model_metrics["dataset"].eq(dataset)
        & model_metrics["scope"].eq(scope)
    ].copy()
    if candidate.empty:
        raise RuntimeError("No model-level metrics available for prompt selection.")

    expected = set(ENABLED_RUN_NAMES)
    observed = candidate.groupby("prompt_name")["run_name"].apply(set)
    incomplete = {name: sorted(expected - names) for name, names in observed.items() if names != expected}
    if incomplete:
        raise RuntimeError(
            "Prompt selection requires every enabled candidate model for every prompt. "
            f"Missing: {incomplete}"
        )

    summary = (
        candidate.groupby(["prompt_name", "prompt_hash"], as_index=False)
        .agg(
            mean_precision=("precision", "mean"),
            mean_recall=("recall", "mean"),
            mean_f1=("f1", "mean"),
            mean_auroc=("auroc", "mean"),
            mean_auprc=("auprc", "mean"),
            mean_accuracy=("accuracy", "mean"),
            mean_brier=("brier", "mean"),
            model_configurations=("run_name", "nunique"),
            providers=("provider", "nunique"),
        )
    )
    summary = summary.sort_values(
        [
            "mean_precision", "mean_recall", "mean_f1", "mean_auprc",
            "mean_auroc", "mean_accuracy", "mean_brier", "prompt_name",
        ],
        ascending=[False, False, False, False, False, False, True, True],
        kind="mergesort",
    ).reset_index(drop=True)
    return str(summary.iloc[0]["prompt_name"]), summary

def deterministic_seed(*parts: Any) -> int:
    digest = sha256_text(json.dumps(parts, sort_keys=True, default=json_default))
    return int(digest[:8], 16)

def stratified_bootstrap_indices(
    y_true: np.ndarray,
    rng: np.random.Generator,
) -> np.ndarray:
    positive = np.flatnonzero(y_true == 1)
    negative = np.flatnonzero(y_true == 0)
    if len(positive) == 0 or len(negative) == 0:
        return rng.integers(0, len(y_true), size=len(y_true))
    idx = np.concatenate([
        rng.choice(positive, size=len(positive), replace=True),
        rng.choice(negative, size=len(negative), replace=True),
    ])
    rng.shuffle(idx)
    return idx

def holm_adjust(p_values: Sequence[float]) -> np.ndarray:
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    adjusted = np.empty_like(p)
    running = 0.0
    m = len(p)
    for rank, index in enumerate(order):
        value = min(1.0, (m - rank) * p[index])
        running = max(running, value)
        adjusted[index] = running
    return adjusted

print("Core metric and selection utilities loaded.")


Core metric and selection utilities loaded.


In [ ]:
# Paired confidence intervals and significance tests.

def aligned_prompt_matrices(
    predictions: pd.DataFrame,
    dataset_name: str,
    prompt_a: str,
    prompt_b: str,
    run_names: Optional[Sequence[str]] = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[str]]:
    selected_runs = list(run_names or ENABLED_RUN_NAMES)
    frame = predictions[
        predictions["dataset"].eq(dataset_name)
        & predictions["prompt_name"].isin([prompt_a, prompt_b])
        & predictions["run_name"].isin(selected_runs)
        & predictions["replicate"].eq(0)
    ].copy()
    if frame.empty:
        raise RuntimeError("No aligned prompt predictions found.")
    if frame.duplicated(["passage_id", "run_name", "prompt_name"]).any():
        raise RuntimeError("Duplicate passage/model/prompt predictions prevent paired inference.")

    metadata = (
        frame[["passage_id", "gold", "document"]]
        .drop_duplicates("passage_id")
        .set_index("passage_id")
    )
    def matrix_for(prompt_name: str) -> pd.DataFrame:
        return (
            frame[frame["prompt_name"].eq(prompt_name)]
            .pivot(index="passage_id", columns="run_name", values="unlearning_probability")
            .reindex(columns=selected_runs)
        )
    a_df = matrix_for(prompt_a)
    b_df = matrix_for(prompt_b)
    common = metadata.index.intersection(a_df.index).intersection(b_df.index)
    metadata = metadata.loc[common]
    a_df = a_df.loc[common]
    b_df = b_df.loc[common]
    if a_df.isna().any().any() or b_df.isna().any().any():
        raise RuntimeError("Missing model probabilities in paired prompt matrices.")
    return (
        metadata["gold"].to_numpy(dtype=int),
        a_df.to_numpy(dtype=float),
        b_df.to_numpy(dtype=float),
        metadata["document"].to_numpy(dtype=str),
        common.to_numpy(dtype=str),
        selected_runs,
    )

def mean_model_metric(
    y_true: np.ndarray,
    probability_matrix: np.ndarray,
    metric: str,
    threshold: float = 0.50,
) -> float:
    values = [
        metric_from_predictions(y_true, probability_matrix[:, j], metric, threshold)
        for j in range(probability_matrix.shape[1])
    ]
    return float(np.nanmean(values))

def prompt_ab_inference(
    predictions: pd.DataFrame,
    dataset_name: str,
    prompt_a: str,
    prompt_b: str,
    *,
    run_names: Optional[Sequence[str]] = None,
    threshold: float = 0.50,
    n_bootstrap: int = N_BOOTSTRAP,
    n_permutations: int = N_PERMUTATIONS,
    seed: int = ANALYSIS_RANDOM_SEED,
) -> pd.DataFrame:
    y, prob_a, prob_b, documents, passage_ids, selected_runs = aligned_prompt_matrices(
        predictions, dataset_name, prompt_a, prompt_b, run_names
    )
    alpha = 1.0 - CI_LEVEL
    lower_q, upper_q = 100 * alpha / 2, 100 * (1 - alpha / 2)
    rows: list[dict[str, Any]] = []

    for metric in METRIC_NAMES:
        observed_a = mean_model_metric(y, prob_a, metric, threshold)
        observed_b = mean_model_metric(y, prob_b, metric, threshold)
        observed_difference = observed_b - observed_a

        rng_boot = np.random.default_rng(deterministic_seed(seed, prompt_a, prompt_b, metric, "boot"))
        boot_a = np.empty(n_bootstrap, dtype=float)
        boot_b = np.empty(n_bootstrap, dtype=float)
        for i in range(n_bootstrap):
            idx = stratified_bootstrap_indices(y, rng_boot)
            boot_a[i] = mean_model_metric(y[idx], prob_a[idx], metric, threshold)
            boot_b[i] = mean_model_metric(y[idx], prob_b[idx], metric, threshold)
        boot_diff = boot_b - boot_a

        rng_perm = np.random.default_rng(deterministic_seed(seed, prompt_a, prompt_b, metric, "perm"))
        null = np.empty(n_permutations, dtype=float)
        # Swap the two prompt assignments for every model within a passage as one cluster.
        for i in range(n_permutations):
            swap = rng_perm.random(len(y)) < 0.5
            perm_a = np.where(swap[:, None], prob_b, prob_a)
            perm_b = np.where(swap[:, None], prob_a, prob_b)
            null[i] = (
                mean_model_metric(y, perm_b, metric, threshold)
                - mean_model_metric(y, perm_a, metric, threshold)
            )
        p_value = (1 + int(np.sum(np.abs(null) >= abs(observed_difference)))) / (
            n_permutations + 1
        )
        rows.append({
            "dataset": dataset_name,
            "prompt_a": prompt_a,
            "prompt_b": prompt_b,
            "metric": metric,
            "threshold": threshold,
            "n_passages": len(y),
            "n_model_configurations": len(selected_runs),
            "prompt_a_score": observed_a,
            "prompt_a_ci_low": float(np.nanpercentile(boot_a, lower_q)),
            "prompt_a_ci_high": float(np.nanpercentile(boot_a, upper_q)),
            "prompt_b_score": observed_b,
            "prompt_b_ci_low": float(np.nanpercentile(boot_b, lower_q)),
            "prompt_b_ci_high": float(np.nanpercentile(boot_b, upper_q)),
            "difference_b_minus_a": observed_difference,
            "difference_ci_low": float(np.nanpercentile(boot_diff, lower_q)),
            "difference_ci_high": float(np.nanpercentile(boot_diff, upper_q)),
            "approx_randomization_p_value_two_sided": p_value,
            "inference_status": "exploratory_same_gpt_test_used_for_selection",
        })

    result = pd.DataFrame(rows)
    result["holm_adjusted_p_value"] = holm_adjust(
        result["approx_randomization_p_value_two_sided"]
    )
    result["significant_after_holm_0_05"] = result["holm_adjusted_p_value"] < 0.05
    return result

def mcnemar_exact(
    y_true: Sequence[int],
    probabilities_a: Sequence[float],
    probabilities_b: Sequence[float],
    threshold_a: float = 0.50,
    threshold_b: float = 0.50,
) -> dict[str, Any]:
    y = np.asarray(y_true, dtype=int)
    pred_a = np.asarray(probabilities_a, dtype=float) >= threshold_a
    pred_b = np.asarray(probabilities_b, dtype=float) >= threshold_b
    correct_a = pred_a == y
    correct_b = pred_b == y
    a_only = int(np.sum(correct_a & ~correct_b))
    b_only = int(np.sum(~correct_a & correct_b))
    discordant = a_only + b_only
    p_value = (
        float(binomtest(min(a_only, b_only), discordant, 0.5, alternative="two-sided").pvalue)
        if discordant > 0 else 1.0
    )
    return {
        "a_correct_b_wrong": a_only,
        "a_wrong_b_correct": b_only,
        "discordant_pairs": discordant,
        "exact_two_sided_p_value": p_value,
    }

def paired_system_inference(
    y_true: Sequence[int],
    probabilities_a: Sequence[float],
    probabilities_b: Sequence[float],
    *,
    system_a: str,
    system_b: str,
    threshold_a: float = 0.50,
    threshold_b: float = 0.50,
    n_bootstrap: int = N_BOOTSTRAP,
    n_permutations: int = N_PERMUTATIONS,
    seed: int = ANALYSIS_RANDOM_SEED,
    inference_status: str = "confirmatory_or_descriptive_as_registered",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    y = np.asarray(y_true, dtype=int)
    p_a = np.asarray(probabilities_a, dtype=float)
    p_b = np.asarray(probabilities_b, dtype=float)
    if not (len(y) == len(p_a) == len(p_b)):
        raise ValueError("Paired system inputs must have equal length.")

    alpha = 1.0 - CI_LEVEL
    lower_q, upper_q = 100 * alpha / 2, 100 * (1 - alpha / 2)
    rows: list[dict[str, Any]] = []
    for metric in METRIC_NAMES:
        observed_a = metric_from_predictions(y, p_a, metric, threshold_a)
        observed_b = metric_from_predictions(y, p_b, metric, threshold_b)
        observed_diff = observed_b - observed_a

        rng_boot = np.random.default_rng(deterministic_seed(seed, system_a, system_b, metric, "boot"))
        boot_a = np.empty(n_bootstrap, dtype=float)
        boot_b = np.empty(n_bootstrap, dtype=float)
        for i in range(n_bootstrap):
            idx = stratified_bootstrap_indices(y, rng_boot)
            boot_a[i] = metric_from_predictions(y[idx], p_a[idx], metric, threshold_a)
            boot_b[i] = metric_from_predictions(y[idx], p_b[idx], metric, threshold_b)
        boot_diff = boot_b - boot_a

        rng_perm = np.random.default_rng(deterministic_seed(seed, system_a, system_b, metric, "perm"))
        null = np.empty(n_permutations, dtype=float)
        for i in range(n_permutations):
            swap = rng_perm.random(len(y)) < 0.5
            perm_a = np.where(swap, p_b, p_a)
            perm_b = np.where(swap, p_a, p_b)
            null[i] = (
                metric_from_predictions(y, perm_b, metric, threshold_b)
                - metric_from_predictions(y, perm_a, metric, threshold_a)
            )
        p_value = (1 + int(np.sum(np.abs(null) >= abs(observed_diff)))) / (
            n_permutations + 1
        )
        rows.append({
            "system_a": system_a,
            "system_b": system_b,
            "metric": metric,
            "n": len(y),
            "threshold_a": threshold_a,
            "threshold_b": threshold_b,
            "system_a_score": observed_a,
            "system_a_ci_low": float(np.nanpercentile(boot_a, lower_q)),
            "system_a_ci_high": float(np.nanpercentile(boot_a, upper_q)),
            "system_b_score": observed_b,
            "system_b_ci_low": float(np.nanpercentile(boot_b, lower_q)),
            "system_b_ci_high": float(np.nanpercentile(boot_b, upper_q)),
            "difference_b_minus_a": observed_diff,
            "difference_ci_low": float(np.nanpercentile(boot_diff, lower_q)),
            "difference_ci_high": float(np.nanpercentile(boot_diff, upper_q)),
            "approx_randomization_p_value_two_sided": p_value,
            "inference_status": inference_status,
        })
    inference = pd.DataFrame(rows)
    inference["holm_adjusted_p_value"] = holm_adjust(
        inference["approx_randomization_p_value_two_sided"]
    )
    inference["significant_after_holm_0_05"] = inference["holm_adjusted_p_value"] < 0.05
    mcnemar = pd.DataFrame([{
        "system_a": system_a,
        "system_b": system_b,
        **mcnemar_exact(y, p_a, p_b, threshold_a, threshold_b),
    }])
    return inference, mcnemar

print("Paired inference utilities loaded.")


Paired inference utilities loaded.


In [ ]:
# ============================================================
# STAGE 1: Examples vs. no examples
# FAST SELECTION + CHECKPOINT
# Significance analysis is intentionally run separately later.
# ============================================================

STAGE1_PREDICTIONS = materialize_prediction_table(
    "GPT Test",
    STAGE1_PROMPTS,
    require_complete=True,
)

# Save the already-completed LLM predictions.
atomic_to_csv(
    STAGE1_PREDICTIONS,
    SUBDIRS["analysis"] / "stage1_gpt_test_predictions.csv",
)
atomic_to_parquet(
    STAGE1_PREDICTIONS,
    SUBDIRS["analysis"] / "stage1_gpt_test_predictions.parquet",
)

# ------------------------------------------------------------
# Descriptive metrics
# ------------------------------------------------------------

STAGE1_MODEL_METRICS = model_level_metrics(
    STAGE1_PREDICTIONS,
    PROMPT_SELECTION_THRESHOLD,
)

STAGE1_DOCUMENT_METRICS = document_level_metrics(
    STAGE1_PREDICTIONS,
    PROMPT_SELECTION_THRESHOLD,
)

STAGE1_LABELER_METRICS = labeler_level_metrics(
    STAGE1_PREDICTIONS,
    PROMPT_SELECTION_THRESHOLD,
)

# ------------------------------------------------------------
# Choose winner using the PRE-SPECIFIED rule:
# arithmetic mean precision across the 3 fixed LLMs
# ------------------------------------------------------------

STAGE1_WINNER_NAME, STAGE1_SELECTION_SUMMARY = choose_prompt_configuration(
    STAGE1_MODEL_METRICS,
    dataset="GPT Test",
    scope="all_rows",
)

STAGE1_WINNER_SPEC = PROMPT_REGISTRY[STAGE1_WINNER_NAME]

# ------------------------------------------------------------
# McNemar tests are very fast, so keep them here.
# These compare binary errors for each fixed LLM.
# ------------------------------------------------------------

stage1_mcnemar_rows = []

for run_name in ENABLED_RUN_NAMES:

    a = STAGE1_PREDICTIONS[
        STAGE1_PREDICTIONS["run_name"].eq(run_name)
        & STAGE1_PREDICTIONS["prompt_name"].eq(
            STAGE1_PROMPTS[0].prompt_name
        )
    ].set_index("passage_id")

    b = STAGE1_PREDICTIONS[
        STAGE1_PREDICTIONS["run_name"].eq(run_name)
        & STAGE1_PREDICTIONS["prompt_name"].eq(
            STAGE1_PROMPTS[1].prompt_name
        )
    ].set_index("passage_id")

    common = a.index.intersection(b.index)

    stage1_mcnemar_rows.append({
        "run_name": run_name,
        "provider": MODEL_BY_RUN_NAME[run_name]["provider"],
        "model": MODEL_BY_RUN_NAME[run_name]["model"],
        "prompt_a": STAGE1_PROMPTS[0].prompt_name,
        "prompt_b": STAGE1_PROMPTS[1].prompt_name,
        **mcnemar_exact(
            a.loc[common, "gold"],
            a.loc[common, "unlearning_probability"],
            b.loc[common, "unlearning_probability"],
        ),
    })

STAGE1_PER_MODEL_MCNEMAR = pd.DataFrame(stage1_mcnemar_rows)

if not STAGE1_PER_MODEL_MCNEMAR.empty:
    STAGE1_PER_MODEL_MCNEMAR["holm_adjusted_p_value"] = holm_adjust(
        STAGE1_PER_MODEL_MCNEMAR["exact_two_sided_p_value"]
    )

# ------------------------------------------------------------
# Save the Stage 1 descriptive results immediately.
# ------------------------------------------------------------

for name, frame in {
    "stage1_model_metrics.csv": STAGE1_MODEL_METRICS,
    "stage1_document_metrics.csv": STAGE1_DOCUMENT_METRICS,
    "stage1_labeler_metrics.csv": STAGE1_LABELER_METRICS,
    "stage1_selection_summary.csv": STAGE1_SELECTION_SUMMARY,
    "stage1_per_model_mcnemar.csv": STAGE1_PER_MODEL_MCNEMAR,
}.items():
    atomic_to_csv(
        frame,
        SUBDIRS["analysis"] / name,
    )

# ------------------------------------------------------------
# Lock Stage 1 winner NOW.
# Significance inference is NOT required for advancing stages.
# ------------------------------------------------------------

STAGE1_CHECKPOINT = {
    "stage": "stage1_examples_ab",
    "selection_dataset": "GPT Test",
    "selection_scope": "all_rows",
    "selection_threshold": PROMPT_SELECTION_THRESHOLD,

    "primary_selection_metric":
        "arithmetic mean of model-level precision",

    "tie_breakers": [
        "mean_recall",
        "mean_f1",
        "mean_auprc",
        "mean_auroc",
        "mean_accuracy",
        "lower_mean_brier",
        "lexical_prompt_name",
    ],

    "winner_prompt_name": STAGE1_WINNER_NAME,
    "winner_prompt_hash": STAGE1_WINNER_SPEC.prompt_hash,
    "winner_include_examples": STAGE1_WINNER_SPEC.include_examples,

    "candidate_run_names": ENABLED_RUN_NAMES,

    "selection_summary":
        STAGE1_SELECTION_SUMMARY.to_dict(orient="records"),

    "inference_status":
        "Deferred to dedicated Stage 1 statistical-inference cell",

    "inference_interpretation":
        "Prompt selection was based only on the prespecified mean-precision rule. "
        "Statistical inference is reported separately and does not alter the winner.",

    "source_workbook_sha256":
        SOURCE_WORKBOOK_SHA256_AT_LOAD,

    "codebook_whole_sheet_sha256":
        CODEBOOK_WHOLE_SHEET_SHA256,

    "created_at_utc":
        utc_now(),
}

atomic_write_json(
    SUBDIRS["checkpoints"] / "stage1_selection.json",
    STAGE1_CHECKPOINT,
)

print("=" * 70)
print("STAGE 1 COMPLETE")
print("=" * 70)
print("Winner:", STAGE1_WINNER_NAME)
print("Examples included:", STAGE1_WINNER_SPEC.include_examples)
print(
    "Checkpoint:",
    SUBDIRS["checkpoints"] / "stage1_selection.json",
)

display(STAGE1_SELECTION_SUMMARY)
display(STAGE1_MODEL_METRICS)

if not STAGE1_PER_MODEL_MCNEMAR.empty:
    display(STAGE1_PER_MODEL_MCNEMAR)

STAGE 1 COMPLETE
Winner: codebook_no_examples_no_checklist
Examples included: False
Checkpoint: /content/drive/MyDrive/Unlearning_Paragraph_Level_LLM_Experiment/paragraph_level_gpt_test_first_v2_20260821/checkpoints/stage1_selection.json


,prompt_name,prompt_hash,mean_precision,mean_recall,mean_f1,mean_auroc,mean_auprc,mean_accuracy,mean_brier,model_configurations,providers
0,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,0.793850,0.293651,0.421117,0.692555,0.686579,0.603175,0.276399,3,3
1,codebook_with_examples_no_checklist,c81e46faac5d0f8d24ff99cf635f491a89e69c02d901ccedbc210e3f59965053,0.751165,0.357143,0.472223,0.724112,0.716269,0.611111,0.254363,3,3


,dataset,prompt_name,prompt_hash,provider,run_name,model_requested,model_returned,scope,n,positive_prevalence,threshold,predicted_positive_count,tn,fp,fn,tp,accuracy,precision,recall,f1,auroc,auprc,brier
0,GPT Test,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,gpt-5.6-terra,all_rows,84,0.500000,0.5,12,41,1,31,11,0.619048,0.916667,0.261905,0.407407,0.695578,0.732280,0.293800
1,GPT Test,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,gpt-5.6-terra,human_labeled_only,84,0.500000,0.5,12,41,1,31,11,0.619048,0.916667,0.261905,0.407407,0.695578,0.732280,0.293800
2,GPT Test,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,gpt-5.6-terra,review_unflagged,50,0.560000,0.5,11,21,1,18,10,0.620000,0.909091,0.357143,0.512821,0.722403,0.799058,0.286118
3,GPT Test,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,gpt-5.6-terra,example_leakage_safe,83,0.493976,0.5,11,41,1,31,10,0.614458,0.909091,0.243902,0.384615,0.688153,0.719342,0.297329
4,GPT Test,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,anthropic,anthropic_claude_haiku_4_5_no_thinking,claude-haiku-4-5-20251001,claude-haiku-4-5-20251001,all_rows,84,0.500000,0.5,23,35,7,26,16,0.607143,0.695652,0.380952,0.492308,0.716837,0.669019,0.244951
5,GPT Test,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,anthropic,anthropic_claude_haiku_4_5_no_thinking,claude-haiku-4-5-20251001,claude-haiku-4-5-20251001,human_labeled_only,84,0.500000,0.5,23,35,7,26,16,0.607143,0.695652,0.380952,0.492308,0.716837,0.669019,0.244951
6,GPT Test,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,anthropic,anthropic_claude_haiku_4_5_no_thinking,claude-haiku-4-5-20251001,claude-haiku-4-5-20251001,review_unflagged,50,0.560000,0.5,16,18,4,16,12,0.600000,0.750000,0.428571,0.545455,0.731331,0.736373,0.245674
7,GPT Test,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,anthropic,anthropic_claude_haiku_4_5_no_thinking,claude-haiku-4-5-20251001,claude-haiku-4-5-20251001,example_leakage_safe,83,0.493976,0.5,22,35,7,26,15,0.602410,0.681818,0.365854,0.476190,0.709930,0.651122,0.247149
8,GPT Test,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,gemini-3.1-flash-lite,all_rows,84,0.500000,0.5,13,39,3,32,10,0.583333,0.769231,0.238095,0.363636,0.665249,0.658438,0.290446
9,GPT Test,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,gemini-3.1-flash-lite,human_labeled_only,84,0.500000,0.5,13,39,3,32,10,0.583333,0.769231,0.238095,0.363636,0.665249,0.658438,0.290446


,run_name,provider,model,prompt_a,prompt_b,a_correct_b_wrong,a_wrong_b_correct,discordant_pairs,exact_two_sided_p_value,holm_adjusted_p_value
0,openai_gpt_5_6_terra_low,openai,gpt-5.6-terra,codebook_no_examples_no_checklist,codebook_with_examples_no_checklist,5,2,7,0.453125,0.90625
1,anthropic_claude_haiku_4_5_no_thinking,anthropic,claude-haiku-4-5-20251001,codebook_no_examples_no_checklist,codebook_with_examples_no_checklist,6,6,12,1.000000,1.00000
2,google_gemini_3_1_flash_lite_low,google,gemini-3.1-flash-lite,codebook_no_examples_no_checklist,codebook_with_examples_no_checklist,1,6,7,0.125000,0.37500


## Stage 2 — Winning examples setting without versus with the checklist

This stage also runs on **GPT Test only**. The no-checklist arm is inherited from Stage 1 without another API call. Only the matching checklist arm is new. Prompt parity is asserted after removing the exact checklist block.

In [ ]:
# Load Stage 1 checkpoint and construct only the matching checklist arm.
stage1_checkpoint_path = SUBDIRS["checkpoints"] / "stage1_selection.json"
if not stage1_checkpoint_path.exists():
    raise RuntimeError("Run and complete Stage 1 analysis first.")
STAGE1_CHECKPOINT_LOADED = json.loads(stage1_checkpoint_path.read_text(encoding="utf-8"))
STAGE1_WINNER_NAME = STAGE1_CHECKPOINT_LOADED["winner_prompt_name"]
STAGE1_WINNER_SPEC = PROMPT_REGISTRY[STAGE1_WINNER_NAME]

if STAGE1_WINNER_SPEC.include_examples:
    checklist_prompt_name = "codebook_with_examples_with_checklist"
else:
    checklist_prompt_name = "codebook_no_examples_with_checklist"

STAGE2_NO_CHECKLIST_SPEC = STAGE1_WINNER_SPEC
STAGE2_WITH_CHECKLIST_SPEC = PROMPT_REGISTRY[checklist_prompt_name]
STAGE2_COMPARISON_PROMPTS = [STAGE2_NO_CHECKLIST_SPEC, STAGE2_WITH_CHECKLIST_SPEC]
STAGE2_NEW_CALL_PROMPTS = [STAGE2_WITH_CHECKLIST_SPEC]

# Factor parity: remove the checklist section and the two prompts must be identical.
sample_text = GPT_TEST_DF.iloc[0]["text"]
no_check = render_user_prompt(STAGE2_NO_CHECKLIST_SPEC, sample_text)
with_check = render_user_prompt(STAGE2_WITH_CHECKLIST_SPEC, sample_text)
if with_check.replace(CHECKLIST_TEXT + "\n\n", "") != no_check:
    raise AssertionError("Stage 2 prompts differ outside the checklist factor.")

print("Stage 1 winner reused without new calls:", STAGE2_NO_CHECKLIST_SPEC.prompt_name)
print("Only new Stage 2 arm:", STAGE2_WITH_CHECKLIST_SPEC.prompt_name)


Stage 1 winner reused without new calls: codebook_no_examples_no_checklist
Only new Stage 2 arm: codebook_no_examples_with_checklist


In [ ]:
# Stage 2 preflight — only the checklist arm requires new calls.
STAGE2_PREFLIGHT = pd.concat([
    build_job_rows("GPT Test", STAGE2_NEW_CALL_PROMPTS, PROVIDER_MODEL_CONFIGS[p])
    for p in ENABLED_PROVIDERS
], ignore_index=True)
STAGE2_PREFLIGHT_SUMMARY = (
    STAGE2_PREFLIGHT.groupby(
        ["provider", "run_name", "model", "prompt_name"], as_index=False
    )
    .agg(
        logical_rows=("run_key", "size"),
        cached_rows=("cached_success", "sum"),
        rough_estimated_cost_usd=("estimated_cost_usd", "sum"),
    )
)
STAGE2_PREFLIGHT_SUMMARY["new_calls"] = (
    STAGE2_PREFLIGHT_SUMMARY["logical_rows"]
    - STAGE2_PREFLIGHT_SUMMARY["cached_rows"]
)
atomic_to_csv(
    STAGE2_PREFLIGHT_SUMMARY,
    SUBDIRS["audits"] / "stage2_gpt_test_preflight.csv",
)
display(STAGE2_PREFLIGHT_SUMMARY)


,provider,run_name,model,prompt_name,logical_rows,cached_rows,rough_estimated_cost_usd,new_calls
0,anthropic,anthropic_claude_haiku_4_5_no_thinking,claude-haiku-4-5-20251001,codebook_no_examples_with_checklist,84,0,0.334807,84
1,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,codebook_no_examples_with_checklist,84,0,0.087482,84
2,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,codebook_no_examples_with_checklist,84,0,0.699854,84


In [ ]:
# Stage 2 — GPT Test — OpenAI candidate models — checklist arm only.
STAGE2_OPENAI_JOB = run_prediction_job(
    "GPT Test",
    STAGE2_NEW_CALL_PROMPTS,
    PROVIDER_MODEL_CONFIGS["openai"],
    execute=RUN_STAGE2_GPT_TEST,
)


openai: 168 successful cached run keys; 168 total records.
anthropic: 168 successful cached run keys; 170 total records.
google: 168 successful cached run keys; 186 total records.


,provider,run_name,model,prompt_name,logical_rows,cached_rows,estimated_total_cost_usd,new_calls
0,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,codebook_no_examples_with_checklist,84,0,0.699854,84


{'dataset': 'GPT Test', 'provider': 'openai', 'stage': 'stage2_checklist_ab', 'new_calls': 84, 'rough_new_cost_usd': 0.6999, 'execute': True}


openai | openai_gpt_5_6_terra_low:   0%|          | 0/84 [00:00<?, ?it/s]

Job completed without final/permanent model failures.


In [ ]:
# Stage 2 — GPT Test — Anthropic candidate models — checklist arm only.
STAGE2_ANTHROPIC_JOB = run_prediction_job(
    "GPT Test",
    STAGE2_NEW_CALL_PROMPTS,
    PROVIDER_MODEL_CONFIGS["anthropic"],
    execute=RUN_STAGE2_GPT_TEST,
)


openai: 252 successful cached run keys; 252 total records.
anthropic: 168 successful cached run keys; 170 total records.
google: 168 successful cached run keys; 186 total records.


,provider,run_name,model,prompt_name,logical_rows,cached_rows,estimated_total_cost_usd,new_calls
0,anthropic,anthropic_claude_haiku_4_5_no_thinking,claude-haiku-4-5-20251001,codebook_no_examples_with_checklist,84,0,0.334807,84


{'dataset': 'GPT Test', 'provider': 'anthropic', 'stage': 'stage2_checklist_ab', 'new_calls': 84, 'rough_new_cost_usd': 0.3348, 'execute': True}


anthropic | anthropic_claude_haiku_4_5_no_thinking:   0%|          | 0/84 [00:00<?, ?it/s]

Job completed without final/permanent model failures.


In [ ]:
# Stage 2 — GPT Test — Google candidate models — checklist arm only.
STAGE2_GOOGLE_JOB = run_prediction_job(
    "GPT Test",
    STAGE2_NEW_CALL_PROMPTS,
    PROVIDER_MODEL_CONFIGS["google"],
    execute=RUN_STAGE2_GPT_TEST,
)


openai: 252 successful cached run keys; 252 total records.
anthropic: 252 successful cached run keys; 254 total records.
google: 168 successful cached run keys; 186 total records.


,provider,run_name,model,prompt_name,logical_rows,cached_rows,estimated_total_cost_usd,new_calls
0,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,codebook_no_examples_with_checklist,84,0,0.087482,84


{'dataset': 'GPT Test', 'provider': 'google', 'stage': 'stage2_checklist_ab', 'new_calls': 84, 'rough_new_cost_usd': 0.0875, 'execute': True}


google | google_gemini_3_1_flash_lite_low:   0%|          | 0/84 [00:00<?, ?it/s]


google_gemini_3_1_flash_lite_low attempt 1 failed (transient, status=429); retrying in 2.5s: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.1-flash-lite\nPlease retry in 18.512212959s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPe

In [ ]:
# ============================================================
# STAGE 2: Checklist vs. no checklist
# FAST SELECTION + CHECKPOINT
# Statistical inference is intentionally deferred.
# ============================================================

STAGE2_PREDICTIONS = materialize_prediction_table(
    "GPT Test",
    STAGE2_COMPARISON_PROMPTS,
    require_complete=True,
)

# Save predictions immediately
atomic_to_csv(
    STAGE2_PREDICTIONS,
    SUBDIRS["analysis"] / "stage2_gpt_test_predictions.csv",
)
atomic_to_parquet(
    STAGE2_PREDICTIONS,
    SUBDIRS["analysis"] / "stage2_gpt_test_predictions.parquet",
)

# ------------------------------------------------------------
# Descriptive metrics
# ------------------------------------------------------------

STAGE2_MODEL_METRICS = model_level_metrics(
    STAGE2_PREDICTIONS,
    PROMPT_SELECTION_THRESHOLD,
)

STAGE2_DOCUMENT_METRICS = document_level_metrics(
    STAGE2_PREDICTIONS,
    PROMPT_SELECTION_THRESHOLD,
)

STAGE2_LABELER_METRICS = labeler_level_metrics(
    STAGE2_PREDICTIONS,
    PROMPT_SELECTION_THRESHOLD,
)

# ------------------------------------------------------------
# Select checklist winner using your prespecified rule:
# arithmetic mean precision across the 3 fixed LLMs
# ------------------------------------------------------------

STAGE2_WINNER_NAME, STAGE2_SELECTION_SUMMARY = choose_prompt_configuration(
    STAGE2_MODEL_METRICS,
    dataset="GPT Test",
    scope="all_rows",
)

FINAL_PROMPT_NAME = STAGE2_WINNER_NAME
FINAL_PROMPT_SPEC = PROMPT_REGISTRY[FINAL_PROMPT_NAME]

# ------------------------------------------------------------
# Fast per-model McNemar tests
# ------------------------------------------------------------

stage2_mcnemar_rows = []

for run_name in ENABLED_RUN_NAMES:

    a = STAGE2_PREDICTIONS[
        STAGE2_PREDICTIONS["run_name"].eq(run_name)
        & STAGE2_PREDICTIONS["prompt_name"].eq(
            STAGE2_NO_CHECKLIST_SPEC.prompt_name
        )
    ].set_index("passage_id")

    b = STAGE2_PREDICTIONS[
        STAGE2_PREDICTIONS["run_name"].eq(run_name)
        & STAGE2_PREDICTIONS["prompt_name"].eq(
            STAGE2_WITH_CHECKLIST_SPEC.prompt_name
        )
    ].set_index("passage_id")

    common = a.index.intersection(b.index)

    stage2_mcnemar_rows.append({
        "run_name": run_name,
        "provider": MODEL_BY_RUN_NAME[run_name]["provider"],
        "model": MODEL_BY_RUN_NAME[run_name]["model"],
        "prompt_a": STAGE2_NO_CHECKLIST_SPEC.prompt_name,
        "prompt_b": STAGE2_WITH_CHECKLIST_SPEC.prompt_name,
        **mcnemar_exact(
            a.loc[common, "gold"],
            a.loc[common, "unlearning_probability"],
            b.loc[common, "unlearning_probability"],
        ),
    })

STAGE2_PER_MODEL_MCNEMAR = pd.DataFrame(stage2_mcnemar_rows)

if not STAGE2_PER_MODEL_MCNEMAR.empty:
    STAGE2_PER_MODEL_MCNEMAR["holm_adjusted_p_value"] = holm_adjust(
        STAGE2_PER_MODEL_MCNEMAR["exact_two_sided_p_value"]
    )

# ------------------------------------------------------------
# Save all normal Stage 2 results immediately
# ------------------------------------------------------------

for name, frame in {
    "stage2_model_metrics.csv": STAGE2_MODEL_METRICS,
    "stage2_document_metrics.csv": STAGE2_DOCUMENT_METRICS,
    "stage2_labeler_metrics.csv": STAGE2_LABELER_METRICS,
    "stage2_selection_summary.csv": STAGE2_SELECTION_SUMMARY,
    "stage2_per_model_mcnemar.csv": STAGE2_PER_MODEL_MCNEMAR,
}.items():
    atomic_to_csv(
        frame,
        SUBDIRS["analysis"] / name,
    )

# ------------------------------------------------------------
# Lock the FINAL PROMPT before expensive inference
# ------------------------------------------------------------

STAGE2_CHECKPOINT = {
    "stage": "stage2_checklist_ab",
    "selection_dataset": "GPT Test",
    "selection_scope": "all_rows",
    "selection_threshold": PROMPT_SELECTION_THRESHOLD,

    "stage1_winner_prompt_name":
        STAGE2_NO_CHECKLIST_SPEC.prompt_name,

    "winner_prompt_name":
        FINAL_PROMPT_NAME,

    "winner_prompt_hash":
        FINAL_PROMPT_SPEC.prompt_hash,

    "winner_include_examples":
        FINAL_PROMPT_SPEC.include_examples,

    "winner_include_checklist":
        FINAL_PROMPT_SPEC.include_checklist,

    "primary_selection_metric":
        "arithmetic mean of model-level precision",

    "tie_breakers": [
        "mean_recall",
        "mean_f1",
        "mean_auprc",
        "mean_auroc",
        "mean_accuracy",
        "lower_mean_brier",
        "lexical_prompt_name",
    ],

    "candidate_run_names":
        ENABLED_RUN_NAMES,

    "selection_summary":
        STAGE2_SELECTION_SUMMARY.to_dict(orient="records"),

    "inference_status":
        "Deferred to dedicated Stage 2 statistical-inference cell",

    "inference_interpretation":
        "Checklist selection was based only on the prespecified "
        "mean-precision rule. Statistical inference is reported "
        "separately and does not alter the selected final prompt.",

    "source_workbook_sha256":
        SOURCE_WORKBOOK_SHA256_AT_LOAD,

    "codebook_whole_sheet_sha256":
        CODEBOOK_WHOLE_SHEET_SHA256,

    "created_at_utc":
        utc_now(),
}

atomic_write_json(
    SUBDIRS["checkpoints"] / "stage2_selection.json",
    STAGE2_CHECKPOINT,
)

print("=" * 70)
print("STAGE 2 COMPLETE")
print("=" * 70)

print("FINAL PROMPT:", FINAL_PROMPT_NAME)
print("Examples included:", FINAL_PROMPT_SPEC.include_examples)
print("Checklist included:", FINAL_PROMPT_SPEC.include_checklist)
print(
    "Checkpoint:",
    SUBDIRS["checkpoints"] / "stage2_selection.json",
)

display(STAGE2_SELECTION_SUMMARY)
display(STAGE2_MODEL_METRICS)

if not STAGE2_PER_MODEL_MCNEMAR.empty:
    display(STAGE2_PER_MODEL_MCNEMAR)

STAGE 2 COMPLETE
FINAL PROMPT: codebook_no_examples_with_checklist
Examples included: False
Checklist included: True
Checkpoint: /content/drive/MyDrive/Unlearning_Paragraph_Level_LLM_Experiment/paragraph_level_gpt_test_first_v2_20260821/checkpoints/stage2_selection.json


,prompt_name,prompt_hash,mean_precision,mean_recall,mean_f1,mean_auroc,mean_auprc,mean_accuracy,mean_brier,model_configurations,providers
0,codebook_no_examples_with_checklist,5ef0ed097ed63718a949e101ae68ad12b372d8fb0b0f5d1f6902d6007c2fa4c9,0.898551,0.269841,0.399397,0.726096,0.722129,0.607143,0.279974,3,3
1,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,0.793850,0.293651,0.421117,0.692555,0.686579,0.603175,0.276399,3,3


,dataset,prompt_name,prompt_hash,provider,run_name,model_requested,model_returned,scope,n,positive_prevalence,threshold,predicted_positive_count,tn,fp,fn,tp,accuracy,precision,recall,f1,auroc,auprc,brier
0,GPT Test,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,gpt-5.6-terra,all_rows,84,0.500000,0.5,12,41,1,31,11,0.619048,0.916667,0.261905,0.407407,0.695578,0.732280,0.293800
1,GPT Test,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,gpt-5.6-terra,human_labeled_only,84,0.500000,0.5,12,41,1,31,11,0.619048,0.916667,0.261905,0.407407,0.695578,0.732280,0.293800
2,GPT Test,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,gpt-5.6-terra,review_unflagged,50,0.560000,0.5,11,21,1,18,10,0.620000,0.909091,0.357143,0.512821,0.722403,0.799058,0.286118
3,GPT Test,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,gpt-5.6-terra,example_leakage_safe,83,0.493976,0.5,11,41,1,31,10,0.614458,0.909091,0.243902,0.384615,0.688153,0.719342,0.297329
4,GPT Test,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,anthropic,anthropic_claude_haiku_4_5_no_thinking,claude-haiku-4-5-20251001,claude-haiku-4-5-20251001,all_rows,84,0.500000,0.5,23,35,7,26,16,0.607143,0.695652,0.380952,0.492308,0.716837,0.669019,0.244951
5,GPT Test,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,anthropic,anthropic_claude_haiku_4_5_no_thinking,claude-haiku-4-5-20251001,claude-haiku-4-5-20251001,human_labeled_only,84,0.500000,0.5,23,35,7,26,16,0.607143,0.695652,0.380952,0.492308,0.716837,0.669019,0.244951
6,GPT Test,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,anthropic,anthropic_claude_haiku_4_5_no_thinking,claude-haiku-4-5-20251001,claude-haiku-4-5-20251001,review_unflagged,50,0.560000,0.5,16,18,4,16,12,0.600000,0.750000,0.428571,0.545455,0.731331,0.736373,0.245674
7,GPT Test,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,anthropic,anthropic_claude_haiku_4_5_no_thinking,claude-haiku-4-5-20251001,claude-haiku-4-5-20251001,example_leakage_safe,83,0.493976,0.5,22,35,7,26,15,0.602410,0.681818,0.365854,0.476190,0.709930,0.651122,0.247149
8,GPT Test,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,gemini-3.1-flash-lite,all_rows,84,0.500000,0.5,13,39,3,32,10,0.583333,0.769231,0.238095,0.363636,0.665249,0.658438,0.290446
9,GPT Test,codebook_no_examples_no_checklist,606c2c59c87cbb99cf42d3aac351a881ecbf0328ca0c2ef249fd9da313710dfe,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,gemini-3.1-flash-lite,human_labeled_only,84,0.500000,0.5,13,39,3,32,10,0.583333,0.769231,0.238095,0.363636,0.665249,0.658438,0.290446


,run_name,provider,model,prompt_a,prompt_b,a_correct_b_wrong,a_wrong_b_correct,discordant_pairs,exact_two_sided_p_value,holm_adjusted_p_value
0,openai_gpt_5_6_terra_low,openai,gpt-5.6-terra,codebook_no_examples_no_checklist,codebook_no_examples_with_checklist,3,2,5,1.0000,1.0
1,anthropic_claude_haiku_4_5_no_thinking,anthropic,claude-haiku-4-5-20251001,codebook_no_examples_no_checklist,codebook_no_examples_with_checklist,3,3,6,1.0000,1.0
2,google_gemini_3_1_flash_lite_low,google,gemini-3.1-flash-lite,codebook_no_examples_no_checklist,codebook_no_examples_with_checklist,2,4,6,0.6875,1.0


## Stage 3 — Select one best model per provider and export GPT Test predictions

Under the locked final prompt, the notebook selects one model within OpenAI, Anthropic, and Google using GPT Test precision at threshold 0.50, followed by declared deterministic tie-breakers. Bootstrap intervals and exploratory paired winner-versus-runner tests are saved. The first manual-review workbook contains every GPT Test row, Anmol and Prerana label columns, and only the selected three provider predictions.

In [ ]:
# ============================================================
# LOCK THE THREE FIXED PROVIDER MODELS
# No within-provider model selection is being performed.
# ============================================================

stage2_checkpoint_path = SUBDIRS["checkpoints"] / "stage2_selection.json"

if not stage2_checkpoint_path.exists():
    raise RuntimeError(
        "Complete Stage 2 analysis before locking the final models."
    )

STAGE2_CHECKPOINT_LOADED = json.loads(
    stage2_checkpoint_path.read_text(encoding="utf-8")
)

FINAL_PROMPT_NAME = STAGE2_CHECKPOINT_LOADED["winner_prompt_name"]
FINAL_PROMPT_SPEC = PROMPT_REGISTRY[FINAL_PROMPT_NAME]

# ------------------------------------------------------------
# Materialize predictions already generated for the final prompt.
# This does NOT make new API calls.
# ------------------------------------------------------------

FINAL_PROMPT_PREDICTIONS = materialize_prediction_table(
    "GPT Test",
    [FINAL_PROMPT_SPEC],
    require_complete=True,
)

atomic_to_csv(
    FINAL_PROMPT_PREDICTIONS,
    SUBDIRS["analysis"] / "final_prompt_gpt_test_predictions.csv",
)

atomic_to_parquet(
    FINAL_PROMPT_PREDICTIONS,
    SUBDIRS["analysis"] / "final_prompt_gpt_test_predictions.parquet",
)

# ------------------------------------------------------------
# Ordinary model metrics at threshold 0.50
# ------------------------------------------------------------

FINAL_PROMPT_MODEL_METRICS = model_level_metrics(
    FINAL_PROMPT_PREDICTIONS,
    threshold=0.50,
)

FINAL_PROMPT_MODEL_METRICS_ALL = (
    FINAL_PROMPT_MODEL_METRICS[
        FINAL_PROMPT_MODEL_METRICS["scope"].eq("all_rows")
    ]
    .copy()
)

# ------------------------------------------------------------
# The experiment uses exactly ONE preselected model/provider.
# Therefore ENABLED_RUN_NAMES themselves are the selected models.
# ------------------------------------------------------------

SELECTED_RUN_NAMES = list(ENABLED_RUN_NAMES)

SELECTED_MODEL_CONFIGS = configs_from_run_names(
    SELECTED_RUN_NAMES
)

if len(SELECTED_RUN_NAMES) != 3:
    raise AssertionError(
        f"Expected exactly 3 enabled models, found "
        f"{len(SELECTED_RUN_NAMES)}: {SELECTED_RUN_NAMES}"
    )

selected_providers = {
    config["provider"]
    for config in SELECTED_MODEL_CONFIGS
}

if selected_providers != set(EXPECTED_PROVIDERS):
    raise AssertionError(
        "Expected exactly one model from each provider. "
        f"Found providers: {sorted(selected_providers)}"
    )

# Explicitly verify one model per provider.
provider_counts = pd.Series(
    [config["provider"] for config in SELECTED_MODEL_CONFIGS]
).value_counts()

if not (provider_counts == 1).all():
    raise AssertionError(
        "There must be exactly one enabled model per provider.\n"
        f"{provider_counts}"
    )

# ------------------------------------------------------------
# Build selected-model table from the already-computed metrics.
# ------------------------------------------------------------

SELECTED_MODEL_TABLE = (
    FINAL_PROMPT_MODEL_METRICS_ALL[
        FINAL_PROMPT_MODEL_METRICS_ALL["run_name"].isin(
            SELECTED_RUN_NAMES
        )
    ]
    .copy()
)

if len(SELECTED_MODEL_TABLE) != 3:
    raise AssertionError(
        "Expected exactly three selected-model metric rows."
    )

# Put them in consistent provider order.
SELECTED_MODEL_TABLE["provider_order"] = (
    SELECTED_MODEL_TABLE["provider"]
    .map({
        provider: i
        for i, provider in enumerate(EXPECTED_PROVIDERS)
    })
)

SELECTED_MODEL_TABLE = (
    SELECTED_MODEL_TABLE
    .sort_values("provider_order")
    .drop(columns="provider_order")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Save metrics and locked model list.
# ------------------------------------------------------------

atomic_to_csv(
    FINAL_PROMPT_MODEL_METRICS,
    SUBDIRS["analysis"] / "final_prompt_model_metrics.csv",
)

atomic_to_csv(
    SELECTED_MODEL_TABLE,
    SUBDIRS["analysis"] / "selected_model_per_provider.csv",
)

# ------------------------------------------------------------
# Save checkpoint
# ------------------------------------------------------------

MODEL_SELECTION_CHECKPOINT = {
    "stage": "fixed_models_locked",

    "selection_dataset": "GPT Test",

    "prompt_name": FINAL_PROMPT_NAME,
    "prompt_hash": FINAL_PROMPT_SPEC.prompt_hash,

    "selection_threshold": 0.50,

    "model_selection_performed": False,

    "reason": (
        "The experimental design prespecified exactly one model "
        "from each provider; no within-provider model comparison "
        "was intended."
    ),

    "selected_run_names": SELECTED_RUN_NAMES,

    "selected_models": [
        {
            key: config.get(key)
            for key in [
                "run_name",
                "provider",
                "model",
                "display_name",
                "reasoning_effort",
                "thinking",
                "effort",
                "thinking_level",
            ]
            if key in config
        }
        for config in SELECTED_MODEL_CONFIGS
    ],

    "selected_model_metrics":
        SELECTED_MODEL_TABLE.to_dict(orient="records"),

    "source_workbook_sha256":
        SOURCE_WORKBOOK_SHA256_AT_LOAD,

    "codebook_whole_sheet_sha256":
        CODEBOOK_WHOLE_SHEET_SHA256,

    "created_at_utc":
        utc_now(),
}

atomic_write_json(
    SUBDIRS["checkpoints"] / "selected_models.json",
    MODEL_SELECTION_CHECKPOINT,
)

print("=" * 70)
print("THREE FIXED PROVIDER MODELS LOCKED")
print("=" * 70)

print("Final prompt:", FINAL_PROMPT_NAME)
print("Selected runs:", SELECTED_RUN_NAMES)

display(
    SELECTED_MODEL_TABLE[[
        "provider",
        "run_name",
        "model_requested",
        "precision",
        "recall",
        "f1",
        "auroc",
        "auprc",
        "accuracy",
        "brier",
    ]]
)

THREE FIXED PROVIDER MODELS LOCKED
Final prompt: codebook_no_examples_with_checklist
Selected runs: ['openai_gpt_5_6_terra_low', 'anthropic_claude_haiku_4_5_no_thinking', 'google_gemini_3_1_flash_lite_low']


,provider,run_name,model_requested,precision,recall,f1,auroc,auprc,accuracy,brier
0,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,1.000000,0.214286,0.352941,0.754252,0.760710,0.607143,0.308287
1,anthropic,anthropic_claude_haiku_4_5_no_thinking,claude-haiku-4-5-20251001,0.695652,0.380952,0.492308,0.691043,0.660127,0.607143,0.257737
2,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,1.000000,0.214286,0.352941,0.732993,0.745550,0.607143,0.273899


In [ ]:
# ============================================================
# Compatibility tables for downstream exports
# ============================================================
# No within-provider model selection was performed because the
# experiment uses exactly one prespecified model per provider.
#
# Model-performance confidence intervals are deferred to the
# dedicated statistical-inference section so they do not block
# the experimental workflow.

FINAL_PROMPT_MODEL_CIS = pd.DataFrame([{
    "status": "deferred",
    "analysis": "fixed_model_metric_confidence_intervals",
    "reason": (
        "Confidence intervals for the three fixed provider models "
        "will be calculated in the dedicated statistical-inference stage."
    ),
}])

MODEL_SELECTION_INFERENCE = pd.DataFrame([{
    "status": "not_applicable",
    "analysis": "within_provider_model_selection_inference",
    "reason": (
        "Exactly one prespecified model was used from each provider; "
        "there were no within-provider runner-up models to compare."
    ),
}])

MODEL_SELECTION_MCNEMAR = pd.DataFrame([{
    "status": "not_applicable",
    "analysis": "within_provider_model_selection_mcnemar",
    "reason": (
        "Exactly one prespecified model was used from each provider; "
        "there were no paired winner-versus-runner model comparisons."
    ),
}])

print("Downstream compatibility tables created.")
display(FINAL_PROMPT_MODEL_CIS)
display(MODEL_SELECTION_INFERENCE)
display(MODEL_SELECTION_MCNEMAR)

Downstream compatibility tables created.


,status,analysis,reason
0,deferred,fixed_model_metric_confidence_intervals,Confidence intervals for the three fixed provider models will be calculated in the dedicated statistical-inference stage.


,status,analysis,reason
0,not_applicable,within_provider_model_selection_inference,Exactly one prespecified model was used from each provider; there were no within-provider runner-up models to compare.


,status,analysis,reason
0,not_applicable,within_provider_model_selection_mcnemar,Exactly one prespecified model was used from each provider; there were no paired winner-versus-runner model comparisons.


In [ ]:
# Create the selected-model GPT Test manual-review workbook.
SELECTED_GPT_TEST_PREDICTIONS = materialize_prediction_table(
    "GPT Test",
    [FINAL_PROMPT_SPEC],
    run_names=SELECTED_RUN_NAMES,
    require_complete=True,
)

review_base_columns = [
    "passage_id", "original_number", "reference", "text", "document", "source_file",
    "gold_label", "gold", "old_gold_unlearning", "new_gold_unlearning",
    "anmol_unlearning", "prerana_unlearning", "kyle_unlearning",
    "original_labeler", "human_labelers", "selected_codes",
    "adjudication_rationale", "latest_label_source", "rationale_source",
    "review_flag", "review_issue", "label_history", "source_records",
    "example_overlap_risk", "example_max_token_jaccard",
]
MANUAL_REVIEW_DF = GPT_TEST_DF[review_base_columns].copy()
MANUAL_REVIEW_DF = MANUAL_REVIEW_DF.rename(columns={
    "passage_id": "Passage_ID",
    "original_number": "Original_Number",
    "reference": "Reference",
    "text": "Text_Content",
    "document": "Document",
    "source_file": "Source_File",
    "gold_label": "Gold_Unlearning",
    "gold": "Gold_Binary",
    "old_gold_unlearning": "Old_Gold_Unlearning",
    "new_gold_unlearning": "New_Gold_Unlearning",
    "anmol_unlearning": "Anmol_Label",
    "prerana_unlearning": "Prerana_Label",
    "kyle_unlearning": "Kyle_Label",
    "original_labeler": "Original_Labeler",
    "human_labelers": "Human_Labelers",
    "selected_codes": "Selected_Codes",
    "adjudication_rationale": "Adjudication_Rationale",
    "latest_label_source": "Latest_Label_Source",
    "rationale_source": "Rationale_Source",
    "review_flag": "Extraction_Review_Flag",
    "review_issue": "Extraction_Review_Issue",
    "label_history": "Label_History",
    "source_records": "Source_Records",
    "example_overlap_risk": "Codebook_Example_Overlap_Risk",
    "example_max_token_jaccard": "Codebook_Example_Max_Jaccard",
})

selected_long = SELECTED_GPT_TEST_PREDICTIONS.copy()
for provider in EXPECTED_PROVIDERS:
    provider_frame = selected_long[selected_long["provider"].eq(provider)].copy()
    if provider_frame["run_name"].nunique() != 1:
        raise AssertionError(f"Expected one selected {provider} model.")
    provider_frame["correct_at_0_5"] = (
        provider_frame["predicted_label_at_0_5"].astype(int)
        == provider_frame["gold"].astype(int)
    )
    provider_frame = provider_frame[[
        "passage_id", "run_name", "model_requested", "model_returned",
        "unlearning_probability", "unlearning_label", "predicted_label_at_0_5",
        "correct_at_0_5", "label_probability_inconsistent", "target_category",
        "target_label_inconsistent", "evidence_quote", "evidence_quote_valid",
        "rationale",
    ]].rename(columns={
        "passage_id": "Passage_ID",
        "run_name": f"{provider}_run_name",
        "model_requested": f"{provider}_model_requested",
        "model_returned": f"{provider}_model_returned",
        "unlearning_probability": f"{provider}_probability",
        "unlearning_label": f"{provider}_returned_label",
        "predicted_label_at_0_5": f"{provider}_prediction_0_5",
        "correct_at_0_5": f"{provider}_correct_0_5",
        "label_probability_inconsistent": f"{provider}_label_probability_inconsistent",
        "target_category": f"{provider}_target_category",
        "target_label_inconsistent": f"{provider}_target_label_inconsistent",
        "evidence_quote": f"{provider}_evidence_quote",
        "evidence_quote_valid": f"{provider}_evidence_valid",
        "rationale": f"{provider}_rationale",
    })
    MANUAL_REVIEW_DF = MANUAL_REVIEW_DF.merge(
        provider_frame, on="Passage_ID", how="left", validate="one_to_one"
    )

probability_columns = [f"{p}_probability" for p in EXPECTED_PROVIDERS]
prediction_columns = [f"{p}_prediction_0_5" for p in EXPECTED_PROVIDERS]
correct_columns = [f"{p}_correct_0_5" for p in EXPECTED_PROVIDERS]
MANUAL_REVIEW_DF["Selected_Model_Mean_Probability"] = MANUAL_REVIEW_DF[probability_columns].mean(axis=1)
MANUAL_REVIEW_DF["Selected_Model_Simple_Average_Prediction_0_5"] = (
    MANUAL_REVIEW_DF["Selected_Model_Mean_Probability"] >= 0.50
).astype(int)
MANUAL_REVIEW_DF["Selected_Model_Vote_Yes_Count"] = MANUAL_REVIEW_DF[prediction_columns].sum(axis=1)
MANUAL_REVIEW_DF["Selected_Model_Majority_Prediction"] = (
    MANUAL_REVIEW_DF["Selected_Model_Vote_Yes_Count"] >= 2
).astype(int)
MANUAL_REVIEW_DF["Selected_Model_Wrong_Count"] = (
    3 - MANUAL_REVIEW_DF[correct_columns].astype(int).sum(axis=1)
)
MANUAL_REVIEW_DF["All_Three_Selected_Models_Wrong"] = MANUAL_REVIEW_DF[
    "Selected_Model_Wrong_Count"
].eq(3)
MANUAL_REVIEW_DF["Most_Selected_Models_Wrong"] = MANUAL_REVIEW_DF[
    "Selected_Model_Wrong_Count"
].ge(2)
MANUAL_REVIEW_DF["Selected_Models_Disagree"] = MANUAL_REVIEW_DF[
    "Selected_Model_Vote_Yes_Count"
].isin([1, 2])
MANUAL_REVIEW_DF["Majority_Vote_Correct"] = (
    MANUAL_REVIEW_DF["Selected_Model_Majority_Prediction"]
    == MANUAL_REVIEW_DF["Gold_Binary"]
)
MANUAL_REVIEW_DF["Mean_Probability_Distance_From_0_5"] = (
    MANUAL_REVIEW_DF["Selected_Model_Mean_Probability"] - 0.50
).abs()

# Lower numeric value = earlier manual review.
MANUAL_REVIEW_DF["Review_Priority_Group"] = np.select(
    [
        MANUAL_REVIEW_DF["All_Three_Selected_Models_Wrong"],
        MANUAL_REVIEW_DF["Most_Selected_Models_Wrong"],
        ~MANUAL_REVIEW_DF["Majority_Vote_Correct"],
        MANUAL_REVIEW_DF["Selected_Models_Disagree"],
        MANUAL_REVIEW_DF["Extraction_Review_Flag"].fillna(False).astype(bool),
    ],
    [1, 2, 3, 4, 5],
    default=6,
)
MANUAL_REVIEW_DF = MANUAL_REVIEW_DF.sort_values(
    [
        "Review_Priority_Group", "Selected_Model_Wrong_Count",
        "Mean_Probability_Distance_From_0_5", "Document", "Passage_ID",
    ],
    ascending=[True, False, True, True, True],
    kind="mergesort",
).reset_index(drop=True)
MANUAL_REVIEW_DF.insert(0, "Manual_Review_Order", np.arange(1, len(MANUAL_REVIEW_DF) + 1))

MOST_MODELS_WRONG_DF = MANUAL_REVIEW_DF[
    MANUAL_REVIEW_DF["Most_Selected_Models_Wrong"]
].copy()

review_readme = pd.DataFrame({
    "Item": [
        "Purpose", "Predictions included", "Human labeler columns",
        "Priority group 1", "Priority group 2", "Priority group 3",
        "Important limitation",
    ],
    "Description": [
        "Manual error analysis of every GPT Test paragraph after selecting one model per provider.",
        "Only the selected OpenAI, Anthropic, and Google model under the locked final prompt.",
        "Anmol_Label and Prerana_Label preserve Yes, No, or Not labeled; no missing label is silently converted to No.",
        "All three selected models are wrong at threshold 0.50.",
        "At least two of the three selected models are wrong.",
        "The selected-model majority vote is wrong.",
        "Model and prompt comparisons are exploratory because GPT Test is also used for selection.",
    ],
})

SELECTED_MODEL_REVIEW_XLSX = SUBDIRS["exports"] / "GPT_Test_Selected_Models_Manual_Review.xlsx"
with pd.ExcelWriter(SELECTED_MODEL_REVIEW_XLSX, engine="xlsxwriter") as writer:
    review_readme.to_excel(writer, index=False, sheet_name="README")
    MANUAL_REVIEW_DF.to_excel(writer, index=False, sheet_name="Manual Review")
    MOST_MODELS_WRONG_DF.to_excel(writer, index=False, sheet_name="Most Models Wrong")
    SELECTED_GPT_TEST_PREDICTIONS.to_excel(writer, index=False, sheet_name="Selected Predictions Long")
    SELECTED_MODEL_TABLE.to_excel(writer, index=False, sheet_name="Selected Models")
    FINAL_PROMPT_MODEL_METRICS.to_excel(writer, index=False, sheet_name="Candidate Metrics")
    FINAL_PROMPT_MODEL_CIS.to_excel(writer, index=False, sheet_name="Metric CIs")
    MODEL_SELECTION_INFERENCE.to_excel(writer, index=False, sheet_name="Model Comparisons")
    MODEL_SELECTION_MCNEMAR.to_excel(writer, index=False, sheet_name="McNemar")
    STAGE1_SELECTION_SUMMARY.to_excel(writer, index=False, sheet_name="Examples A-B")
    STAGE2_SELECTION_SUMMARY.to_excel(writer, index=False, sheet_name="Checklist A-B")

    workbook = writer.book
    header_format = workbook.add_format({
        "bold": True, "text_wrap": True, "valign": "top", "border": 1,
        "bg_color": "#D9EAF7",
    })
    wrap_format = workbook.add_format({"text_wrap": True, "valign": "top"})
    percent_format = workbook.add_format({"num_format": "0.000", "valign": "top"})
    for sheet_name, frame in {
        "README": review_readme,
        "Manual Review": MANUAL_REVIEW_DF,
        "Most Models Wrong": MOST_MODELS_WRONG_DF,
        "Selected Predictions Long": SELECTED_GPT_TEST_PREDICTIONS,
        "Selected Models": SELECTED_MODEL_TABLE,
        "Candidate Metrics": FINAL_PROMPT_MODEL_METRICS,
        "Metric CIs": FINAL_PROMPT_MODEL_CIS,
        "Model Comparisons": MODEL_SELECTION_INFERENCE,
        "McNemar": MODEL_SELECTION_MCNEMAR,
        "Examples A-B": STAGE1_SELECTION_SUMMARY,
        "Checklist A-B": STAGE2_SELECTION_SUMMARY,
    }.items():
        ws = writer.sheets[sheet_name]
        ws.freeze_panes(1, 0)
        ws.autofilter(0, 0, max(len(frame), 1), max(len(frame.columns) - 1, 0))
        for col_idx, column in enumerate(frame.columns):
            ws.write(0, col_idx, column, header_format)
            series = frame[column].astype(str) if len(frame) else pd.Series(dtype=str)
            max_len = max([len(str(column)), *(series.head(500).map(len).tolist())] or [10])
            width = min(max(max_len + 2, 10), 55)
            fmt = wrap_format
            if any(token in column.lower() for token in ["probability", "precision", "recall", "accuracy", "f1", "auroc", "auprc", "brier", "ci_"]):
                fmt = percent_format
            ws.set_column(col_idx, col_idx, width, fmt)
        ws.set_default_row(30)

print("Selected-model manual review workbook:", SELECTED_MODEL_REVIEW_XLSX)
print("Rows where most selected models are wrong:", len(MOST_MODELS_WRONG_DF))


Selected-model manual review workbook: /content/drive/MyDrive/Unlearning_Paragraph_Level_LLM_Experiment/paragraph_level_gpt_test_first_v2_20260821/exports/GPT_Test_Selected_Models_Manual_Review.xlsx
Rows where most selected models are wrong: 33


## Stage 4 — Direct-only baseline on GPT Test

Only the three selected provider models are called. The baseline system is the simple arithmetic mean of their direct-only probabilities at the fixed threshold 0.50. No codebook or checklist is supplied.

In [ ]:
# Load selected-model checkpoint and preflight the direct-only GPT Test baseline.
selected_checkpoint_path = SUBDIRS["checkpoints"] / "selected_models.json"
if not selected_checkpoint_path.exists():
    raise RuntimeError("Complete model selection before running the direct-only baseline.")
SELECTED_MODEL_CHECKPOINT_LOADED = json.loads(
    selected_checkpoint_path.read_text(encoding="utf-8")
)
SELECTED_RUN_NAMES = SELECTED_MODEL_CHECKPOINT_LOADED["selected_run_names"]
SELECTED_MODEL_CONFIGS = configs_from_run_names(SELECTED_RUN_NAMES)
SELECTED_BY_PROVIDER = {
    config["provider"]: config for config in SELECTED_MODEL_CONFIGS
}
if set(SELECTED_BY_PROVIDER) != set(EXPECTED_PROVIDERS):
    raise AssertionError("Selected checkpoint must contain one model per provider.")

BASELINE_PREFLIGHT = pd.concat([
    build_job_rows("GPT Test", [DIRECT_BASELINE_SPEC], [SELECTED_BY_PROVIDER[p]])
    for p in EXPECTED_PROVIDERS
], ignore_index=True)
BASELINE_PREFLIGHT_SUMMARY = (
    BASELINE_PREFLIGHT.groupby(
        ["provider", "run_name", "model", "prompt_name"], as_index=False
    )
    .agg(
        logical_rows=("run_key", "size"),
        cached_rows=("cached_success", "sum"),
        rough_estimated_cost_usd=("estimated_cost_usd", "sum"),
    )
)
BASELINE_PREFLIGHT_SUMMARY["new_calls"] = (
    BASELINE_PREFLIGHT_SUMMARY["logical_rows"]
    - BASELINE_PREFLIGHT_SUMMARY["cached_rows"]
)
atomic_to_csv(
    BASELINE_PREFLIGHT_SUMMARY,
    SUBDIRS["audits"] / "baseline_gpt_test_preflight.csv",
)
display(BASELINE_PREFLIGHT_SUMMARY)


,provider,run_name,model,prompt_name,logical_rows,cached_rows,rough_estimated_cost_usd,new_calls
0,anthropic,anthropic_claude_haiku_4_5_no_thinking,claude-haiku-4-5-20251001,direct_target_only,84,0,0.121665,84
1,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,direct_target_only,84,0,0.034196,84
2,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,direct_target_only,84,0,0.273570,84


In [ ]:
# Direct-only baseline — GPT Test — selected OpenAI model only.
BASELINE_OPENAI_JOB = run_prediction_job(
    "GPT Test",
    [DIRECT_BASELINE_SPEC],
    [SELECTED_BY_PROVIDER["openai"]],
    execute=RUN_BASELINE_GPT_TEST,
)


openai: 252 successful cached run keys; 252 total records.
anthropic: 252 successful cached run keys; 254 total records.
google: 252 successful cached run keys; 276 total records.


,provider,run_name,model,prompt_name,logical_rows,cached_rows,estimated_total_cost_usd,new_calls
0,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,direct_target_only,84,0,0.27357,84


{'dataset': 'GPT Test', 'provider': 'openai', 'stage': 'baseline', 'new_calls': 84, 'rough_new_cost_usd': 0.2736, 'execute': True}


openai | openai_gpt_5_6_terra_low:   0%|          | 0/84 [00:00<?, ?it/s]

Job completed without final/permanent model failures.


In [ ]:
# Direct-only baseline — GPT Test — selected Anthropic model only.
BASELINE_ANTHROPIC_JOB = run_prediction_job(
    "GPT Test",
    [DIRECT_BASELINE_SPEC],
    [SELECTED_BY_PROVIDER["anthropic"]],
    execute=RUN_BASELINE_GPT_TEST,
)


openai: 336 successful cached run keys; 336 total records.
anthropic: 252 successful cached run keys; 254 total records.
google: 252 successful cached run keys; 276 total records.


,provider,run_name,model,prompt_name,logical_rows,cached_rows,estimated_total_cost_usd,new_calls
0,anthropic,anthropic_claude_haiku_4_5_no_thinking,claude-haiku-4-5-20251001,direct_target_only,84,0,0.121665,84


{'dataset': 'GPT Test', 'provider': 'anthropic', 'stage': 'baseline', 'new_calls': 84, 'rough_new_cost_usd': 0.1217, 'execute': True}


anthropic | anthropic_claude_haiku_4_5_no_thinking:   0%|          | 0/84 [00:00<?, ?it/s]

Job completed without final/permanent model failures.


In [ ]:
# Direct-only baseline — GPT Test — selected Google model only.
BASELINE_GOOGLE_JOB = run_prediction_job(
    "GPT Test",
    [DIRECT_BASELINE_SPEC],
    [SELECTED_BY_PROVIDER["google"]],
    execute=RUN_BASELINE_GPT_TEST,
)


openai: 336 successful cached run keys; 336 total records.
anthropic: 336 successful cached run keys; 338 total records.
google: 252 successful cached run keys; 276 total records.


,provider,run_name,model,prompt_name,logical_rows,cached_rows,estimated_total_cost_usd,new_calls
0,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,direct_target_only,84,0,0.034196,84


{'dataset': 'GPT Test', 'provider': 'google', 'stage': 'baseline', 'new_calls': 84, 'rough_new_cost_usd': 0.0342, 'execute': True}


google | google_gemini_3_1_flash_lite_low:   0%|          | 0/84 [00:00<?, ?it/s]


google_gemini_3_1_flash_lite_low attempt 1 failed (transient, status=503); retrying in 2.5s: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

google_gemini_3_1_flash_lite_low attempt 1 failed (transient, status=503); retrying in 2.5s: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

google_gemini_3_1_flash_lite_low attempt 1 failed (transient, status=429); retrying in 2.5s: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metri

In [ ]:
# Materialize selected-model final-prompt and direct-baseline probability matrices.
SELECTED_FINAL_PROMPT_PREDICTIONS = materialize_prediction_table(
    "GPT Test",
    [FINAL_PROMPT_SPEC],
    run_names=SELECTED_RUN_NAMES,
    require_complete=True,
)
SELECTED_BASELINE_PREDICTIONS = materialize_prediction_table(
    "GPT Test",
    [DIRECT_BASELINE_SPEC],
    run_names=SELECTED_RUN_NAMES,
    require_complete=True,
)

def probability_matrix(
    predictions: pd.DataFrame,
    run_names: Sequence[str],
) -> pd.DataFrame:
    metadata = GPT_TEST_DF[[
        "passage_id", "gold", "document", "text", "text_sha256"
    ]].set_index("passage_id")
    pivot = predictions.pivot(
        index="passage_id", columns="run_name", values="unlearning_probability"
    ).reindex(index=metadata.index, columns=list(run_names))
    if pivot.isna().any().any():
        raise RuntimeError("Missing selected-model probabilities in the matrix.")
    return metadata.join(pivot)

FINAL_SELECTED_MATRIX = probability_matrix(
    SELECTED_FINAL_PROMPT_PREDICTIONS, SELECTED_RUN_NAMES
)
BASELINE_SELECTED_MATRIX = probability_matrix(
    SELECTED_BASELINE_PREDICTIONS, SELECTED_RUN_NAMES
)

if not FINAL_SELECTED_MATRIX.index.equals(BASELINE_SELECTED_MATRIX.index):
    raise AssertionError("Final-prompt and baseline matrices are not row-aligned.")

BASELINE_SIMPLE_AVERAGE_PROBABILITY = BASELINE_SELECTED_MATRIX[
    SELECTED_RUN_NAMES
].mean(axis=1).to_numpy(dtype=float)
BASELINE_SIMPLE_AVERAGE_PREDICTION = (
    BASELINE_SIMPLE_AVERAGE_PROBABILITY >= 0.50
).astype(int)
GPT_TEST_Y = FINAL_SELECTED_MATRIX["gold"].to_numpy(dtype=int)

BASELINE_SIMPLE_AVERAGE_METRICS = pd.DataFrame([{
    "system": "direct_only_simple_average_selected_models",
    **compute_binary_metrics(
        GPT_TEST_Y, BASELINE_SIMPLE_AVERAGE_PROBABILITY, 0.50
    ),
}])

atomic_to_csv(
    SELECTED_BASELINE_PREDICTIONS,
    SUBDIRS["analysis"] / "selected_model_direct_baseline_predictions_gpt_test.csv",
)
atomic_to_csv(
    BASELINE_SIMPLE_AVERAGE_METRICS,
    SUBDIRS["analysis"] / "baseline_simple_average_metrics_gpt_test.csv",
)
display(BASELINE_SIMPLE_AVERAGE_METRICS)


,system,n,positive_prevalence,threshold,predicted_positive_count,tn,fp,fn,tp,accuracy,precision,recall,f1,auroc,auprc,brier
0,direct_only_simple_average_selected_models,84,0.5,0.5,13,40,2,31,11,0.607143,0.846154,0.261905,0.4,0.673186,0.710252,0.258074


## Stage 5 — Threshold and weighted ensemble optimization on GPT Test

The final prompt and selected provider models are fixed before post-processing. The notebook searches individual thresholds, the threshold for an equal-probability average, and nonnegative three-model weights summing to one jointly with a threshold. The registered primary search maximizes precision subject to minimum-recall and minimum-predicted-positive safeguards; the unconstrained winner is also archived for transparency.

In [ ]:
# Deterministic threshold and three-model weight-search utilities.

def threshold_search_table(
    y_true: Sequence[int],
    probabilities: Sequence[float],
    *,
    system_name: str,
    extra: Optional[dict[str, Any]] = None,
) -> pd.DataFrame:
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(probabilities, dtype=float)
    rows = []
    for threshold in THRESHOLD_GRID:
        rows.append({
            "system_name": system_name,
            **(extra or {}),
            **compute_binary_metrics(y, p, float(threshold)),
        })
    return pd.DataFrame(rows)

def weight_vectors_3(step: float = WEIGHT_GRID_STEP) -> list[tuple[float, float, float]]:
    units = int(round(1.0 / step))
    if not np.isclose(units * step, 1.0):
        raise ValueError("WEIGHT_GRID_STEP must divide 1.0 exactly.")
    vectors = []
    for i in range(units + 1):
        for j in range(units - i + 1):
            k = units - i - j
            vectors.append((i / units, j / units, k / units))
    return vectors

def weighted_threshold_search_table(
    y_true: Sequence[int],
    probability_matrix: np.ndarray,
    run_names: Sequence[str],
    *,
    weight_step: float = WEIGHT_GRID_STEP,
) -> pd.DataFrame:
    y = np.asarray(y_true, dtype=int)
    matrix = np.asarray(probability_matrix, dtype=float)
    if matrix.shape[1] != 3 or len(run_names) != 3:
        raise ValueError("Weighted search requires exactly three selected models.")

    rows: list[dict[str, Any]] = []
    for weights in weight_vectors_3(weight_step):
        p = matrix @ np.asarray(weights, dtype=float)
        constant_auroc = float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else np.nan
        constant_auprc = float(average_precision_score(y, p)) if int(y.sum()) > 0 else np.nan
        constant_brier = float(brier_score_loss(y, p))
        for threshold in THRESHOLD_GRID:
            pred = (p >= threshold).astype(int)
            tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
            rows.append({
                "system_name": "weighted_probability_ensemble",
                "threshold": float(threshold),
                f"weight__{run_names[0]}": weights[0],
                f"weight__{run_names[1]}": weights[1],
                f"weight__{run_names[2]}": weights[2],
                "predicted_positive_count": int(pred.sum()),
                "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
                "accuracy": float(accuracy_score(y, pred)),
                "precision": float(precision_score(y, pred, zero_division=0)),
                "recall": float(recall_score(y, pred, zero_division=0)),
                "f1": float(f1_score(y, pred, zero_division=0)),
                "auroc": constant_auroc,
                "auprc": constant_auprc,
                "brier": constant_brier,
                "n": len(y),
                "positive_prevalence": float(y.mean()),
            })
    return pd.DataFrame(rows)

def rank_search_candidates(
    surface: pd.DataFrame,
    *,
    constrained: bool,
) -> pd.DataFrame:
    candidate = surface.copy()
    candidate["meets_recall_guardrail"] = (
        candidate["recall"] >= MIN_RECALL_FOR_PRIMARY_SEARCH
    )
    candidate["meets_positive_count_guardrail"] = (
        candidate["predicted_positive_count"]
        >= MIN_PREDICTED_POSITIVES_FOR_PRIMARY_SEARCH
    )
    candidate["eligible_primary"] = (
        candidate["meets_recall_guardrail"]
        & candidate["meets_positive_count_guardrail"]
    )
    if constrained:
        eligible = candidate[candidate["eligible_primary"]].copy()
        if not eligible.empty:
            candidate = eligible
        else:
            warnings.warn(
                "No search candidate met both guardrails; falling back to the full surface.",
                stacklevel=2,
            )
    weight_columns = sorted(c for c in candidate.columns if c.startswith("weight__"))
    sort_columns = [
        "precision", "recall", "f1", "auprc", "auroc", "accuracy", "brier",
        "threshold", *weight_columns,
    ]
    ascending = [False, False, False, False, False, False, True, True] + [False] * len(weight_columns)
    return candidate.sort_values(
        sort_columns,
        ascending=ascending,
        kind="mergesort",
    ).reset_index(drop=True)

def best_weighted_candidate(
    y_true: Sequence[int],
    probability_matrix: np.ndarray,
    run_names: Sequence[str],
) -> tuple[dict[str, Any], pd.DataFrame]:
    surface = weighted_threshold_search_table(y_true, probability_matrix, run_names)
    ranked = rank_search_candidates(surface, constrained=True)
    return ranked.iloc[0].to_dict(), surface

def apply_weighted_candidate(
    probability_matrix: np.ndarray,
    run_names: Sequence[str],
    candidate: dict[str, Any],
) -> tuple[np.ndarray, np.ndarray]:
    weights = np.asarray(
        [float(candidate[f"weight__{name}"]) for name in run_names],
        dtype=float,
    )
    p = np.asarray(probability_matrix, dtype=float) @ weights
    pred = (p >= float(candidate["threshold"])).astype(int)
    return p, pred

print("Optimization utilities loaded.")


Optimization utilities loaded.


In [ ]:
# Full GPT Test threshold and weight searches; lock the deployment configuration.
SELECTED_FINAL_PROB_MATRIX = FINAL_SELECTED_MATRIX[SELECTED_RUN_NAMES].to_numpy(dtype=float)
GPT_TEST_Y = FINAL_SELECTED_MATRIX["gold"].to_numpy(dtype=int)

individual_surfaces = []
individual_best_rows = []
individual_unconstrained_rows = []
for j, run_name in enumerate(SELECTED_RUN_NAMES):
    surface = threshold_search_table(
        GPT_TEST_Y,
        SELECTED_FINAL_PROB_MATRIX[:, j],
        system_name=run_name,
        extra={"search_type": "individual_model_threshold"},
    )
    individual_surfaces.append(surface)
    individual_best_rows.append(
        rank_search_candidates(surface, constrained=True).iloc[0].to_dict()
    )
    individual_unconstrained_rows.append(
        rank_search_candidates(surface, constrained=False).iloc[0].to_dict()
    )
INDIVIDUAL_THRESHOLD_SURFACE = pd.concat(individual_surfaces, ignore_index=True)
INDIVIDUAL_BEST_THRESHOLDS = pd.DataFrame(individual_best_rows)
INDIVIDUAL_UNCONSTRAINED_BEST_THRESHOLDS = pd.DataFrame(individual_unconstrained_rows)

EQUAL_WEIGHT_PROBABILITY = SELECTED_FINAL_PROB_MATRIX.mean(axis=1)
EQUAL_WEIGHT_THRESHOLD_SURFACE = threshold_search_table(
    GPT_TEST_Y,
    EQUAL_WEIGHT_PROBABILITY,
    system_name="equal_weight_final_prompt_ensemble",
    extra={"search_type": "equal_weight_threshold"},
)
EQUAL_WEIGHT_BEST = rank_search_candidates(
    EQUAL_WEIGHT_THRESHOLD_SURFACE, constrained=True
).iloc[0].to_dict()
EQUAL_WEIGHT_UNCONSTRAINED_BEST = rank_search_candidates(
    EQUAL_WEIGHT_THRESHOLD_SURFACE, constrained=False
).iloc[0].to_dict()

WEIGHTED_THRESHOLD_SURFACE = weighted_threshold_search_table(
    GPT_TEST_Y,
    SELECTED_FINAL_PROB_MATRIX,
    SELECTED_RUN_NAMES,
)
WEIGHTED_BEST = rank_search_candidates(
    WEIGHTED_THRESHOLD_SURFACE, constrained=True
).iloc[0].to_dict()
WEIGHTED_UNCONSTRAINED_BEST = rank_search_candidates(
    WEIGHTED_THRESHOLD_SURFACE, constrained=False
).iloc[0].to_dict()

LOCKED_ENSEMBLE_PROBABILITY, LOCKED_ENSEMBLE_PREDICTION = apply_weighted_candidate(
    SELECTED_FINAL_PROB_MATRIX,
    SELECTED_RUN_NAMES,
    WEIGHTED_BEST,
)
LOCKED_ENSEMBLE_METRICS = pd.DataFrame([{
    "system": "locked_full_gpt_test_weighted_ensemble_apparent",
    **compute_binary_metrics(
        GPT_TEST_Y,
        LOCKED_ENSEMBLE_PROBABILITY,
        float(WEIGHTED_BEST["threshold"]),
    ),
}])

for name, frame in {
    "individual_threshold_surface.csv": INDIVIDUAL_THRESHOLD_SURFACE,
    "individual_best_thresholds.csv": INDIVIDUAL_BEST_THRESHOLDS,
    "individual_unconstrained_best_thresholds.csv": INDIVIDUAL_UNCONSTRAINED_BEST_THRESHOLDS,
    "equal_weight_threshold_surface.csv": EQUAL_WEIGHT_THRESHOLD_SURFACE,
    "weighted_threshold_surface.csv": WEIGHTED_THRESHOLD_SURFACE,
    "locked_ensemble_metrics_apparent.csv": LOCKED_ENSEMBLE_METRICS,
}.items():
    atomic_to_csv(frame, SUBDIRS["optimization"] / name)
atomic_to_parquet(
    WEIGHTED_THRESHOLD_SURFACE,
    SUBDIRS["optimization"] / "weighted_threshold_surface.parquet",
)

DEPLOYMENT_CHECKPOINT = {
    "stage": "locked_deployment_configuration",
    "tuning_dataset": "GPT Test",
    "prompt_name": FINAL_PROMPT_NAME,
    "prompt_hash": FINAL_PROMPT_SPEC.prompt_hash,
    "selected_run_names": SELECTED_RUN_NAMES,
    "selected_models": MODEL_SELECTION_CHECKPOINT["selected_models"],
    "selection_objective": (
        "Maximize precision subject to minimum recall and minimum predicted-positive "
        "count; tie-break recall, F1, AUPRC, AUROC, accuracy, Brier, threshold, weights."
    ),
    "minimum_recall": MIN_RECALL_FOR_PRIMARY_SEARCH,
    "minimum_predicted_positives": MIN_PREDICTED_POSITIVES_FOR_PRIMARY_SEARCH,
    "weight_grid_step": WEIGHT_GRID_STEP,
    "threshold_grid": THRESHOLD_GRID.tolist(),
    "locked_threshold": float(WEIGHTED_BEST["threshold"]),
    "locked_weights": {
        run_name: float(WEIGHTED_BEST[f"weight__{run_name}"])
        for run_name in SELECTED_RUN_NAMES
    },
    "locked_apparent_metrics": {
        metric: WEIGHTED_BEST.get(metric)
        for metric in [
            "accuracy", "precision", "recall", "f1", "auroc", "auprc", "brier",
            "tn", "fp", "fn", "tp", "predicted_positive_count",
        ]
    },
    "equal_weight_best_threshold": float(EQUAL_WEIGHT_BEST["threshold"]),
    "equal_weight_best_metrics": {
        metric: EQUAL_WEIGHT_BEST.get(metric)
        for metric in ["accuracy", "precision", "recall", "f1", "auroc", "auprc", "brier"]
    },
    "unconstrained_weighted_winner": {
        key: value for key, value in WEIGHTED_UNCONSTRAINED_BEST.items()
        if key in [
            "threshold", "precision", "recall", "f1", "auroc", "auprc", "accuracy",
            "brier", "predicted_positive_count", *[f"weight__{n}" for n in SELECTED_RUN_NAMES],
        ]
    },
    "source_workbook_sha256": SOURCE_WORKBOOK_SHA256_AT_LOAD,
    "codebook_whole_sheet_sha256": CODEBOOK_WHOLE_SHEET_SHA256,
    "created_at_utc": utc_now(),
}
atomic_write_json(
    SUBDIRS["checkpoints"] / "deployment_configuration.json",
    DEPLOYMENT_CHECKPOINT,
)

print("LOCKED WEIGHTS:", DEPLOYMENT_CHECKPOINT["locked_weights"])
print("LOCKED THRESHOLD:", DEPLOYMENT_CHECKPOINT["locked_threshold"])
display(INDIVIDUAL_BEST_THRESHOLDS)
display(pd.DataFrame([EQUAL_WEIGHT_BEST]))
display(pd.DataFrame([WEIGHTED_BEST]))


LOCKED WEIGHTS: {'openai_gpt_5_6_terra_low': 0.35, 'anthropic_claude_haiku_4_5_no_thinking': 0.0, 'google_gemini_3_1_flash_lite_low': 0.65}
LOCKED THRESHOLD: 0.24


,system_name,search_type,n,positive_prevalence,threshold,predicted_positive_count,tn,fp,fn,tp,accuracy,precision,recall,f1,auroc,auprc,brier,meets_recall_guardrail,meets_positive_count_guardrail,eligible_primary
0,openai_gpt_5_6_terra_low,individual_model_threshold,84,0.5,0.13,30,34,8,20,22,0.666667,0.733333,0.523810,0.611111,0.754252,0.760710,0.308287,True,True,True
1,anthropic_claude_haiku_4_5_no_thinking,individual_model_threshold,84,0.5,0.26,33,32,10,19,23,0.654762,0.696970,0.547619,0.613333,0.691043,0.660127,0.257737,True,True,True
2,google_gemini_3_1_flash_lite_low,individual_model_threshold,84,0.5,0.16,41,30,12,13,29,0.702381,0.707317,0.690476,0.698795,0.732993,0.745550,0.273899,True,True,True


,system_name,search_type,n,positive_prevalence,threshold,predicted_positive_count,tn,fp,fn,tp,accuracy,precision,recall,f1,auroc,auprc,brier,meets_recall_guardrail,meets_positive_count_guardrail,eligible_primary
0,equal_weight_final_prompt_ensemble,equal_weight_threshold,84,0.5,0.32,26,37,5,21,21,0.690476,0.807692,0.5,0.617647,0.754819,0.784154,0.258881,True,True,True


,system_name,threshold,weight__openai_gpt_5_6_terra_low,weight__anthropic_claude_haiku_4_5_no_thinking,weight__google_gemini_3_1_flash_lite_low,predicted_positive_count,tn,fp,fn,tp,accuracy,precision,recall,f1,auroc,auprc,brier,n,positive_prevalence,meets_recall_guardrail,meets_positive_count_guardrail,eligible_primary
0,weighted_probability_ensemble,0.24,0.35,0.0,0.65,23,40,2,21,21,0.72619,0.913043,0.5,0.646154,0.758503,0.803991,0.277551,84,0.5,True,True,True


## Stage 6 — Out-of-fold performance estimation and significance testing

Five-fold out-of-fold tuning provides the primary optimized-versus-baseline performance estimate. Repeated folds and leave-one-document-out analysis assess stability. Paired stratified bootstrap confidence intervals, approximate-randomization p-values, exact McNemar tests, Holm correction, and a document-cluster bootstrap sensitivity are saved. These methods reduce threshold/weight tuning optimism, but prompt and model selection are not nested and remain a stated limitation.

In [ ]:
# ============================================================
# PRIMARY 5-FOLD OOF ENSEMBLE VALIDATION
# Fast version: defer repeated CV and heavy resampling inference
# ============================================================

def compute_metrics_with_binary(
    y_true,
    probabilities,
    binary_predictions,
):
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(probabilities, dtype=float)
    pred = np.asarray(binary_predictions, dtype=int)

    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()

    return {
        "n": len(y),
        "predicted_positive_count": int(pred.sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": float(accuracy_score(y, pred)),
        "precision": float(
            precision_score(y, pred, zero_division=0)
        ),
        "recall": float(
            recall_score(y, pred, zero_division=0)
        ),
        "f1": float(
            f1_score(y, pred, zero_division=0)
        ),
        "auroc": (
            float(roc_auc_score(y, p))
            if len(np.unique(y)) == 2
            else np.nan
        ),
        "auprc": (
            float(average_precision_score(y, p))
            if int(y.sum()) > 0
            else np.nan
        ),
        "brier": float(brier_score_loss(y, p)),
    }


def tune_and_predict_folds(
    y_true,
    probability_matrix,
    passage_ids,
    documents,
    *,
    n_splits,
    random_state,
    repeat_index=0,
):
    splitter = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    prediction_rows = []
    fold_rows = []

    for fold_index, (train_idx, test_idx) in enumerate(
        splitter.split(probability_matrix, y_true),
        start=1,
    ):
        print(
            f"Optimizing fold {fold_index}/{n_splits} "
            f"(train={len(train_idx)}, test={len(test_idx)})"
        )

        best, _ = best_weighted_candidate(
            y_true[train_idx],
            probability_matrix[train_idx],
            SELECTED_RUN_NAMES,
        )

        test_prob, test_pred = apply_weighted_candidate(
            probability_matrix[test_idx],
            SELECTED_RUN_NAMES,
            best,
        )

        fold_rows.append({
            "repeat": repeat_index,
            "fold": fold_index,
            "train_n": len(train_idx),
            "test_n": len(test_idx),
            "threshold": float(best["threshold"]),
            "train_precision": float(best["precision"]),
            "train_recall": float(best["recall"]),
            "train_f1": float(best["f1"]),
            **{
                f"weight__{name}":
                    float(best[f"weight__{name}"])
                for name in SELECTED_RUN_NAMES
            },
        })

        for local_pos, row_index in enumerate(test_idx):
            prediction_rows.append({
                "repeat": repeat_index,
                "fold": fold_index,
                "row_index": int(row_index),
                "passage_id": str(
                    passage_ids[row_index]
                ),
                "document": str(
                    documents[row_index]
                ),
                "gold": int(y_true[row_index]),
                "oof_probability":
                    float(test_prob[local_pos]),
                "oof_prediction":
                    int(test_pred[local_pos]),
                "fold_threshold":
                    float(best["threshold"]),
                **{
                    f"fold_weight__{name}":
                        float(best[f"weight__{name}"])
                    for name in SELECTED_RUN_NAMES
                },
            })

    predictions = (
        pd.DataFrame(prediction_rows)
        .sort_values("row_index")
        .reset_index(drop=True)
    )

    folds = pd.DataFrame(fold_rows)

    if (
        len(predictions) != len(y_true)
        or predictions["row_index"].duplicated().any()
    ):
        raise AssertionError(
            "OOF construction did not produce "
            "one prediction per row."
        )

    return predictions, folds


# ------------------------------------------------------------
# Primary 5-fold OOF tuning
# ------------------------------------------------------------

PASSAGE_IDS = FINAL_SELECTED_MATRIX.index.to_numpy(
    dtype=str
)

DOCUMENTS = FINAL_SELECTED_MATRIX[
    "document"
].to_numpy(dtype=str)

PRIMARY_OOF_PREDICTIONS, PRIMARY_OOF_FOLDS = (
    tune_and_predict_folds(
        GPT_TEST_Y,
        SELECTED_FINAL_PROB_MATRIX,
        PASSAGE_IDS,
        DOCUMENTS,
        n_splits=CV_FOLDS,
        random_state=ANALYSIS_RANDOM_SEED,
        repeat_index=0,
    )
)

PRIMARY_OOF_PROBABILITY = (
    PRIMARY_OOF_PREDICTIONS[
        "oof_probability"
    ].to_numpy(dtype=float)
)

PRIMARY_OOF_BINARY = (
    PRIMARY_OOF_PREDICTIONS[
        "oof_prediction"
    ].to_numpy(dtype=int)
)

PRIMARY_OOF_METRICS = pd.DataFrame([{
    "system":
        "oof_optimized_final_prompt_weighted_ensemble",
    **compute_metrics_with_binary(
        GPT_TEST_Y,
        PRIMARY_OOF_PROBABILITY,
        PRIMARY_OOF_BINARY,
    ),
}])


# ------------------------------------------------------------
# Baseline metrics on the same GPT Test rows
# ------------------------------------------------------------

BASELINE_METRICS_FOR_OOF_COMPARISON = pd.DataFrame([{
    "system":
        "direct_only_simple_average_selected_models",
    **compute_metrics_with_binary(
        GPT_TEST_Y,
        BASELINE_SIMPLE_AVERAGE_PROBABILITY,
        BASELINE_SIMPLE_AVERAGE_PREDICTION,
    ),
}])


# ------------------------------------------------------------
# Simple comparison table
# ------------------------------------------------------------

PRIMARY_OOF_COMPARISON = pd.concat(
    [
        BASELINE_METRICS_FOR_OOF_COMPARISON,
        PRIMARY_OOF_METRICS,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# Fast exact McNemar test
# ------------------------------------------------------------

PRIMARY_SYSTEM_MCNEMAR = pd.DataFrame([{
    "system_a":
        "direct_only_simple_average_selected_models",
    "system_b":
        "oof_optimized_final_prompt_weighted_ensemble",
    **mcnemar_exact(
        GPT_TEST_Y,
        BASELINE_SIMPLE_AVERAGE_PREDICTION.astype(
            float
        ),
        PRIMARY_OOF_BINARY.astype(float),
        threshold_a=0.50,
        threshold_b=0.50,
    ),
}])


# ------------------------------------------------------------
# Save immediately
# ------------------------------------------------------------

atomic_to_csv(
    PRIMARY_OOF_PREDICTIONS,
    SUBDIRS["optimization"]
    / "primary_oof_predictions.csv",
)

atomic_to_csv(
    PRIMARY_OOF_FOLDS,
    SUBDIRS["optimization"]
    / "primary_oof_fold_selections.csv",
)

atomic_to_csv(
    PRIMARY_OOF_METRICS,
    SUBDIRS["optimization"]
    / "primary_oof_metrics.csv",
)

atomic_to_csv(
    PRIMARY_OOF_COMPARISON,
    SUBDIRS["optimization"]
    / "primary_oof_vs_baseline_metrics.csv",
)

atomic_to_csv(
    PRIMARY_SYSTEM_MCNEMAR,
    SUBDIRS["inference"]
    / "primary_system_mcnemar.csv",
)

print("=" * 70)
print("PRIMARY OOF ANALYSIS COMPLETE")
print("=" * 70)

print("\nBaseline vs OOF optimized ensemble:")
display(PRIMARY_OOF_COMPARISON)

print("\nFold-specific selected thresholds and weights:")
display(PRIMARY_OOF_FOLDS)

print("\nExact McNemar test:")
display(PRIMARY_SYSTEM_MCNEMAR)

Optimizing fold 1/5 (train=67, test=17)
Optimizing fold 2/5 (train=67, test=17)
Optimizing fold 3/5 (train=67, test=17)
Optimizing fold 4/5 (train=67, test=17)
Optimizing fold 5/5 (train=68, test=16)
PRIMARY OOF ANALYSIS COMPLETE

Baseline vs OOF optimized ensemble:


,system,n,predicted_positive_count,tn,fp,fn,tp,accuracy,precision,recall,f1,auroc,auprc,brier
0,direct_only_simple_average_selected_models,84,13,40,2,31,11,0.607143,0.846154,0.261905,0.400000,0.673186,0.710252,0.258074
1,oof_optimized_final_prompt_weighted_ensemble,84,23,37,5,24,18,0.654762,0.782609,0.428571,0.553846,0.749433,0.793635,0.278903



Fold-specific selected thresholds and weights:


,repeat,fold,train_n,test_n,threshold,train_precision,train_recall,train_f1,weight__openai_gpt_5_6_terra_low,weight__anthropic_claude_haiku_4_5_no_thinking,weight__google_gemini_3_1_flash_lite_low
0,0,1,67,17,0.24,0.772727,0.515152,0.618182,0.40,0.15,0.45
1,0,2,67,17,0.19,0.809524,0.515152,0.629630,0.90,0.00,0.10
2,0,3,67,17,0.28,0.894737,0.500000,0.641509,0.50,0.20,0.30
3,0,4,67,17,0.25,1.000000,0.529412,0.692308,0.35,0.10,0.55
4,0,5,68,16,0.27,0.944444,0.500000,0.653846,0.55,0.10,0.35



Exact McNemar test:


,system_a,system_b,a_correct_b_wrong,a_wrong_b_correct,discordant_pairs,exact_two_sided_p_value
0,direct_only_simple_average_selected_models,oof_optimized_final_prompt_weighted_ensemble,4,8,12,0.387695


In [ ]:
# Enrich the GPT Test manual-review workbook with the locked ensemble, the
# out-of-fold optimized estimate, and the direct-only simple-average baseline.
# The three selected provider-model predictions remain the main review columns.

FINAL_MANUAL_REVIEW_DF = MANUAL_REVIEW_DF.copy()
for column in ["Manual_Review_Order", "Review_Priority_Group"]:
    if column in FINAL_MANUAL_REVIEW_DF.columns:
        FINAL_MANUAL_REVIEW_DF = FINAL_MANUAL_REVIEW_DF.drop(columns=column)

locked_map = pd.DataFrame({
    "Passage_ID": FINAL_SELECTED_MATRIX.index.astype(str),
    "Locked_Weighted_Ensemble_Probability": LOCKED_ENSEMBLE_PROBABILITY,
    "Locked_Weighted_Ensemble_Prediction": LOCKED_ENSEMBLE_PREDICTION,
})
locked_map["Locked_Weighted_Ensemble_Correct"] = (
    locked_map["Locked_Weighted_Ensemble_Prediction"].to_numpy(dtype=int)
    == GPT_TEST_Y
)

baseline_map = pd.DataFrame({
    "Passage_ID": BASELINE_SELECTED_MATRIX.index.astype(str),
    "Direct_Baseline_Simple_Average_Probability": BASELINE_SIMPLE_AVERAGE_PROBABILITY,
    "Direct_Baseline_Simple_Average_Prediction": BASELINE_SIMPLE_AVERAGE_PREDICTION,
})
baseline_map["Direct_Baseline_Simple_Average_Correct"] = (
    baseline_map["Direct_Baseline_Simple_Average_Prediction"].to_numpy(dtype=int)
    == GPT_TEST_Y
)

oof_map = PRIMARY_OOF_PREDICTIONS[[
    "passage_id", "oof_probability", "oof_prediction", "fold", "fold_threshold",
]].rename(columns={
    "passage_id": "Passage_ID",
    "oof_probability": "OOF_Optimized_Ensemble_Probability",
    "oof_prediction": "OOF_Optimized_Ensemble_Prediction",
    "fold": "OOF_Fold",
    "fold_threshold": "OOF_Fold_Threshold",
})
oof_map["OOF_Optimized_Ensemble_Correct"] = (
    oof_map["OOF_Optimized_Ensemble_Prediction"].to_numpy(dtype=int)
    == GPT_TEST_Y
)

for addition in [locked_map, oof_map, baseline_map]:
    FINAL_MANUAL_REVIEW_DF = FINAL_MANUAL_REVIEW_DF.merge(
        addition, on="Passage_ID", how="left", validate="one_to_one"
    )

FINAL_MANUAL_REVIEW_DF["Locked_Ensemble_Wrong"] = ~FINAL_MANUAL_REVIEW_DF[
    "Locked_Weighted_Ensemble_Correct"
].astype(bool)
FINAL_MANUAL_REVIEW_DF["OOF_Optimized_Ensemble_Wrong"] = ~FINAL_MANUAL_REVIEW_DF[
    "OOF_Optimized_Ensemble_Correct"
].astype(bool)
FINAL_MANUAL_REVIEW_DF["Direct_Baseline_Wrong"] = ~FINAL_MANUAL_REVIEW_DF[
    "Direct_Baseline_Simple_Average_Correct"
].astype(bool)

# Lower numeric value means earlier review. The selected-model disagreement/error
# pattern remains primary; ensemble/baseline errors refine priority within it.
FINAL_MANUAL_REVIEW_DF["Review_Priority_Group"] = np.select(
    [
        FINAL_MANUAL_REVIEW_DF["All_Three_Selected_Models_Wrong"].astype(bool),
        FINAL_MANUAL_REVIEW_DF["Most_Selected_Models_Wrong"].astype(bool)
        & FINAL_MANUAL_REVIEW_DF["Locked_Ensemble_Wrong"].astype(bool),
        FINAL_MANUAL_REVIEW_DF["Most_Selected_Models_Wrong"].astype(bool),
        FINAL_MANUAL_REVIEW_DF["OOF_Optimized_Ensemble_Wrong"].astype(bool),
        FINAL_MANUAL_REVIEW_DF["Selected_Models_Disagree"].astype(bool),
        FINAL_MANUAL_REVIEW_DF["Extraction_Review_Flag"].fillna(False).astype(bool),
    ],
    [1, 2, 3, 4, 5, 6],
    default=7,
)
FINAL_MANUAL_REVIEW_DF["Reviewer_Final_Label"] = ""
FINAL_MANUAL_REVIEW_DF["Reviewer_Error_Category"] = ""
FINAL_MANUAL_REVIEW_DF["Reviewer_Notes"] = ""

FINAL_MANUAL_REVIEW_DF = FINAL_MANUAL_REVIEW_DF.sort_values(
    [
        "Review_Priority_Group",
        "Selected_Model_Wrong_Count",
        "Locked_Ensemble_Wrong",
        "OOF_Optimized_Ensemble_Wrong",
        "Mean_Probability_Distance_From_0_5",
        "Document",
        "Passage_ID",
    ],
    ascending=[True, False, False, False, True, True, True],
    kind="mergesort",
).reset_index(drop=True)
FINAL_MANUAL_REVIEW_DF.insert(
    0, "Manual_Review_Order", np.arange(1, len(FINAL_MANUAL_REVIEW_DF) + 1)
)
FINAL_MOST_MODELS_WRONG_DF = FINAL_MANUAL_REVIEW_DF[
    FINAL_MANUAL_REVIEW_DF["Most_Selected_Models_Wrong"].astype(bool)
].copy()

final_review_readme = pd.DataFrame({
    "Item": [
        "Purpose",
        "Provider predictions",
        "Human labeler columns",
        "Most Models Wrong sheet",
        "Locked weighted ensemble",
        "OOF optimized ensemble",
        "Direct baseline",
        "Inference limitation",
    ],
    "Description": [
        "Manual error analysis of every GPT Test paragraph after selecting one model per provider.",
        "Only the selected OpenAI, Anthropic, and Google model under the final prompt are shown.",
        "Anmol_Label and Prerana_Label retain Yes, No, or Not labeled. Missing human decisions are never converted to No.",
        "Contains rows where at least two of the three selected provider models are wrong at threshold 0.50.",
        "Full-GPT-Test weighted ensemble using the deployment weights and threshold; its apparent score is optimistically biased because the same set was used for tuning.",
        "Five-fold out-of-fold threshold/weight tuning used for the primary optimized-versus-baseline performance estimate.",
        "Direct-only prompt, equal probability average of the three selected provider models, threshold 0.50.",
        "Prompt and model selection were not nested inside cross-validation; prompt/model comparison p-values are exploratory. The notebook labels each inferential table accordingly.",
    ],
})

FINAL_MODEL_REVIEW_XLSX = (
    SUBDIRS["exports"] / "GPT_Test_Selected_Models_and_Ensembles_Manual_Review.xlsx"
)
with pd.ExcelWriter(FINAL_MODEL_REVIEW_XLSX, engine="xlsxwriter") as writer:
    sheets = {
    "README": final_review_readme,
    "Manual Review": FINAL_MANUAL_REVIEW_DF,
    "Most Models Wrong": FINAL_MOST_MODELS_WRONG_DF,
    "Selected Predictions Long": SELECTED_GPT_TEST_PREDICTIONS,
    "Selected Models": SELECTED_MODEL_TABLE,
    "Candidate Metrics": FINAL_PROMPT_MODEL_METRICS,
    "Metric CIs": FINAL_PROMPT_MODEL_CIS,
    "Model Comparisons": MODEL_SELECTION_INFERENCE,
    "Model McNemar": MODEL_SELECTION_MCNEMAR,
    "Examples A-B": STAGE1_SELECTION_SUMMARY,
    "Checklist A-B": STAGE2_SELECTION_SUMMARY,
    "Threshold Best": INDIVIDUAL_BEST_THRESHOLDS,
    "Equal Weight Best": pd.DataFrame([EQUAL_WEIGHT_BEST]),
    "Weighted Best": pd.DataFrame([WEIGHTED_BEST]),
    "Primary OOF Metrics": PRIMARY_OOF_METRICS,
    "OOF vs Baseline Metrics": PRIMARY_OOF_COMPARISON,
    "Primary McNemar": PRIMARY_SYSTEM_MCNEMAR,
}
    for sheet_name, frame in sheets.items():
        frame.to_excel(writer, index=False, sheet_name=sheet_name)

    workbook = writer.book
    header_format = workbook.add_format({
        "bold": True,
        "text_wrap": True,
        "valign": "top",
        "border": 1,
        "bg_color": "#D9EAF7",
    })
    wrap_format = workbook.add_format({"text_wrap": True, "valign": "top"})
    numeric_format = workbook.add_format({"num_format": "0.000", "valign": "top"})
    error_format = workbook.add_format({"bg_color": "#FCE4D6"})
    for sheet_name, frame in sheets.items():
        ws = writer.sheets[sheet_name]
        ws.freeze_panes(1, 0)
        if len(frame.columns):
            ws.autofilter(0, 0, max(len(frame), 1), len(frame.columns) - 1)
        for col_idx, column in enumerate(frame.columns):
            ws.write(0, col_idx, column, header_format)
            series = frame[column].astype(str) if len(frame) else pd.Series(dtype=str)
            max_len = max([len(str(column)), *(series.head(500).map(len).tolist())] or [10])
            width = min(max(max_len + 2, 10), 58)
            fmt = numeric_format if any(
                token in column.lower()
                for token in [
                    "probability", "precision", "recall", "accuracy", "f1",
                    "auroc", "auprc", "brier", "ci_", "threshold", "weight",
                ]
            ) else wrap_format
            ws.set_column(col_idx, col_idx, width, fmt)
        ws.set_default_row(30)

    review_ws = writer.sheets["Manual Review"]
    if len(FINAL_MANUAL_REVIEW_DF):
        priority_col = FINAL_MANUAL_REVIEW_DF.columns.get_loc("Review_Priority_Group")
        review_ws.conditional_format(
            1, priority_col, len(FINAL_MANUAL_REVIEW_DF), priority_col,
            {"type": "cell", "criteria": "<=", "value": 3, "format": error_format},
        )

atomic_to_csv(
    FINAL_MANUAL_REVIEW_DF,
    SUBDIRS["exports"] / "gpt_test_selected_models_manual_review.csv",
)
print("Final GPT Test manual-review workbook:", FINAL_MODEL_REVIEW_XLSX)
print("Rows where at least two selected models are wrong:", len(FINAL_MOST_MODELS_WRONG_DF))


Final GPT Test manual-review workbook: /content/drive/MyDrive/Unlearning_Paragraph_Level_LLM_Experiment/paragraph_level_gpt_test_first_v2_20260821/exports/GPT_Test_Selected_Models_and_Ensembles_Manual_Review.xlsx
Rows where at least two selected models are wrong: 33


## Stage 7 — One-time final GPT Test ALL deployment

This is the **only** stage that makes GPT Test ALL API calls. A hard gate verifies all earlier checkpoints, the immutable source/codebook hashes, the final prompt hash, the selected model list, and the locked ensemble parameters. Only one selected model from each provider receives the final prompt. Exact GPT Test paragraphs are reused from cache. No direct baseline, alternate prompt, runner-up model, threshold search, or weight search is run on GPT Test ALL.

In [ ]:
# Hard gate and preflight for the single final GPT Test ALL run.
# No GPT Test ALL model call is permitted until prompt selection, model selection,
# threshold/weight locking, baseline analysis, and the GPT Test review export exist.

required_checkpoint_paths = {
    "stage1": SUBDIRS["checkpoints"] / "stage1_selection.json",
    "stage2": SUBDIRS["checkpoints"] / "stage2_selection.json",
    "selected_models": SUBDIRS["checkpoints"] / "selected_models.json",
    "deployment": SUBDIRS["checkpoints"] / "deployment_configuration.json",
}
missing_checkpoints = [
    name for name, path in required_checkpoint_paths.items() if not path.exists()
]
if missing_checkpoints:
    raise RuntimeError(
        "GPT Test ALL is locked until all GPT Test stages are complete. Missing: "
        + ", ".join(missing_checkpoints)
    )
if not FINAL_MODEL_REVIEW_XLSX.exists():
    raise RuntimeError("Create the final GPT Test manual-review workbook before GPT Test ALL.")

FINAL_ALL_CHECKPOINTS = {
    name: json.loads(path.read_text(encoding="utf-8"))
    for name, path in required_checkpoint_paths.items()
}
for name, checkpoint in FINAL_ALL_CHECKPOINTS.items():
    if checkpoint.get("source_workbook_sha256") != SOURCE_WORKBOOK_SHA256_AT_LOAD:
        raise RuntimeError(f"{name} checkpoint belongs to a different source workbook.")
    if checkpoint.get("codebook_whole_sheet_sha256") != CODEBOOK_WHOLE_SHEET_SHA256:
        raise RuntimeError(f"{name} checkpoint belongs to a different codebook sheet.")

if FINAL_ALL_CHECKPOINTS["stage2"]["winner_prompt_name"] != FINAL_PROMPT_NAME:
    raise RuntimeError("In-memory final prompt differs from the locked Stage 2 checkpoint.")
if FINAL_ALL_CHECKPOINTS["stage2"]["winner_prompt_hash"] != FINAL_PROMPT_SPEC.prompt_hash:
    raise RuntimeError("Final prompt hash differs from the locked Stage 2 checkpoint.")
if FINAL_ALL_CHECKPOINTS["selected_models"]["selected_run_names"] != SELECTED_RUN_NAMES:
    raise RuntimeError("Selected run names differ from the selected-model checkpoint.")
if FINAL_ALL_CHECKPOINTS["deployment"]["selected_run_names"] != SELECTED_RUN_NAMES:
    raise RuntimeError("Selected run names differ from the deployment checkpoint.")
if FINAL_ALL_CHECKPOINTS["deployment"]["prompt_hash"] != FINAL_PROMPT_SPEC.prompt_hash:
    raise RuntimeError("Deployment prompt hash differs from the final prompt.")

FINAL_ALL_PROMPT_SPEC = FINAL_PROMPT_SPEC
FINAL_ALL_MODEL_CONFIGS = SELECTED_MODEL_CONFIGS
if len(FINAL_ALL_MODEL_CONFIGS) != 3:
    raise AssertionError("Final GPT Test ALL run must contain exactly three model configs.")
if {c["provider"] for c in FINAL_ALL_MODEL_CONFIGS} != set(EXPECTED_PROVIDERS):
    raise AssertionError("Final GPT Test ALL run must contain one model per provider.")

# Detect and block any accidental earlier GPT Test ALL calls under this experiment root
# that used an unselected model or a non-final prompt.
reload_caches()
existing_all_success = pd.DataFrame([
    record
    for records in CACHE_ALL_RECORDS.values()
    for record in records
    if record.get("status") == "success"
    and record.get("source_dataset") == "GPT Test ALL"
])
if not existing_all_success.empty:
    unauthorized = existing_all_success[
        ~existing_all_success["run_name"].isin(SELECTED_RUN_NAMES)
        | existing_all_success["prompt_name"].ne(FINAL_PROMPT_NAME)
    ]
    if not unauthorized.empty:
        unauthorized_path = SUBDIRS["audits"] / "unauthorized_gpt_test_all_successes.csv"
        atomic_to_csv(unauthorized, unauthorized_path)
        raise RuntimeError(
            "This experiment root contains GPT Test ALL predictions from unselected "
            f"models or prompts. Use a fresh EXPERIMENT_NAME. See {unauthorized_path}."
        )

FINAL_ALL_PREFLIGHT = build_job_rows(
    "GPT Test ALL", [FINAL_ALL_PROMPT_SPEC], FINAL_ALL_MODEL_CONFIGS
)
FINAL_ALL_PREFLIGHT_SUMMARY = (
    FINAL_ALL_PREFLIGHT.groupby(
        ["provider", "run_name", "model", "prompt_name"], as_index=False
    )
    .agg(
        logical_rows=("run_key", "size"),
        cached_rows=("cached_success", "sum"),
        rough_total_cost_usd=("estimated_cost_usd", "sum"),
    )
)
FINAL_ALL_PREFLIGHT_SUMMARY["new_calls"] = (
    FINAL_ALL_PREFLIGHT_SUMMARY["logical_rows"]
    - FINAL_ALL_PREFLIGHT_SUMMARY["cached_rows"]
)
FINAL_ALL_PREFLIGHT_SUMMARY["reused_share"] = (
    FINAL_ALL_PREFLIGHT_SUMMARY["cached_rows"]
    / FINAL_ALL_PREFLIGHT_SUMMARY["logical_rows"]
)

if set(FINAL_ALL_PREFLIGHT_SUMMARY["run_name"]) != set(SELECTED_RUN_NAMES):
    raise AssertionError("Final-all preflight contains an unselected or missing model.")
if FINAL_ALL_PREFLIGHT_SUMMARY["prompt_name"].nunique() != 1:
    raise AssertionError("Final-all preflight contains more than one prompt.")

atomic_to_csv(
    FINAL_ALL_PREFLIGHT,
    SUBDIRS["manifests"] / "final_gpt_test_all_preflight_rows.csv",
)
atomic_to_csv(
    FINAL_ALL_PREFLIGHT_SUMMARY,
    SUBDIRS["manifests"] / "final_gpt_test_all_preflight_summary.csv",
)

print("FINAL GPT TEST ALL PREFLIGHT — selected models and final prompt only")
display(FINAL_ALL_PREFLIGHT_SUMMARY)
print({
    "gpt_test_all_rows": len(GPT_TEST_ALL_DF),
    "logical_predictions": len(FINAL_ALL_PREFLIGHT),
    "cached_from_prior_exact_text_runs": int(FINAL_ALL_PREFLIGHT["cached_success"].sum()),
    "new_paid_calls_if_executed": int((~FINAL_ALL_PREFLIGHT["cached_success"]).sum()),
    "run_switch": RUN_FINAL_GPT_TEST_ALL,
    "paid_gate": ALLOW_PAID_API_CALLS,
})


openai: 336 successful cached run keys; 336 total records.
anthropic: 336 successful cached run keys; 338 total records.
google: 336 successful cached run keys; 364 total records.
FINAL GPT TEST ALL PREFLIGHT — selected models and final prompt only


,provider,run_name,model,prompt_name,logical_rows,cached_rows,rough_total_cost_usd,new_calls,reused_share
0,anthropic,anthropic_claude_haiku_4_5_no_thinking,claude-haiku-4-5-20251001,codebook_no_examples_with_checklist,677,79,2.687715,598,0.116691
1,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,codebook_no_examples_with_checklist,677,79,0.702394,598,0.116691
2,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,codebook_no_examples_with_checklist,677,79,5.619150,598,0.116691


{'gpt_test_all_rows': 677, 'logical_predictions': 2031, 'cached_from_prior_exact_text_runs': 237, 'new_paid_calls_if_executed': 1794, 'run_switch': True, 'paid_gate': True}


In [ ]:
# Final GPT Test ALL — selected OpenAI model only, final prompt only.
FINAL_ALL_OPENAI_JOB = run_prediction_job(
    "GPT Test ALL",
    [FINAL_ALL_PROMPT_SPEC],
    [c for c in FINAL_ALL_MODEL_CONFIGS if c["provider"] == "openai"],
    execute=RUN_FINAL_GPT_TEST_ALL,
)


openai: 336 successful cached run keys; 336 total records.
anthropic: 336 successful cached run keys; 338 total records.
google: 336 successful cached run keys; 364 total records.


,provider,run_name,model,prompt_name,logical_rows,cached_rows,estimated_total_cost_usd,new_calls
0,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,codebook_no_examples_with_checklist,677,79,5.61915,598


{'dataset': 'GPT Test ALL', 'provider': 'openai', 'stage': 'stage2_checklist_ab', 'new_calls': 598, 'rough_new_cost_usd': 4.961, 'execute': True}


openai | openai_gpt_5_6_terra_low:   0%|          | 0/598 [00:00<?, ?it/s]

Job completed without final/permanent model failures.


In [ ]:
# Final GPT Test ALL — selected Anthropic model only, final prompt only.
FINAL_ALL_ANTHROPIC_JOB = run_prediction_job(
    "GPT Test ALL",
    [FINAL_ALL_PROMPT_SPEC],
    [c for c in FINAL_ALL_MODEL_CONFIGS if c["provider"] == "anthropic"],
    execute=RUN_FINAL_GPT_TEST_ALL,
)


openai: 934 successful cached run keys; 934 total records.
anthropic: 336 successful cached run keys; 338 total records.
google: 336 successful cached run keys; 364 total records.


,provider,run_name,model,prompt_name,logical_rows,cached_rows,estimated_total_cost_usd,new_calls
0,anthropic,anthropic_claude_haiku_4_5_no_thinking,claude-haiku-4-5-20251001,codebook_no_examples_with_checklist,677,79,2.687715,598


{'dataset': 'GPT Test ALL', 'provider': 'anthropic', 'stage': 'stage2_checklist_ab', 'new_calls': 598, 'rough_new_cost_usd': 2.3729, 'execute': True}


anthropic | anthropic_claude_haiku_4_5_no_thinking:   0%|          | 0/598 [00:00<?, ?it/s]

Job completed without final/permanent model failures.


In [ ]:
# Final GPT Test ALL — selected Google model only, final prompt only.
FINAL_ALL_GOOGLE_JOB = run_prediction_job(
    "GPT Test ALL",
    [FINAL_ALL_PROMPT_SPEC],
    [c for c in FINAL_ALL_MODEL_CONFIGS if c["provider"] == "google"],
    execute=RUN_FINAL_GPT_TEST_ALL,
)


openai: 934 successful cached run keys; 934 total records.
anthropic: 934 successful cached run keys; 936 total records.
google: 492 successful cached run keys; 591 total records.


,provider,run_name,model,prompt_name,logical_rows,cached_rows,estimated_total_cost_usd,new_calls
0,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,codebook_no_examples_with_checklist,677,235,0.702394,442


{'dataset': 'GPT Test ALL', 'provider': 'google', 'stage': 'stage2_checklist_ab', 'new_calls': 442, 'rough_new_cost_usd': 0.4599, 'execute': True}


google | google_gemini_3_1_flash_lite_low:   0%|          | 0/442 [00:00<?, ?it/s]


google_gemini_3_1_flash_lite_low attempt 1 failed (transient, status=429); retrying in 2.5s: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 500, model: gemini-3.1-flash-lite\nPlease retry in 35.476546795s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerM

In [ ]:
# Materialize and evaluate the one-time final GPT Test ALL run. Parameters are
# applied exactly as locked on GPT Test; nothing is reselected or retuned here.

FINAL_ALL_PREDICTIONS = materialize_prediction_table(
    "GPT Test ALL",
    [FINAL_ALL_PROMPT_SPEC],
    run_names=SELECTED_RUN_NAMES,
    require_complete=True,
)
if set(FINAL_ALL_PREDICTIONS["run_name"]) != set(SELECTED_RUN_NAMES):
    raise AssertionError("Final GPT Test ALL predictions contain the wrong model set.")
if FINAL_ALL_PREDICTIONS["prompt_name"].nunique() != 1 or (
    FINAL_ALL_PREDICTIONS["prompt_name"].iloc[0] != FINAL_PROMPT_NAME
):
    raise AssertionError("Final GPT Test ALL predictions contain the wrong prompt.")


def probability_matrix_for_dataset(
    dataset_name: str,
    predictions: pd.DataFrame,
    run_names: Sequence[str],
) -> pd.DataFrame:
    source = PASSAGE_TABLES[dataset_name]
    metadata_columns = [
        "passage_id", "gold", "gold_label", "document", "text", "text_sha256",
        "source_file", "reference", "human_label_status", "human_labelers",
        "matched_gpt_test_ids", "label_basis", "adjudication_rationale",
        "paragraph_type", "review_flag", "review_issue",
        "example_overlap_risk", "example_max_token_jaccard",
    ]
    metadata = source[metadata_columns].set_index("passage_id")
    pivot = predictions.pivot(
        index="passage_id", columns="run_name", values="unlearning_probability"
    ).reindex(index=metadata.index, columns=list(run_names))
    if pivot.isna().any().any():
        missing = pivot.isna().sum().to_dict()
        raise RuntimeError(f"Missing final GPT Test ALL probabilities: {missing}")
    return metadata.join(pivot)


FINAL_ALL_MATRIX = probability_matrix_for_dataset(
    "GPT Test ALL", FINAL_ALL_PREDICTIONS, SELECTED_RUN_NAMES
)
FINAL_ALL_Y = FINAL_ALL_MATRIX["gold"].to_numpy(dtype=int)
FINAL_ALL_PROB_MATRIX = FINAL_ALL_MATRIX[SELECTED_RUN_NAMES].to_numpy(dtype=float)

locked_weights = FINAL_ALL_CHECKPOINTS["deployment"]["locked_weights"]
locked_threshold = float(FINAL_ALL_CHECKPOINTS["deployment"]["locked_threshold"])
if set(locked_weights) != set(SELECTED_RUN_NAMES):
    raise RuntimeError("Locked weight names do not match selected run names.")
weight_vector = np.asarray(
    [float(locked_weights[name]) for name in SELECTED_RUN_NAMES], dtype=float
)
if not np.isclose(weight_vector.sum(), 1.0, atol=1e-9):
    raise RuntimeError("Locked weights do not sum to one.")

FINAL_ALL_LOCKED_PROBABILITY = FINAL_ALL_PROB_MATRIX @ weight_vector
FINAL_ALL_LOCKED_PREDICTION = (
    FINAL_ALL_LOCKED_PROBABILITY >= locked_threshold
).astype(int)
FINAL_ALL_SIMPLE_AVERAGE_PROBABILITY = FINAL_ALL_PROB_MATRIX.mean(axis=1)
FINAL_ALL_SIMPLE_AVERAGE_PREDICTION = (
    FINAL_ALL_SIMPLE_AVERAGE_PROBABILITY >= 0.50
).astype(int)

GPT_TEST_TEXT_HASHES = set(GPT_TEST_DF["text_sha256"])
FINAL_ALL_MATRIX["is_exact_gpt_test_overlap"] = FINAL_ALL_MATRIX[
    "text_sha256"
].isin(GPT_TEST_TEXT_HASHES)


def final_all_scope_mask(frame: pd.DataFrame, scope: str) -> pd.Series:
    if scope == "all_rows":
        return pd.Series(True, index=frame.index)
    if scope == "human_labeled_only":
        status = frame["human_label_status"].fillna("").str.strip().str.lower()
        return status.ne("") & status.ne("unlabeled")
    if scope == "review_unflagged":
        return ~frame["review_flag"].fillna(False).astype(bool)
    if scope == "example_leakage_safe":
        return ~frame["example_overlap_risk"].fillna(False).astype(bool)
    if scope == "exact_gpt_test_overlap":
        return frame["is_exact_gpt_test_overlap"].astype(bool)
    if scope == "non_gpt_test_exact_text":
        return ~frame["is_exact_gpt_test_overlap"].astype(bool)
    raise ValueError(f"Unknown final-all scope: {scope}")


FINAL_ALL_SCOPES = [
    "all_rows",
    "human_labeled_only",
    "review_unflagged",
    "example_leakage_safe",
    "exact_gpt_test_overlap",
    "non_gpt_test_exact_text",
]

FINAL_ALL_SYSTEMS: dict[str, dict[str, Any]] = {}
for j, run_name in enumerate(SELECTED_RUN_NAMES):
    FINAL_ALL_SYSTEMS[f"selected_model__{run_name}"] = {
        "probability": FINAL_ALL_PROB_MATRIX[:, j],
        "prediction": (FINAL_ALL_PROB_MATRIX[:, j] >= 0.50).astype(int),
        "threshold": 0.50,
        "system_type": "selected_provider_model",
    }
FINAL_ALL_SYSTEMS["final_prompt_selected_models_simple_average"] = {
    "probability": FINAL_ALL_SIMPLE_AVERAGE_PROBABILITY,
    "prediction": FINAL_ALL_SIMPLE_AVERAGE_PREDICTION,
    "threshold": 0.50,
    "system_type": "equal_probability_average",
}
FINAL_ALL_SYSTEMS["locked_weighted_ensemble"] = {
    "probability": FINAL_ALL_LOCKED_PROBABILITY,
    "prediction": FINAL_ALL_LOCKED_PREDICTION,
    "threshold": locked_threshold,
    "system_type": "locked_gpt_test_weights_and_threshold",
}

metric_rows: list[dict[str, Any]] = []
for system_name, system in FINAL_ALL_SYSTEMS.items():
    for scope in FINAL_ALL_SCOPES:
        mask = final_all_scope_mask(FINAL_ALL_MATRIX, scope).to_numpy(dtype=bool)
        if not mask.any():
            continue
        metric_rows.append({
            "dataset": "GPT Test ALL",
            "system": system_name,
            "system_type": system["system_type"],
            "scope": scope,
            **compute_metrics_with_binary(
                FINAL_ALL_Y[mask],
                np.asarray(system["probability"])[mask],
                np.asarray(system["prediction"])[mask],
            ),
            "applied_threshold": float(system["threshold"]),
        })
FINAL_ALL_METRICS = pd.DataFrame(metric_rows)

# Document-level locked-ensemble metrics are descriptive; some documents may not
# contain both classes, in which case AUROC is undefined.
document_rows = []
for document in FINAL_ALL_MATRIX["document"].drop_duplicates():
    mask = FINAL_ALL_MATRIX["document"].eq(document).to_numpy(dtype=bool)
    document_rows.append({
        "document": document,
        **compute_metrics_with_binary(
            FINAL_ALL_Y[mask],
            FINAL_ALL_LOCKED_PROBABILITY[mask],
            FINAL_ALL_LOCKED_PREDICTION[mask],
        ),
        "applied_threshold": locked_threshold,
    })
FINAL_ALL_DOCUMENT_METRICS = pd.DataFrame(document_rows)


def finite_percentile(values: np.ndarray, q: float) -> float:
    finite = values[np.isfinite(values)]
    return float(np.percentile(finite, q)) if len(finite) else np.nan


def final_all_metric_ci_table(
    n_bootstrap: int = N_BOOTSTRAP,
) -> pd.DataFrame:
    alpha = 1.0 - CI_LEVEL
    lower_q, upper_q = 100 * alpha / 2, 100 * (1 - alpha / 2)
    rows: list[dict[str, Any]] = []
    for system_name, system in FINAL_ALL_SYSTEMS.items():
        probabilities = np.asarray(system["probability"], dtype=float)
        predictions = np.asarray(system["prediction"], dtype=int)
        for scope in ["all_rows", "human_labeled_only", "non_gpt_test_exact_text"]:
            mask = final_all_scope_mask(FINAL_ALL_MATRIX, scope).to_numpy(dtype=bool)
            if not mask.any():
                continue
            y = FINAL_ALL_Y[mask]
            p = probabilities[mask]
            pred = predictions[mask]
            for metric in METRIC_NAMES:
                estimate = custom_metric_value(y, p, pred, metric)
                rng = np.random.default_rng(
                    deterministic_seed(
                        ANALYSIS_RANDOM_SEED, "final_all_ci", system_name, scope, metric
                    )
                )
                samples = np.empty(n_bootstrap, dtype=float)
                for i in range(n_bootstrap):
                    idx = stratified_bootstrap_indices(y, rng)
                    samples[i] = custom_metric_value(y[idx], p[idx], pred[idx], metric)
                rows.append({
                    "dataset": "GPT Test ALL",
                    "system": system_name,
                    "scope": scope,
                    "metric": metric,
                    "estimate": estimate,
                    "ci_low": finite_percentile(samples, lower_q),
                    "ci_high": finite_percentile(samples, upper_q),
                    "bootstrap_iterations": n_bootstrap,
                    "ci_method": "stratified passage bootstrap",
                })
    return pd.DataFrame(rows)


FINAL_ALL_METRIC_CIS = final_all_metric_ci_table()

# Secondary comparison only: locked weighted ensemble versus the equal probability
# average under the same final prompt. Direct-only baseline is intentionally not run
# on GPT Test ALL, preserving the user's one-time selected-model-only rule.
final_all_inference_frames = []
final_all_mcnemar_frames = []
for scope in ["all_rows", "human_labeled_only", "non_gpt_test_exact_text"]:
    mask = final_all_scope_mask(FINAL_ALL_MATRIX, scope).to_numpy(dtype=bool)
    if not mask.any():
        continue
    inference, mcnemar = paired_system_inference(
        FINAL_ALL_Y[mask],
        FINAL_ALL_SIMPLE_AVERAGE_PROBABILITY[mask],
        FINAL_ALL_LOCKED_PROBABILITY[mask],
        system_a="final_prompt_selected_models_simple_average",
        system_b="locked_weighted_ensemble",
        threshold_a=0.50,
        threshold_b=locked_threshold,
        inference_status=(
            "secondary_locked_parameter_comparison; GPT Test ALL contains default-No "
            "unlabeled rows and may contain exact GPT Test overlaps"
        ),
    )
    inference.insert(0, "scope", scope)
    mcnemar.insert(0, "scope", scope)
    final_all_inference_frames.append(inference)
    final_all_mcnemar_frames.append(mcnemar)
FINAL_ALL_ENSEMBLE_INFERENCE = pd.concat(
    final_all_inference_frames, ignore_index=True
) if final_all_inference_frames else pd.DataFrame()
FINAL_ALL_ENSEMBLE_MCNEMAR = pd.concat(
    final_all_mcnemar_frames, ignore_index=True
) if final_all_mcnemar_frames else pd.DataFrame()

# Wide row-level review output.
final_all_base_columns = [
    "reference", "text", "document", "source_file",
    "gold_label", "gold", "human_label_status", "human_labelers",
    "matched_gpt_test_ids", "label_basis", "adjudication_rationale",
    "paragraph_type", "review_flag", "review_issue", "example_overlap_risk",
    "example_max_token_jaccard", "text_sha256", "is_exact_gpt_test_overlap",
]
FINAL_ALL_REVIEW_DF = FINAL_ALL_MATRIX[final_all_base_columns].reset_index().copy()
FINAL_ALL_REVIEW_DF = FINAL_ALL_REVIEW_DF.rename(columns={
    "passage_id": "Passage_ID",
    "reference": "Reference",
    "text": "Text_Content",
    "document": "Document",
    "source_file": "Source_File",
    "gold_label": "Gold_Unlearning",
    "gold": "Gold_Binary",
    "human_label_status": "Human_Label_Status",
    "human_labelers": "Human_Labelers",
    "matched_gpt_test_ids": "Matched_GPT_Test_IDs",
    "label_basis": "Label_Basis",
    "adjudication_rationale": "Adjudication_Rationale",
    "paragraph_type": "Paragraph_Type",
    "review_flag": "Extraction_Review_Flag",
    "review_issue": "Extraction_Review_Issue",
    "example_overlap_risk": "Codebook_Example_Overlap_Risk",
    "example_max_token_jaccard": "Codebook_Example_Max_Jaccard",
    "text_sha256": "Text_SHA256",
    "is_exact_gpt_test_overlap": "Exact_GPT_Test_Text_Overlap",
})

for provider in EXPECTED_PROVIDERS:
    provider_frame = FINAL_ALL_PREDICTIONS[
        FINAL_ALL_PREDICTIONS["provider"].eq(provider)
    ].copy()
    if provider_frame["run_name"].nunique() != 1:
        raise AssertionError(f"Expected one selected {provider} model in final all.")
    provider_frame["correct_0_5"] = (
        provider_frame["predicted_label_at_0_5"].astype(int)
        == provider_frame["gold"].astype(int)
    )
    provider_frame = provider_frame[[
        "passage_id", "run_name", "model_requested", "model_returned",
        "unlearning_probability", "unlearning_label", "predicted_label_at_0_5",
        "correct_0_5", "target_category", "evidence_quote",
        "evidence_quote_valid", "rationale", "label_probability_inconsistent",
        "target_label_inconsistent", "cache_source_dataset", "cache_source_passage_id",
    ]].rename(columns={
        "passage_id": "Passage_ID",
        "run_name": f"{provider}_run_name",
        "model_requested": f"{provider}_model_requested",
        "model_returned": f"{provider}_model_returned",
        "unlearning_probability": f"{provider}_probability",
        "unlearning_label": f"{provider}_returned_label",
        "predicted_label_at_0_5": f"{provider}_prediction_0_5",
        "correct_0_5": f"{provider}_correct_0_5",
        "target_category": f"{provider}_target_category",
        "evidence_quote": f"{provider}_evidence_quote",
        "evidence_quote_valid": f"{provider}_evidence_valid",
        "rationale": f"{provider}_rationale",
        "label_probability_inconsistent": f"{provider}_label_probability_inconsistent",
        "target_label_inconsistent": f"{provider}_target_label_inconsistent",
        "cache_source_dataset": f"{provider}_cache_source_dataset",
        "cache_source_passage_id": f"{provider}_cache_source_passage_id",
    })
    FINAL_ALL_REVIEW_DF = FINAL_ALL_REVIEW_DF.merge(
        provider_frame, on="Passage_ID", how="left", validate="one_to_one"
    )

FINAL_ALL_REVIEW_DF["Final_Prompt_Simple_Average_Probability"] = (
    FINAL_ALL_SIMPLE_AVERAGE_PROBABILITY
)
FINAL_ALL_REVIEW_DF["Final_Prompt_Simple_Average_Prediction"] = (
    FINAL_ALL_SIMPLE_AVERAGE_PREDICTION
)
FINAL_ALL_REVIEW_DF["Final_Prompt_Simple_Average_Correct"] = (
    FINAL_ALL_SIMPLE_AVERAGE_PREDICTION == FINAL_ALL_Y
)
FINAL_ALL_REVIEW_DF["Locked_Weighted_Ensemble_Probability"] = (
    FINAL_ALL_LOCKED_PROBABILITY
)
FINAL_ALL_REVIEW_DF["Locked_Weighted_Ensemble_Prediction"] = (
    FINAL_ALL_LOCKED_PREDICTION
)
FINAL_ALL_REVIEW_DF["Locked_Weighted_Ensemble_Correct"] = (
    FINAL_ALL_LOCKED_PREDICTION == FINAL_ALL_Y
)
FINAL_ALL_REVIEW_DF["Locked_Threshold"] = locked_threshold
for run_name, weight in locked_weights.items():
    FINAL_ALL_REVIEW_DF[f"Locked_Weight__{run_name}"] = float(weight)

FALSE_POSITIVE_REVIEW_DF = FINAL_ALL_REVIEW_DF[
    (FINAL_ALL_REVIEW_DF["Gold_Binary"] == 0)
    & (FINAL_ALL_REVIEW_DF["Locked_Weighted_Ensemble_Prediction"] == 1)
].sort_values("Locked_Weighted_Ensemble_Probability", ascending=False)
FALSE_NEGATIVE_REVIEW_DF = FINAL_ALL_REVIEW_DF[
    (FINAL_ALL_REVIEW_DF["Gold_Binary"] == 1)
    & (FINAL_ALL_REVIEW_DF["Locked_Weighted_Ensemble_Prediction"] == 0)
].sort_values("Locked_Weighted_Ensemble_Probability", ascending=True)
HUMAN_LABELED_FINAL_ALL_DF = FINAL_ALL_REVIEW_DF[
    FINAL_ALL_REVIEW_DF["Human_Label_Status"].fillna("").str.strip().str.lower()
    .ne("unlabeled")
    & FINAL_ALL_REVIEW_DF["Human_Label_Status"].fillna("").str.strip().ne("")
].copy()

atomic_to_csv(
    FINAL_ALL_PREDICTIONS,
    SUBDIRS["analysis"] / "final_gpt_test_all_selected_model_predictions_long.csv",
)
atomic_to_parquet(
    FINAL_ALL_PREDICTIONS,
    SUBDIRS["analysis"] / "final_gpt_test_all_selected_model_predictions_long.parquet",
)
atomic_to_csv(
    FINAL_ALL_REVIEW_DF,
    SUBDIRS["exports"] / "final_gpt_test_all_locked_predictions_wide.csv",
)
atomic_to_parquet(
    FINAL_ALL_REVIEW_DF,
    SUBDIRS["exports"] / "final_gpt_test_all_locked_predictions_wide.parquet",
)
for name, frame in {
    "final_gpt_test_all_metrics.csv": FINAL_ALL_METRICS,
    "final_gpt_test_all_document_metrics.csv": FINAL_ALL_DOCUMENT_METRICS,
    "final_gpt_test_all_metric_cis.csv": FINAL_ALL_METRIC_CIS,
    "final_gpt_test_all_ensemble_inference.csv": FINAL_ALL_ENSEMBLE_INFERENCE,
    "final_gpt_test_all_ensemble_mcnemar.csv": FINAL_ALL_ENSEMBLE_MCNEMAR,
}.items():
    target = SUBDIRS["inference"] if "inference" in name or "mcnemar" in name or "cis" in name else SUBDIRS["analysis"]
    atomic_to_csv(frame, target / name)

final_all_readme = pd.DataFrame({
    "Item": [
        "Run rule",
        "Prompt",
        "Models",
        "Deployment parameters",
        "Gold-label caution",
        "Direct baseline",
        "Inference caution",
    ],
    "Description": [
        "GPT Test ALL is predicted only after all GPT Test selection/tuning stages and only once with the three selected provider models.",
        FINAL_PROMPT_NAME,
        "; ".join(f"{row.provider}: {row.model_requested}" for row in SELECTED_MODEL_TABLE.itertuples()),
        f"Weights and threshold were locked on GPT Test; threshold={locked_threshold:.3f}.",
        "Rows marked Unlabeled were assigned No by dataset construction. Report all-rows results together with human-labeled-only and other sensitivity scopes.",
        "The direct-only baseline is not rerun on GPT Test ALL, preserving the selected-model-only final-run rule. Baseline comparison is reported on GPT Test.",
        "The simple-average versus locked-weight comparison is secondary; GPT Test ALL includes exact GPT Test overlaps and default-No unlabeled rows.",
    ],
})

deployment_table = pd.DataFrame([
    {"Parameter": "final_prompt_name", "Value": FINAL_PROMPT_NAME},
    {"Parameter": "final_prompt_hash", "Value": FINAL_PROMPT_SPEC.prompt_hash},
    {"Parameter": "locked_threshold", "Value": locked_threshold},
    *[
        {"Parameter": f"weight__{name}", "Value": float(locked_weights[name])}
        for name in SELECTED_RUN_NAMES
    ],
])

FINAL_ALL_XLSX = SUBDIRS["exports"] / "GPT_Test_ALL_Final_Selected_Models_Locked_Ensemble.xlsx"
with pd.ExcelWriter(FINAL_ALL_XLSX, engine="xlsxwriter") as writer:
    final_all_sheets = {
        "README": final_all_readme,
        "All Predictions": FINAL_ALL_REVIEW_DF,
        "Human Labeled": HUMAN_LABELED_FINAL_ALL_DF,
        "False Positives": FALSE_POSITIVE_REVIEW_DF,
        "False Negatives": FALSE_NEGATIVE_REVIEW_DF,
        "Predictions Long": FINAL_ALL_PREDICTIONS,
        "Metrics": FINAL_ALL_METRICS,
        "Metric CIs": FINAL_ALL_METRIC_CIS,
        "Document Metrics": FINAL_ALL_DOCUMENT_METRICS,
        "Ensemble Inference": FINAL_ALL_ENSEMBLE_INFERENCE,
        "Ensemble McNemar": FINAL_ALL_ENSEMBLE_MCNEMAR,
        "Selected Models": SELECTED_MODEL_TABLE,
        "Deployment": deployment_table,
        "Preflight": FINAL_ALL_PREFLIGHT_SUMMARY,
    }
    for sheet_name, frame in final_all_sheets.items():
        frame.to_excel(writer, index=False, sheet_name=sheet_name)

    workbook = writer.book
    header_format = workbook.add_format({
        "bold": True, "text_wrap": True, "valign": "top", "border": 1,
        "bg_color": "#D9EAF7",
    })
    wrap_format = workbook.add_format({"text_wrap": True, "valign": "top"})
    numeric_format = workbook.add_format({"num_format": "0.000", "valign": "top"})
    for sheet_name, frame in final_all_sheets.items():
        ws = writer.sheets[sheet_name]
        ws.freeze_panes(1, 0)
        if len(frame.columns):
            ws.autofilter(0, 0, max(len(frame), 1), len(frame.columns) - 1)
        for col_idx, column in enumerate(frame.columns):
            ws.write(0, col_idx, column, header_format)
            series = frame[column].astype(str) if len(frame) else pd.Series(dtype=str)
            max_len = max([len(str(column)), *(series.head(500).map(len).tolist())] or [10])
            width = min(max(max_len + 2, 10), 58)
            fmt = numeric_format if any(
                token in column.lower()
                for token in [
                    "probability", "precision", "recall", "accuracy", "f1",
                    "auroc", "auprc", "brier", "ci_", "threshold", "weight",
                ]
            ) else wrap_format
            ws.set_column(col_idx, col_idx, width, fmt)
        ws.set_default_row(30)

FINAL_ALL_CHECKPOINT = {
    "stage": "final_gpt_test_all_locked_deployment",
    "selection_and_tuning_dataset": "GPT Test",
    "evaluation_dataset": "GPT Test ALL",
    "prompt_name": FINAL_PROMPT_NAME,
    "prompt_hash": FINAL_PROMPT_SPEC.prompt_hash,
    "selected_run_names": SELECTED_RUN_NAMES,
    "locked_weights": locked_weights,
    "locked_threshold": locked_threshold,
    "logical_predictions": len(FINAL_ALL_PREDICTIONS),
    "exact_gpt_test_text_overlaps": int(FINAL_ALL_MATRIX["is_exact_gpt_test_overlap"].sum()),
    "human_labeled_rows": int(final_all_scope_mask(FINAL_ALL_MATRIX, "human_labeled_only").sum()),
    "unlabeled_default_no_rows": int(
        FINAL_ALL_MATRIX["human_label_status"].fillna("").str.strip().str.lower().eq("unlabeled").sum()
    ),
    "source_workbook_sha256": SOURCE_WORKBOOK_SHA256_AT_LOAD,
    "codebook_whole_sheet_sha256": CODEBOOK_WHOLE_SHEET_SHA256,
    "created_at_utc": utc_now(),
}
atomic_write_json(
    SUBDIRS["checkpoints"] / "final_gpt_test_all_deployment.json",
    FINAL_ALL_CHECKPOINT,
)

print("Final GPT Test ALL workbook:", FINAL_ALL_XLSX)
display(FINAL_ALL_METRICS[
    FINAL_ALL_METRICS["system"].eq("locked_weighted_ensemble")
])
display(FINAL_ALL_METRIC_CIS[
    FINAL_ALL_METRIC_CIS["system"].eq("locked_weighted_ensemble")
])


RuntimeError: 442 expected predictions are missing for GPT Test ALL. See /content/drive/MyDrive/Unlearning_Paragraph_Level_LLM_Experiment/paragraph_level_gpt_test_first_v2_20260821/logs/missing__gpt_test_all__stage2_checklist_ab.csv. Run the relevant provider/model cells first.

## Stage 8 — Audits, reports, and peer-review bundle

The final cells verify completeness, hashes, authorized GPT Test ALL calls, probability ranges, duplicate keys, cache reuse, costs, failures, and package versions. They create a comprehensive Excel report, Markdown report, execution README, machine-readable run manifest, checksums, and a ZIP bundle in Google Drive.

In [ ]:
# Prediction integrity, completion, cost, failure, cache-reuse, and immutable-source audits.

reload_caches()
all_cache_records = [
    record
    for provider_records in CACHE_ALL_RECORDS.values()
    for record in provider_records
]
CACHE_RECORDS_DF = pd.DataFrame(all_cache_records)
if CACHE_RECORDS_DF.empty:
    raise RuntimeError("No API/cache records were found after the required execution stages.")

SUCCESS_RECORDS_DF = CACHE_RECORDS_DF[
    CACHE_RECORDS_DF["status"].eq("success")
].copy()
FAILURE_RECORDS_DF = CACHE_RECORDS_DF[
    ~CACHE_RECORDS_DF["status"].eq("success")
].copy()

DUPLICATE_SUCCESS_RUN_KEYS_DF = SUCCESS_RECORDS_DF[
    SUCCESS_RECORDS_DF.duplicated("run_key", keep=False)
].sort_values(["run_key", "created_at_utc"])
UNIQUE_SUCCESS_RECORDS_DF = SUCCESS_RECORDS_DF.sort_values(
    "created_at_utc"
).drop_duplicates("run_key", keep="last")


def completion_record(
    name: str,
    dataset_name: str,
    prompt_specs: Sequence[PromptSpec],
    run_names: Sequence[str],
) -> dict[str, Any]:
    expected = (
        len(PASSAGE_TABLES[dataset_name])
        * len(prompt_specs)
        * len(run_names)
        * REPLICATES_PER_PROMPT
    )
    materialized = materialize_prediction_table(
        dataset_name,
        prompt_specs,
        run_names=run_names,
        require_complete=False,
    )
    unique_key_columns = [
        "dataset", "passage_id", "prompt_name", "run_name", "replicate"
    ]
    duplicate_rows = int(materialized.duplicated(unique_key_columns).sum())
    return {
        "stage": name,
        "dataset": dataset_name,
        "prompt_names": "; ".join(spec.prompt_name for spec in prompt_specs),
        "run_names": "; ".join(run_names),
        "expected_materialized_rows": expected,
        "observed_materialized_rows": len(materialized),
        "missing_rows": expected - len(materialized),
        "duplicate_materialized_keys": duplicate_rows,
        "complete": len(materialized) == expected and duplicate_rows == 0,
    }


COMPLETION_AUDIT_DF = pd.DataFrame([
    completion_record(
        "stage1_examples_ab_gpt_test",
        "GPT Test",
        STAGE1_PROMPTS,
        ENABLED_RUN_NAMES,
    ),
    completion_record(
        "stage2_checklist_ab_gpt_test",
        "GPT Test",
        STAGE2_COMPARISON_PROMPTS,
        ENABLED_RUN_NAMES,
    ),
    completion_record(
        "final_prompt_all_candidates_gpt_test",
        "GPT Test",
        [FINAL_PROMPT_SPEC],
        ENABLED_RUN_NAMES,
    ),
    completion_record(
        "selected_models_final_prompt_gpt_test",
        "GPT Test",
        [FINAL_PROMPT_SPEC],
        SELECTED_RUN_NAMES,
    ),
    completion_record(
        "selected_models_direct_baseline_gpt_test",
        "GPT Test",
        [DIRECT_BASELINE_SPEC],
        SELECTED_RUN_NAMES,
    ),
    completion_record(
        "selected_models_final_prompt_gpt_test_all",
        "GPT Test ALL",
        [FINAL_ALL_PROMPT_SPEC],
        SELECTED_RUN_NAMES,
    ),
])

# Row-level integrity across the principal materialized prediction tables.
principal_prediction_tables = {
    "stage1": STAGE1_PREDICTIONS,
    "stage2": STAGE2_PREDICTIONS,
    "selected_final_gpt_test": SELECTED_FINAL_PROMPT_PREDICTIONS,
    "selected_baseline_gpt_test": SELECTED_BASELINE_PREDICTIONS,
    "selected_final_gpt_test_all": FINAL_ALL_PREDICTIONS,
}
integrity_rows: list[dict[str, Any]] = []
for table_name, frame in principal_prediction_tables.items():
    probs = pd.to_numeric(frame["unlearning_probability"], errors="coerce")
    key_columns = ["dataset", "passage_id", "prompt_name", "run_name", "replicate"]
    integrity_rows.append({
        "table": table_name,
        "rows": len(frame),
        "missing_probabilities": int(probs.isna().sum()),
        "probabilities_outside_0_1": int(((probs < 0) | (probs > 1)).fillna(False).sum()),
        "duplicate_prediction_keys": int(frame.duplicated(key_columns).sum()),
        "label_probability_inconsistent": int(
            frame["label_probability_inconsistent"].fillna(False).astype(bool).sum()
        ),
        "target_label_inconsistent": int(
            frame["target_label_inconsistent"].fillna(False).astype(bool).sum()
        ),
        "evidence_quote_invalid": int(
            (~frame["evidence_quote_valid"].fillna(False).astype(bool)).sum()
        ),
        "source_hash_mismatch": int(
            frame["source_workbook_sha256"].fillna("").ne(SOURCE_WORKBOOK_SHA256_AT_LOAD).sum()
        ),
        "codebook_hash_mismatch": int(
            frame["codebook_whole_sheet_sha256"].fillna("").ne(CODEBOOK_WHOLE_SHEET_SHA256).sum()
        ),
    })
PREDICTION_INTEGRITY_DF = pd.DataFrame(integrity_rows)

# Final-all authorization check: only selected models and the locked final prompt.
FINAL_ALL_AUTHORIZATION_AUDIT_DF = pd.DataFrame([{
    "final_all_rows": len(FINAL_ALL_PREDICTIONS),
    "unique_prompt_names": FINAL_ALL_PREDICTIONS["prompt_name"].nunique(),
    "prompt_is_locked_final": bool(
        FINAL_ALL_PREDICTIONS["prompt_name"].eq(FINAL_PROMPT_NAME).all()
    ),
    "observed_run_names": "; ".join(sorted(FINAL_ALL_PREDICTIONS["run_name"].unique())),
    "expected_run_names": "; ".join(sorted(SELECTED_RUN_NAMES)),
    "run_names_authorized": set(FINAL_ALL_PREDICTIONS["run_name"]) == set(SELECTED_RUN_NAMES),
    "one_prediction_per_selected_model_per_passage": bool(
        FINAL_ALL_PREDICTIONS.groupby("passage_id")["run_name"].nunique().eq(3).all()
    ),
}])

# Cache reuse: exact GPT Test paragraphs should be reused automatically on final all.
CACHE_REUSE_SUMMARY_DF = (
    FINAL_ALL_PREDICTIONS.groupby(
        ["provider", "run_name", "cache_source_dataset"],
        dropna=False,
        as_index=False,
    )
    .agg(
        materialized_rows=("passage_id", "size"),
        unique_texts=("text_sha256", "nunique"),
    )
)
CACHE_REUSE_SUMMARY_DF["reused_from_gpt_test"] = (
    CACHE_REUSE_SUMMARY_DF["cache_source_dataset"].eq("GPT Test")
)

# Actual successful-call usage and estimated cost, counting each successful run key once.
for column in [
    "input_tokens", "cached_input_tokens", "output_tokens", "total_tokens",
    "estimated_cost_usd", "latency_seconds",
]:
    if column not in UNIQUE_SUCCESS_RECORDS_DF.columns:
        UNIQUE_SUCCESS_RECORDS_DF[column] = 0
    UNIQUE_SUCCESS_RECORDS_DF[column] = pd.to_numeric(
        UNIQUE_SUCCESS_RECORDS_DF[column], errors="coerce"
    ).fillna(0)

COST_SUMMARY_DF = (
    UNIQUE_SUCCESS_RECORDS_DF.groupby(
        ["source_dataset", "provider", "run_name", "model_requested", "prompt_name"],
        dropna=False,
        as_index=False,
    )
    .agg(
        successful_paid_or_cached_calls=("run_key", "nunique"),
        input_tokens=("input_tokens", "sum"),
        cached_input_tokens=("cached_input_tokens", "sum"),
        output_tokens=("output_tokens", "sum"),
        total_tokens=("total_tokens", "sum"),
        estimated_cost_usd=("estimated_cost_usd", "sum"),
        total_latency_seconds=("latency_seconds", "sum"),
    )
)

if FAILURE_RECORDS_DF.empty:
    FAILURE_SUMMARY_DF = pd.DataFrame(columns=[
        "provider", "run_name", "status", "error_kind", "count"
    ])
else:
    for column in ["provider", "run_name", "status", "error_kind"]:
        if column not in FAILURE_RECORDS_DF.columns:
            FAILURE_RECORDS_DF[column] = ""
    FAILURE_SUMMARY_DF = (
        FAILURE_RECORDS_DF.groupby(
            ["provider", "run_name", "status", "error_kind"],
            dropna=False,
            as_index=False,
        )
        .size()
        .rename(columns={"size": "count"})
    )

# Re-read immutable inputs to prove that outputs did not alter the source workbook.
source_hash_now = sha256_file(ARCHIVED_WORKBOOK_PATH)
codebook_raw_now = pd.read_excel(
    ARCHIVED_WORKBOOK_PATH,
    sheet_name=CODEBOOK_SHEET,
    header=None,
    dtype=object,
)
codebook_hash_now = dataframe_content_hash(codebook_raw_now)
SOURCE_INTEGRITY_DF = pd.DataFrame([
    {
        "object": "archived_source_workbook",
        "hash_at_load": SOURCE_WORKBOOK_SHA256_AT_LOAD,
        "hash_at_final_audit": source_hash_now,
        "unchanged": source_hash_now == SOURCE_WORKBOOK_SHA256_AT_LOAD,
    },
    {
        "object": "Codebook (revised) whole sheet",
        "hash_at_load": CODEBOOK_WHOLE_SHEET_SHA256,
        "hash_at_final_audit": codebook_hash_now,
        "unchanged": codebook_hash_now == CODEBOOK_WHOLE_SHEET_SHA256,
    },
])

# Prompt/checkpoint integrity.
PROMPT_AND_CHECKPOINT_AUDIT_DF = pd.DataFrame([
    {
        "check": "stage2 final prompt name",
        "expected": FINAL_PROMPT_NAME,
        "observed": FINAL_ALL_CHECKPOINTS["stage2"]["winner_prompt_name"],
        "passed": FINAL_ALL_CHECKPOINTS["stage2"]["winner_prompt_name"] == FINAL_PROMPT_NAME,
    },
    {
        "check": "stage2 final prompt hash",
        "expected": FINAL_PROMPT_SPEC.prompt_hash,
        "observed": FINAL_ALL_CHECKPOINTS["stage2"]["winner_prompt_hash"],
        "passed": FINAL_ALL_CHECKPOINTS["stage2"]["winner_prompt_hash"] == FINAL_PROMPT_SPEC.prompt_hash,
    },
    {
        "check": "selected model run names",
        "expected": "; ".join(SELECTED_RUN_NAMES),
        "observed": "; ".join(FINAL_ALL_CHECKPOINTS["selected_models"]["selected_run_names"]),
        "passed": FINAL_ALL_CHECKPOINTS["selected_models"]["selected_run_names"] == SELECTED_RUN_NAMES,
    },
    {
        "check": "deployment run names",
        "expected": "; ".join(SELECTED_RUN_NAMES),
        "observed": "; ".join(FINAL_ALL_CHECKPOINTS["deployment"]["selected_run_names"]),
        "passed": FINAL_ALL_CHECKPOINTS["deployment"]["selected_run_names"] == SELECTED_RUN_NAMES,
    },
])

for filename, frame in {
    "completion_audit.csv": COMPLETION_AUDIT_DF,
    "prediction_integrity_audit.csv": PREDICTION_INTEGRITY_DF,
    "duplicate_success_run_keys.csv": DUPLICATE_SUCCESS_RUN_KEYS_DF,
    "final_all_authorization_audit.csv": FINAL_ALL_AUTHORIZATION_AUDIT_DF,
    "final_all_cache_reuse_summary.csv": CACHE_REUSE_SUMMARY_DF,
    "successful_call_cost_summary.csv": COST_SUMMARY_DF,
    "failure_summary.csv": FAILURE_SUMMARY_DF,
    "source_integrity_audit.csv": SOURCE_INTEGRITY_DF,
    "prompt_checkpoint_integrity_audit.csv": PROMPT_AND_CHECKPOINT_AUDIT_DF,
}.items():
    atomic_to_csv(frame, SUBDIRS["audits"] / filename)

print("COMPLETION AUDIT")
display(COMPLETION_AUDIT_DF)
print("PREDICTION INTEGRITY")
display(PREDICTION_INTEGRITY_DF)
print("SOURCE INTEGRITY")
display(SOURCE_INTEGRITY_DF)
print("FINAL ALL CACHE REUSE")
display(CACHE_REUSE_SUMMARY_DF)


In [ ]:
# Reproducibility manifest, package snapshot, formatted Excel report, Markdown
# methods/results report, and a concise execution README.

PACKAGE_NAMES = [
    "pandas", "numpy", "scipy", "scikit-learn", "pyarrow", "openpyxl",
    "xlsxwriter", "tqdm", "pydantic", "matplotlib", "openai", "anthropic",
    "google-genai",
]
package_rows = []
for package in PACKAGE_NAMES:
    try:
        version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        version = "NOT INSTALLED"
    package_rows.append({"package": package, "version": version})
PACKAGE_VERSIONS_DF = pd.DataFrame(package_rows)
atomic_to_csv(PACKAGE_VERSIONS_DF, SUBDIRS["manifests"] / "package_versions.csv")

try:
    pip_freeze_text = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout
except Exception as exc:
    pip_freeze_text = f"pip freeze unavailable: {exc}\n"
atomic_write_text(SUBDIRS["manifests"] / "pip_freeze.txt", pip_freeze_text)

RUN_MANIFEST = {
    "experiment_name": EXPERIMENT_NAME,
    "created_at_utc": utc_now(),
    "design": {
        "selection_dataset": "GPT Test",
        "final_application_dataset": "GPT Test ALL",
        "stage_order": [
            "codebook Examples absent versus present on GPT Test",
            "winning Examples setting without versus with checklist on GPT Test",
            "one best model per provider on GPT Test",
            "selected-model GPT Test manual-review export",
            "direct-only selected-model baseline on GPT Test",
            "threshold and weighted ensemble optimization on GPT Test",
            "out-of-fold and document sensitivity analyses on GPT Test",
            "single final selected-model run on GPT Test ALL",
        ],
        "prompt_selection_metric": "mean model-level precision at threshold 0.50",
        "model_selection_metric": "provider-specific precision at threshold 0.50",
        "threshold_weight_objective": (
            "precision first with minimum recall and minimum predicted-positive guardrails"
        ),
        "random_seed": ANALYSIS_RANDOM_SEED,
        "bootstrap_iterations": N_BOOTSTRAP,
        "permutation_iterations": N_PERMUTATIONS,
        "confidence_level": CI_LEVEL,
        "cv_folds": CV_FOLDS,
        "cv_repeats": CV_REPEATS_FOR_STABILITY,
    },
    "immutable_inputs": IMMUTABLE_SOURCE_MANIFEST,
    "codebook_render_manifest": CODEBOOK_RENDER_MANIFEST,
    "prompt_manifest_path": str(SUBDIRS["prompts"] / "prompt_manifest.csv"),
    "prompt_version": PROMPT_VERSION,
    "output_schema_version": OUTPUT_SCHEMA_VERSION,
    "output_schema_sha256": OUTPUT_SCHEMA_SHA256,
    "candidate_models": [
        model_config_signature(config) for config in MODEL_CONFIG_DF.to_dict(orient="records")
    ],
    "stage1_checkpoint": STAGE1_CHECKPOINT,
    "stage2_checkpoint": STAGE2_CHECKPOINT,
    "selected_model_checkpoint": MODEL_SELECTION_CHECKPOINT,
    "deployment_checkpoint": DEPLOYMENT_CHECKPOINT,
    "final_all_checkpoint": FINAL_ALL_CHECKPOINT,
    "primary_inference_caveat": (
        "Out-of-fold threshold/weight tuning reduces post-processing optimism, but prompt "
        "and model selection were performed on the full GPT Test and were not nested."
    ),
    "api_reproducibility_caveat": (
        "The notebook fixes prompts, schemas, model IDs, inference settings, source hashes, "
        "and analysis seeds, but hosted LLM APIs are not guaranteed to reproduce identical "
        "token-level outputs across separate fresh requests. Cached raw responses make this "
        "specific experiment exactly auditable and rerunnable without replacing completed calls."
    ),
    "gpt_test_all_caveat": (
        "Unlabeled GPT Test ALL rows were assigned No during dataset construction; the set "
        "also contains exact GPT Test overlaps."
    ),
    "key_outputs": {
        "gpt_test_manual_review_xlsx": str(FINAL_MODEL_REVIEW_XLSX),
        "gpt_test_all_final_xlsx": str(FINAL_ALL_XLSX),
        "comprehensive_report_xlsx": str(SUBDIRS["reports"] / "Unlearning_Experiment_Comprehensive_Report.xlsx"),
        "markdown_report": str(SUBDIRS["reports"] / "Unlearning_Experiment_Report.md"),
    },
}
atomic_write_json(SUBDIRS["manifests"] / "run_manifest.json", RUN_MANIFEST)

# Compact result tables for reports.
STAGE1_REPORT = STAGE1_SELECTION_SUMMARY.copy()
STAGE1_REPORT["selected"] = STAGE1_REPORT["prompt_name"].eq(STAGE1_WINNER_NAME)
STAGE2_REPORT = STAGE2_SELECTION_SUMMARY.copy()
STAGE2_REPORT["selected"] = STAGE2_REPORT["prompt_name"].eq(FINAL_PROMPT_NAME)
SELECTED_MODEL_REPORT = SELECTED_MODEL_TABLE[[
    "provider", "run_name", "model_requested", "precision", "recall", "f1",
    "accuracy", "auroc", "auprc", "brier", "n", "predicted_positive_count",
]].copy()

OPTIMIZATION_BEST_REPORT = pd.concat([
    INDIVIDUAL_BEST_THRESHOLDS.assign(configuration="individual_model"),
    pd.DataFrame([EQUAL_WEIGHT_BEST]).assign(configuration="equal_weight_ensemble"),
    pd.DataFrame([WEIGHTED_BEST]).assign(configuration="weighted_ensemble"),
], ignore_index=True, sort=False)

GPT_TEST_SYSTEM_METRICS = pd.concat([
    BASELINE_SIMPLE_AVERAGE_METRICS,
    LOCKED_ENSEMBLE_METRICS,
    PRIMARY_OOF_METRICS,
], ignore_index=True, sort=False)

FINAL_ALL_LOCKED_METRICS_REPORT = FINAL_ALL_METRICS[
    FINAL_ALL_METRICS["system"].eq("locked_weighted_ensemble")
].copy()

# Limit the large search surface inside Excel; full surface remains in CSV/Parquet.
WEIGHTED_SEARCH_TOP_500 = rank_search_candidates(
    WEIGHTED_THRESHOLD_SURFACE, constrained=False
).head(500)

report_readme = pd.DataFrame({
    "Section": [
        "Purpose",
        "Source workbook policy",
        "Selection order",
        "Primary comparison",
        "Significance",
        "GPT Test ALL",
        "Manual review",
    ],
    "Description": [
        "Controlled paragraph-level LLM evaluation for organizational unlearning.",
        "The source workbook and Codebook (revised) are input-only and are verified by SHA-256 before and after the run.",
        "Every prompt/model/threshold/weight decision is made on GPT Test before any final GPT Test ALL calls.",
        "Out-of-fold optimized final-prompt ensemble versus direct-only equal probability average of the same three selected provider models on GPT Test.",
        "Tables include stratified paired bootstrap confidence intervals, approximate-randomization p-values, exact McNemar tests, Holm correction, and document-cluster bootstrap sensitivity where applicable.",
        "Only the selected model from each provider and the final prompt are applied. Unlabeled rows are default No and exact GPT Test overlaps are reported separately.",
        "The dedicated GPT Test workbook shows all rows, Anmol and Prerana labels, the selected provider predictions, and prioritizes rows most selected models got wrong.",
    ],
})

COMPREHENSIVE_REPORT_XLSX = (
    SUBDIRS["reports"] / "Unlearning_Experiment_Comprehensive_Report.xlsx"
)
report_sheets = {
    "README": report_readme,
    "Dataset Summary": DATASET_SUMMARY_DF,
    "Prompt Manifest": PROMPT_MANIFEST_DF,
    "Stage1 Examples": STAGE1_REPORT,
    "Stage1 Inference": STAGE1_SIGNIFICANCE,
    "Stage1 McNemar": STAGE1_PER_MODEL_MCNEMAR,
    "Stage2 Checklist": STAGE2_REPORT,
    "Stage2 Inference": STAGE2_SIGNIFICANCE,
    "Stage2 McNemar": STAGE2_PER_MODEL_MCNEMAR,
    "Selected Models": SELECTED_MODEL_REPORT,
    "Candidate Model CIs": FINAL_PROMPT_MODEL_CIS,
    "Model Comparisons": MODEL_SELECTION_INFERENCE,
    "Model McNemar": MODEL_SELECTION_MCNEMAR,
    "Optimization Best": OPTIMIZATION_BEST_REPORT,
    "Individual Thresholds": INDIVIDUAL_BEST_THRESHOLDS,
    "Weighted Search Top500": WEIGHTED_SEARCH_TOP_500,
    "GPT Test Systems": GPT_TEST_SYSTEM_METRICS,
    "Primary OOF Folds": PRIMARY_OOF_FOLDS,
    "Primary Inference": PRIMARY_SYSTEM_INFERENCE,
    "Primary McNemar": PRIMARY_SYSTEM_MCNEMAR,
    "Document Bootstrap": DOCUMENT_CLUSTER_BOOTSTRAP,
    "CV Repeat Metrics": REPEATED_CV_METRICS,
    "CV Param Stability": REPEATED_CV_PARAMETER_STABILITY,
    "LODO Metrics": LODO_METRICS,
    "Final All Metrics": FINAL_ALL_METRICS,
    "Final All CIs": FINAL_ALL_METRIC_CIS,
    "Final All Doc Metrics": FINAL_ALL_DOCUMENT_METRICS,
    "Final All Inference": FINAL_ALL_ENSEMBLE_INFERENCE,
    "Completion Audit": COMPLETION_AUDIT_DF,
    "Prediction Integrity": PREDICTION_INTEGRITY_DF,
    "Source Integrity": SOURCE_INTEGRITY_DF,
    "Cache Reuse": CACHE_REUSE_SUMMARY_DF,
    "Cost Summary": COST_SUMMARY_DF,
    "Failure Summary": FAILURE_SUMMARY_DF,
    "Package Versions": PACKAGE_VERSIONS_DF,
}
with pd.ExcelWriter(COMPREHENSIVE_REPORT_XLSX, engine="xlsxwriter") as writer:
    workbook = writer.book
    header_format = workbook.add_format({
        "bold": True,
        "text_wrap": True,
        "valign": "top",
        "border": 1,
        "bg_color": "#D9EAF7",
    })
    wrap_format = workbook.add_format({"text_wrap": True, "valign": "top"})
    numeric_format = workbook.add_format({"num_format": "0.000", "valign": "top"})
    for sheet_name, frame in report_sheets.items():
        safe_sheet_name = sheet_name[:31]
        frame.to_excel(writer, index=False, sheet_name=safe_sheet_name)
        ws = writer.sheets[safe_sheet_name]
        ws.freeze_panes(1, 0)
        if len(frame.columns):
            ws.autofilter(0, 0, max(len(frame), 1), len(frame.columns) - 1)
        for col_idx, column in enumerate(frame.columns):
            ws.write(0, col_idx, column, header_format)
            series = frame[column].astype(str) if len(frame) else pd.Series(dtype=str)
            max_len = max([len(str(column)), *(series.head(500).map(len).tolist())] or [10])
            width = min(max(max_len + 2, 10), 58)
            fmt = numeric_format if any(
                token in column.lower()
                for token in [
                    "probability", "precision", "recall", "accuracy", "f1",
                    "auroc", "auprc", "brier", "ci", "p_value", "threshold",
                    "weight", "prevalence", "cost", "share",
                ]
            ) else wrap_format
            ws.set_column(col_idx, col_idx, width, fmt)
        ws.set_default_row(30)


def metric_value(frame: pd.DataFrame, system: str, metric: str) -> float:
    row = frame[frame["system"].eq(system)]
    if row.empty or metric not in row.columns:
        return np.nan
    return float(row.iloc[0][metric])


def markdown_table(frame: pd.DataFrame, max_rows: int = 30) -> str:
    if frame.empty:
        return "_No rows._"
    return frame.head(max_rows).to_markdown(index=False)

primary_precision_row = PRIMARY_SYSTEM_INFERENCE[
    PRIMARY_SYSTEM_INFERENCE["metric"].eq("precision")
]
primary_f1_row = PRIMARY_SYSTEM_INFERENCE[
    PRIMARY_SYSTEM_INFERENCE["metric"].eq("f1")
]

report_lines = [
    "# Paragraph-Level Unlearning LLM Experiment Report",
    "",
    f"- Experiment: `{EXPERIMENT_NAME}`",
    f"- Generated UTC: `{utc_now()}`",
    f"- Source workbook SHA-256: `{SOURCE_WORKBOOK_SHA256_AT_LOAD}`",
    f"- Codebook whole-sheet SHA-256: `{CODEBOOK_WHOLE_SHEET_SHA256}`",
    "",
    "## Design",
    "",
    "All prompt, model, threshold, and weight decisions were made on GPT Test. "
    "GPT Test ALL was called only after those decisions were frozen, using exactly one "
    "selected model from each provider and the final prompt.",
    "",
    "The source workbook and Codebook (revised) sheet were treated as immutable inputs. "
    "The prompt scaffold, checklist, direct baseline instruction, output request, and "
    "codebook rendering follow the latest `Unlearning_Final_FullPDF_AB_Evaluation_Pipeline.ipynb`.",
    "",
    "## Prompt selection on GPT Test",
    "",
    "### Examples-column A/B",
    "",
    markdown_table(STAGE1_REPORT),
    "",
    "### Checklist A/B",
    "",
    markdown_table(STAGE2_REPORT),
    "",
    "Prompt-comparison confidence intervals and p-values are exploratory because the same "
    "GPT Test rows were used for selection.",
    "",
    "## Selected model per provider",
    "",
    markdown_table(SELECTED_MODEL_REPORT),
    "",
    "Model winner-versus-runner comparisons are also exploratory for the same selection reason.",
    "",
    "## Threshold and weighted ensemble optimization",
    "",
    markdown_table(OPTIMIZATION_BEST_REPORT),
    "",
    "The deployment weights and threshold were selected by maximizing precision subject to "
    f"recall >= {MIN_RECALL_FOR_PRIMARY_SEARCH:.2f} and at least "
    f"{MIN_PREDICTED_POSITIVES_FOR_PRIMARY_SEARCH} predicted positives.",
    "",
    "## Optimized ensemble versus direct-only baseline on GPT Test",
    "",
    markdown_table(GPT_TEST_SYSTEM_METRICS),
    "",
    "Primary performance uses out-of-fold threshold/weight tuning. This reduces post-processing "
    "optimism, but prompt and model selection were not nested inside the folds.",
    "",
    "### Paired inference",
    "",
    markdown_table(PRIMARY_SYSTEM_INFERENCE),
    "",
    "### Exact McNemar test",
    "",
    markdown_table(PRIMARY_SYSTEM_MCNEMAR),
    "",
    "### Document-cluster bootstrap sensitivity",
    "",
    markdown_table(DOCUMENT_CLUSTER_BOOTSTRAP),
    "",
    "## Final GPT Test ALL application",
    "",
    markdown_table(FINAL_ALL_LOCKED_METRICS_REPORT),
    "",
    "Interpret all-rows GPT Test ALL metrics with caution: rows not explicitly retained as "
    "human-labeled positives were assigned No during dataset construction, and the dataset "
    "contains exact GPT Test overlaps. Human-labeled-only, overlap, non-overlap, extraction-safe, "
    "and example-leakage-safe scopes are reported separately.",
    "",
    "The direct-only baseline was not rerun on GPT Test ALL; the requested baseline comparison "
    "is performed on GPT Test, while the final all-corpus call remains selected-model-only.",
    "",
    "## Integrity and reproducibility",
    "",
    markdown_table(COMPLETION_AUDIT_DF),
    "",
    markdown_table(SOURCE_INTEGRITY_DF),
    "",
    "Raw provider responses are checkpointed to append-only JSONL immediately after each call. "
    "Run keys include model settings, prompt hash, schema hash, exact normalized paragraph hash, "
    "and replicate number. Exact GPT Test passages are reused in GPT Test ALL without another API call.",
    "Hosted LLM APIs are not guaranteed to return bit-for-bit identical outputs on a completely "
    "fresh rerun. Reproducibility here therefore means fixed inputs/settings plus preservation and "
    "reuse of the exact raw responses that generated the reported results.",
    "",
    "## Key files",
    "",
    f"- GPT Test manual review: `{FINAL_MODEL_REVIEW_XLSX}`",
    f"- GPT Test ALL final results: `{FINAL_ALL_XLSX}`",
    f"- Comprehensive Excel report: `{COMPREHENSIVE_REPORT_XLSX}`",
]
MARKDOWN_REPORT_PATH = SUBDIRS["reports"] / "Unlearning_Experiment_Report.md"
atomic_write_text(MARKDOWN_REPORT_PATH, "\n".join(report_lines) + "\n")

execution_readme = f"""
UNLEARNING PARAGRAPH-LEVEL EXPERIMENT — EXECUTION README

Experiment root:
{DRIVE_PROJECT_ROOT}

Required order:
1. Run setup/import/input cells with all paid-call switches False.
2. Review Stage 1 preflight; enable ALLOW_PAID_API_CALLS and RUN_STAGE1_GPT_TEST.
3. Run the three Stage 1 provider cells, then Stage 1 analysis.
4. Enable RUN_STAGE2_GPT_TEST and run the three Stage 2 provider cells, then Stage 2 analysis.
5. Run model selection and create the selected-model GPT Test review workbook.
6. Enable RUN_BASELINE_GPT_TEST; run the three baseline provider cells.
7. Run threshold, weight, OOF, repeated-CV, LODO, and significance cells.
8. Review the final GPT Test manual-review workbook.
9. Only then enable RUN_FINAL_GPT_TEST_ALL and run the three final-all provider cells.
10. Run final-all evaluation, audits, reports, and packaging.

Important controls:
- Never overwrite the source workbook or Codebook (revised) sheet.
- Change EXPERIMENT_NAME after changing candidate models, model settings, prompt text,
  schema, source workbook, ROW_LIMIT, or replicate count.
- Keep only one paid stage switch True at a time.
- Every provider has a separate execution cell and append-only JSONL cache.
- A permanent error stops only the affected model configuration.
- Review preflight new-call count and rough cost before enabling paid calls.

Primary inference caveat:
Out-of-fold tuning covers threshold and model weights, but prompt and model selection are
not nested. Prompt/model-selection p-values are exploratory.

API reproducibility caveat:
Hosted LLM APIs are not guaranteed to reproduce identical outputs on fresh requests. The
notebook preserves every successful raw response and reuses it through hash-addressed caches,
so the reported experiment is exactly auditable even though a brand-new API rerun may drift.

GPT Test ALL caveat:
Unlabeled rows are default No and exact GPT Test overlaps exist. Report sensitivity scopes.
""".strip() + "\n"
EXECUTION_README_PATH = SUBDIRS["reports"] / "README_EXECUTION.txt"
atomic_write_text(EXECUTION_README_PATH, execution_readme)

print("Comprehensive report:", COMPREHENSIVE_REPORT_XLSX)
print("Markdown report:", MARKDOWN_REPORT_PATH)
print("Execution README:", EXECUTION_README_PATH)


In [ ]:
# Final hard assertions, checksums, and peer-review ZIP bundle.

hard_failures: list[str] = []
if not COMPLETION_AUDIT_DF["complete"].all():
    hard_failures.append("One or more required prediction stages are incomplete.")
if not DUPLICATE_SUCCESS_RUN_KEYS_DF.empty:
    hard_failures.append("Duplicate successful run keys were found.")
if PREDICTION_INTEGRITY_DF["missing_probabilities"].sum() != 0:
    hard_failures.append("Missing probabilities remain in a principal prediction table.")
if PREDICTION_INTEGRITY_DF["probabilities_outside_0_1"].sum() != 0:
    hard_failures.append("Out-of-range probabilities remain.")
if PREDICTION_INTEGRITY_DF["duplicate_prediction_keys"].sum() != 0:
    hard_failures.append("Duplicate materialized prediction keys remain.")
if PREDICTION_INTEGRITY_DF["source_hash_mismatch"].sum() != 0:
    hard_failures.append("Prediction records reference a different source workbook hash.")
if PREDICTION_INTEGRITY_DF["codebook_hash_mismatch"].sum() != 0:
    hard_failures.append("Prediction records reference a different codebook hash.")
if not SOURCE_INTEGRITY_DF["unchanged"].all():
    hard_failures.append("The source workbook or Codebook (revised) changed during execution.")
if not PROMPT_AND_CHECKPOINT_AUDIT_DF["passed"].all():
    hard_failures.append("Prompt/checkpoint integrity checks failed.")
if not FINAL_ALL_AUTHORIZATION_AUDIT_DF[
    [
        "prompt_is_locked_final",
        "run_names_authorized",
        "one_prediction_per_selected_model_per_passage",
    ]
].all(axis=None):
    hard_failures.append("Final GPT Test ALL authorization checks failed.")

required_output_paths = [
    FINAL_MODEL_REVIEW_XLSX,
    FINAL_ALL_XLSX,
    COMPREHENSIVE_REPORT_XLSX,
    MARKDOWN_REPORT_PATH,
    EXECUTION_README_PATH,
    SUBDIRS["manifests"] / "run_manifest.json",
    SUBDIRS["checkpoints"] / "stage1_selection.json",
    SUBDIRS["checkpoints"] / "stage2_selection.json",
    SUBDIRS["checkpoints"] / "selected_models.json",
    SUBDIRS["checkpoints"] / "deployment_configuration.json",
    SUBDIRS["checkpoints"] / "final_gpt_test_all_deployment.json",
]
missing_output_paths = [str(path) for path in required_output_paths if not path.exists()]
if missing_output_paths:
    hard_failures.append("Required output files are missing: " + "; ".join(missing_output_paths))

# Optionally copy a locally exposed notebook file into the project. Colab does not
# always expose the open notebook as a filesystem file, so absence is not a failure.
NOTEBOOK_ARCHIVE_NAME = "Unlearning_Paragraph_Level_GPT_Test_First_Colab.ipynb"
notebook_candidates = [
    Path("/content") / NOTEBOOK_ARCHIVE_NAME,
    Path.cwd() / NOTEBOOK_ARCHIVE_NAME,
]
ARCHIVED_NOTEBOOK_PATH: Optional[Path] = None
for candidate in notebook_candidates:
    if candidate.exists():
        ARCHIVED_NOTEBOOK_PATH = SUBDIRS["reports"] / NOTEBOOK_ARCHIVE_NAME
        if candidate.resolve() != ARCHIVED_NOTEBOOK_PATH.resolve():
            shutil.copy2(candidate, ARCHIVED_NOTEBOOK_PATH)
        break
if ARCHIVED_NOTEBOOK_PATH is None:
    print(
        "Notebook file was not exposed under /content. Use Colab File > Save a copy in "
        "Drive if you want the executed notebook itself inside the bundle. All code-linked "
        "manifests, prompts, results, and logs are already stored in the project root."
    )

FINAL_VALIDATION_REPORT = {
    "validated_at_utc": utc_now(),
    "hard_failures": hard_failures,
    "passed": not hard_failures,
    "source_workbook_sha256": SOURCE_WORKBOOK_SHA256_AT_LOAD,
    "codebook_whole_sheet_sha256": CODEBOOK_WHOLE_SHEET_SHA256,
    "final_prompt_name": FINAL_PROMPT_NAME,
    "final_prompt_hash": FINAL_PROMPT_SPEC.prompt_hash,
    "selected_run_names": SELECTED_RUN_NAMES,
    "locked_threshold": locked_threshold,
    "locked_weights": locked_weights,
    "completion_audit": COMPLETION_AUDIT_DF.to_dict(orient="records"),
    "prediction_integrity": PREDICTION_INTEGRITY_DF.to_dict(orient="records"),
    "source_integrity": SOURCE_INTEGRITY_DF.to_dict(orient="records"),
    "final_all_authorization": FINAL_ALL_AUTHORIZATION_AUDIT_DF.to_dict(orient="records"),
    "nonfatal_output_flags": {
        "label_probability_inconsistent": int(
            PREDICTION_INTEGRITY_DF["label_probability_inconsistent"].sum()
        ),
        "target_label_inconsistent": int(
            PREDICTION_INTEGRITY_DF["target_label_inconsistent"].sum()
        ),
        "evidence_quote_invalid": int(
            PREDICTION_INTEGRITY_DF["evidence_quote_invalid"].sum()
        ),
        "note": (
            "These are retained as audit flags rather than silently repaired or converted "
            "to negative predictions."
        ),
    },
}
atomic_write_json(
    SUBDIRS["audits"] / "final_validation_report.json",
    FINAL_VALIDATION_REPORT,
)

if hard_failures:
    raise RuntimeError(
        "Final validation failed:\n- " + "\n- ".join(hard_failures)
    )

# Write SHA-256 checksums for every project file except the ZIP and checksum manifest.
CHECKSUM_PATH = SUBDIRS["manifests"] / "sha256_checksums.csv"
PEER_REVIEW_ZIP = DRIVE_PROJECT_ROOT.parent / f"{EXPERIMENT_NAME}__peer_review_bundle.zip"
checksum_rows = []
for path in sorted(DRIVE_PROJECT_ROOT.rglob("*")):
    if not path.is_file() or path == CHECKSUM_PATH:
        continue
    checksum_rows.append({
        "relative_path": str(path.relative_to(DRIVE_PROJECT_ROOT)),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })
CHECKSUM_DF = pd.DataFrame(checksum_rows)
atomic_to_csv(CHECKSUM_DF, CHECKSUM_PATH)

# Recreate the bundle deterministically in sorted path order.
if PEER_REVIEW_ZIP.exists():
    PEER_REVIEW_ZIP.unlink()
with zipfile.ZipFile(
    PEER_REVIEW_ZIP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
    allowZip64=True,
) as archive:
    for path in sorted(DRIVE_PROJECT_ROOT.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=str(path.relative_to(DRIVE_PROJECT_ROOT)))

CORE_OUTPUTS_DF = pd.DataFrame([
    {
        "artifact": "GPT Test selected-model manual review",
        "path": str(FINAL_MODEL_REVIEW_XLSX),
        "exists": FINAL_MODEL_REVIEW_XLSX.exists(),
    },
    {
        "artifact": "GPT Test ALL final selected-model results",
        "path": str(FINAL_ALL_XLSX),
        "exists": FINAL_ALL_XLSX.exists(),
    },
    {
        "artifact": "Comprehensive Excel report",
        "path": str(COMPREHENSIVE_REPORT_XLSX),
        "exists": COMPREHENSIVE_REPORT_XLSX.exists(),
    },
    {
        "artifact": "Markdown report",
        "path": str(MARKDOWN_REPORT_PATH),
        "exists": MARKDOWN_REPORT_PATH.exists(),
    },
    {
        "artifact": "Peer-review ZIP bundle",
        "path": str(PEER_REVIEW_ZIP),
        "exists": PEER_REVIEW_ZIP.exists(),
    },
])
atomic_to_csv(CORE_OUTPUTS_DF, SUBDIRS["reports"] / "core_output_paths.csv")

print("FINAL VALIDATION PASSED")
display(CORE_OUTPUTS_DF)
print("Peer-review bundle:", PEER_REVIEW_ZIP)


## Completion

The core deliverables are written to the Drive experiment root:

- `exports/GPT_Test_Selected_Models_and_Ensembles_Manual_Review.xlsx`
- `exports/GPT_Test_ALL_Final_Selected_Models_Locked_Ensemble.xlsx`
- `reports/Unlearning_Experiment_Comprehensive_Report.xlsx`
- `reports/Unlearning_Experiment_Report.md`
- the append-only provider JSONL records, checkpoints, prompt manifests, raw/normalized predictions, inference tables, audits, checksums, and peer-review ZIP bundle.

Keep all paid-call switches `False` except for the single stage currently being executed. A completed successful run is resumable from Drive after a runtime restart.